In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/apple_support_clean.csv")

print("Shape:", df.shape)
df.head()

Shape: (106523, 6)


,customer_tweet_id,customer_message,previous_message,has_context,parent_author,brand_response
0,698.0,@AppleSupport https://t.co/NV0yucs0lB,@AppleSupport why are my I️’s changing not sho...,True,115854,@115854 We're here for you. Which version of t...
1,697.0,@AppleSupport The newest update. I️ made sure ...,@115854 We're here for you. Which version of t...,True,AppleSupport,@115854 Lets take a closer look into this issu...
2,702.0,@AppleSupport Tried resetting my settings .. r...,@115855 Any steps tried since it started last ...,True,AppleSupport,@115855 Let's go to DM for the next steps. DM ...
3,704.0,@AppleSupport This is what it looks like https...,@115855 That's great it has iOS 11.1 as we can...,True,AppleSupport,@115855 Any steps tried since it started last ...
4,707.0,@AppleSupport I️ have an iPhone 7 Plus and yes...,@115855 We'd like to look into this with you. ...,True,AppleSupport,@115855 That's great it has iOS 11.1 as we can...


In [2]:
from collections import Counter
import re

# Customer messages only
customer_text = (
    df["customer_message"]
    .fillna("")
    .astype(str)
    .str.lower()
)

# Extract words
words = re.findall(
    r"\b[a-z][a-z0-9']+\b",
    " ".join(customer_text)
)

# Generic words that don't help identify support topics
stop_words = {
    "the", "and", "to", "a", "i", "is", "it", "of", "my", "for",
    "on", "in", "this", "that", "with", "you", "can", "be", "have",
    "do", "does", "how", "why", "what", "please", "help", "me",
    "apple", "support", "iphone", "im", "i'm", "we", "us", "they",
    "your", "are", "was", "but", "not", "just", "so", "or", "if",
    "at", "from", "about", "after", "when", "any", "get", "got",
    "has", "had", "will", "would", "all", "now", "up", "out",
    "there", "really", "like", "as", "an", "dont", "don't",
    "its", "it's", "been", "no", "yes", "also", "more", "one",
    "only", "still", "new", "phone", "device"
}

filtered_words = [
    word for word in words
    if word not in stop_words
    and len(word) > 2
]

word_counts = Counter(filtered_words)

print("Top 100 candidate topic terms:\n")

for word, count in word_counts.most_common(100):
    print(f"{word:<25} {count:,}")

Top 100 candidate topic terms:

applesupport              78,917
ios                       16,128
https                     15,932
update                    15,895
fix                       14,518
battery                   8,527
since                     5,970
app                       5,148
time                      4,943
updated                   4,858
off                       4,608
screen                    4,508
ios11                     4,332
shit                      4,247
apps                      4,185
work                      4,149
thanks                    4,091
issue                     4,047
amp                       4,038
hey                       3,927
problem                   3,845
back                      3,692
every                     3,682
even                      3,657
need                      3,574
working                   3,561
music                     3,452
going                     3,385
keeps                     3,005
use                       2,911
don

In [3]:
# Candidate topic keywords based on the frequency analysis.
# We will inspect real examples before deciding on final intents.

candidate_topics = {
    "battery": ["battery", "charging", "charge", "drain", "power"],
    "ios_update": ["ios", "ios11", "update", "updating", "upgrade"],
    "wifi_connectivity": ["wifi", "wi-fi", "bluetooth", "connection", "connect"],
    "screen_display": ["screen", "display", "brightness", "touch"],
    "apps": ["app", "apps", "application"],
    "icloud": ["icloud"],
    "apple_id": ["apple id", "icloud account", "password"],
    "audio": ["speaker", "sound", "audio", "volume", "headphone"],
    "camera": ["camera", "photo", "photos", "video"],
    "keyboard": ["keyboard", "key", "typing"],
    "mac": ["macbook", "mac", "macos", "osx"],
    "apple_music": ["apple music", "music", "playlist"],
    "airpods": ["airpod", "airpods"],
    "apple_watch": ["apple watch", "watch"],
}

for topic, keywords in candidate_topics.items():
    pattern = "|".join(
        re.escape(keyword)
        for keyword in keywords
    )

    mask = customer_text.str.contains(
        pattern,
        regex=True,
        na=False
    )

    topic_examples = df.loc[
        mask,
        "customer_message"
    ]

    print("\n" + "=" * 70)
    print(f"{topic.upper()} — {len(topic_examples):,} matching messages")
    print("=" * 70)

    if len(topic_examples) > 0:
        examples = topic_examples.sample(
            min(10, len(topic_examples)),
            random_state=42
        )

        for i, message in enumerate(examples, 1):
            print(f"{i}. {message}")


BATTERY — 11,082 matching messages
1. @AppleSupport hey could you please check my iPhone 7? The battery drainage is pretty quick in the recent days not sure why.
Thank you.
2. @AppleSupport I didnt want to re-purchase https://t.co/VpYV2ucsiQ. I got charged today. Please help to unsubcribe and get refund. Thank you!
3. @AppleSupport The battery has been consistently draining, but it definitely has gotten worse with the iOS update. 

All of my apps are up-to-date and I always close the background apps too. For example, my phone was at 100% this morning and drained to 3% in three hours after just texting.
4. @115858 what is up wth your new IOS update? It is KILLING my battery 😫
5. @applesupport 11.0.3 battery
updated to 11.0.3 this a.m. and now iphone 6 battery % is ticking down like a timer....crap update !!!
6. @AppleSupport Hi, are you thinking of doing anything to solve the battery problem? Got worse after every update. My iPhone 6 died because of you and I’m very angry!
7. My phone 

In [4]:
from sklearn.feature_extraction.text import CountVectorizer

# Use customer messages to discover common 2-word and 3-word phrases.
vectorizer = CountVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(2, 3),
    min_df=100,
    max_features=200
)

X = vectorizer.fit_transform(
    df["customer_message"]
    .fillna("")
)

phrase_counts = X.sum(axis=0).A1

phrases = pd.DataFrame({
    "phrase": vectorizer.get_feature_names_out(),
    "count": phrase_counts
}).sort_values(
    "count",
    ascending=False
)

print("Top 100 common phrases:\n")

display(
    phrases.head(100).reset_index(drop=True)
)

Top 100 common phrases:



,phrase,count
0,ios 11,10558
1,115858 applesupport,4910
2,applesupport iphone,3043
3,115858 fix,2976
4,applesupport 115858,2643
...,...,...
95,dear applesupport,434
96,don know,431
97,applesupport having,431
98,times day,430


In [5]:
# Candidate intent families.
# These are hypotheses, NOT final labels.

intent_keywords = {
    "battery_charging": [
        "battery", "battery life", "battery drain",
        "charging", "charge", "charger", "overheating"
    ],
    "ios_updates": [
        "ios", "ios 11", "ios11", "update", "updating",
        "upgrade", "software update"
    ],
    "wifi_connectivity": [
        "wifi", "wi-fi", "bluetooth", "connection",
        "connect", "network", "internet"
    ],
    "apple_id_account": [
        "apple id", "password", "account", "sign in",
        "login", "verification", "authentication"
    ],
    "icloud_backup": [
        "icloud", "icloud backup", "backup", "sync",
        "synchronization"
    ],
    "apps_app_store": [
        "app store", "apps", "app", "application",
        "download app", "update app"
    ],
    "audio": [
        "speaker", "audio", "sound", "volume",
        "headphones", "headphone", "earphone"
    ],
    "camera_photos": [
        "camera", "photos", "photo", "pictures",
        "video", "camera roll"
    ],
    "display_touchscreen": [
        "screen", "display", "touchscreen", "touch",
        "brightness", "black screen", "white screen"
    ],
    "apple_music": [
        "apple music", "music", "playlist",
        "songs", "song"
    ],
    "apple_watch": [
        "apple watch", "applewatch", "watch"
    ],
    "airpods": [
        "airpods", "airpod"
    ],
    "mac_macos": [
        "macbook", "macos", "mac os", "osx"
    ],
}

coverage = []

for intent, keywords in intent_keywords.items():
    pattern = "|".join(
        re.escape(keyword)
        for keyword in keywords
    )

    mask = customer_text.str.contains(
        pattern,
        regex=True,
        na=False
    )

    count = mask.sum()

    coverage.append({
        "intent": intent,
        "matching_messages": count,
        "percentage": round(
            count / len(df) * 100,
            2
        )
    })

coverage_df = (
    pd.DataFrame(coverage)
    .sort_values(
        "matching_messages",
        ascending=False
    )
    .reset_index(drop=True)
)

display(coverage_df)

,intent,matching_messages,percentage
0,apps_app_store,82969,77.89
1,ios_updates,36236,34.02
2,battery_charging,10604,9.95
3,display_touchscreen,6165,5.79
4,wifi_connectivity,4998,4.69
5,apple_music,3739,3.51
6,camera_photos,3593,3.37
7,apple_id_account,2642,2.48
8,audio,2510,2.36
9,icloud_backup,2330,2.19


In [6]:
import pandas as pd

# Create a reproducible inspection sample.
# We will manually review these examples before finalizing intents.

sample_size = 100

inspection_sample = (
    df.sample(
        n=sample_size,
        random_state=42
    )
    [
        [
            "customer_tweet_id",
            "customer_message",
            "previous_message",
            "brand_response"
        ]
    ]
    .reset_index(drop=True)
)

# Add an empty column for our manual intent label.
inspection_sample["intent"] = ""

# Add an optional notes column for ambiguous cases.
inspection_sample["notes"] = ""

display(inspection_sample)

,customer_tweet_id,customer_message,previous_message,brand_response,intent,notes
0,1533007.0,@115858 what’s up with this nonstop “I” issue?...,NaN,@475474 Let's look into this together. Reach o...,,
1,1224310.0,@AppleSupport my iPhone 7 screen is all white ...,NaN,@407226 Hi there! Let's try out these steps: h...,,
2,2000766.0,@461407 @115858 I️ mean write the letter sorry...,@461407 @115858 Mine keeps doing that too! Its...,@591615 We can check it out. Meet us in DM wit...,,
3,1808936.0,@AppleSupport Hi. There is a speaker grille on...,NaN,@326564 Running into audio issues? Let's get t...,,
4,675393.0,Is there some kind of a bug causing random blu...,NaN,@218074 We'd like to help. DM us what iOS vers...,,
...,...,...,...,...,...,...
95,291441.0,@AppleSupport upgraded my iPhone to the latest...,NaN,@185391 We want to help you resolve this. Whic...,,
96,2549529.0,"@AppleSupport Hey, the display on my 2015 MacB...",NaN,@724386 We're here to help! Join us in DM with...,,
97,749784.0,@AppleSupport IPhone 7 Plus And iOS 11.0.3,@299222 Thanks for that info. Which device are...,@299222 Let's look into this further in DM.,,
98,2618338.0,@739950 @AppleSupport Same here!!! iPhone 7 Plus,@AppleSupport I have 11.1.2 and have that prob...,@739951 Let's look into this battery life issu...,,


In [7]:
# Test our candidate intent rules against the 100-example sample.
# This is exploratory only — these are NOT final labels.

def find_matching_intents(message):
    message = str(message).lower()

    matches = []

    for intent, keywords in intent_keywords.items():
        for keyword in keywords:
            if keyword.lower() in message:
                matches.append(intent)
                break

    return matches


inspection_sample["keyword_matches"] = (
    inspection_sample["customer_message"]
    .apply(find_matching_intents)
)

# Number of detected intents per example
inspection_sample["num_matches"] = (
    inspection_sample["keyword_matches"]
    .str.len()
)

print("Examples matching exactly one candidate intent:",
      (inspection_sample["num_matches"] == 1).sum())

print("Examples matching multiple candidate intents:",
      (inspection_sample["num_matches"] > 1).sum())

print("Examples matching no candidate intent:",
      (inspection_sample["num_matches"] == 0).sum())

print("\nExamples with multiple matches:")
display(
    inspection_sample[
        inspection_sample["num_matches"] > 1
    ][
        [
            "customer_message",
            "keyword_matches",
            "brand_response"
        ]
    ]
)

print("\nExamples with no matches:")
display(
    inspection_sample[
        inspection_sample["num_matches"] == 0
    ][
        [
            "customer_message",
            "brand_response"
        ]
    ]
)

Examples matching exactly one candidate intent: 43
Examples matching multiple candidate intents: 42
Examples matching no candidate intent: 15

Examples with multiple matches:


,customer_message,keyword_matches,brand_response
1,@AppleSupport my iPhone 7 screen is all white ...,"[apps_app_store, display_touchscreen]",@407226 Hi there! Let's try out these steps: h...
3,@AppleSupport Hi. There is a speaker grille on...,"[apps_app_store, audio]",@326564 Running into audio issues? Let's get t...
4,Is there some kind of a bug causing random blu...,"[apps_app_store, camera_photos]",@218074 We'd like to help. DM us what iOS vers...
6,@AppleSupport Several apps my phone already bl...,"[apps_app_store, display_touchscreen]",@761386 Please DM us so we can gather further ...
9,@AppleSupport @115858 left AirPod trouble -&gt...,"[wifi_connectivity, apps_app_store, airpods]",@424256 Which device do you use with your AirP...
15,#run actually a lot more. But #AppleWatch3 sto...,"[apps_app_store, apple_watch]",@391996 We'd be happy to help. Send us a DM wi...
16,@115858 why do I have to re-enter my wifi pass...,"[ios_updates, wifi_connectivity, apple_id_acco...",@424136 We're here to help. Which device are y...
18,"After that I updated my iphone, my battery is ...","[battery_charging, ios_updates, apps_app_store]",@285471 Let's take a look at your iPhone's bat...
19,"iPhone 6 randomly restarts, camera roll freque...","[ios_updates, apps_app_store, camera_photos, d...",@714360 That's definitely not the experience w...
20,Dear @AppleSupport @115858 I restored my iPhon...,"[wifi_connectivity, apps_app_store]",@287481 We're here to help! What steps exactly...



Examples with no matches:


,customer_message,brand_response
0,@115858 what’s up with this nonstop “I” issue?...,@475474 Let's look into this together. Reach o...
2,@461407 @115858 I️ mean write the letter sorry...,@591615 We can check it out. Meet us in DM wit...
8,@115858 ya’ll need to get it together 🙄 I’m si...,@495875 Thanks for reaching out to us. We have...
11,My phone has been doing this for two months. C...,@567796 We'd be happy to look into this with y...
21,"@115858 fixes the issue with the I, now we hav...",@346535 An update has been released to assist ...
24,"After a year of use, my @115858 smarak keyboar...",@808828 We'd like to look into this. Send us a...
26,@115858 What’s going on guys? This bug thing i...,@618663 We'd like to ensure we're on the same ...
27,@115858 Unless you listen to an episode straig...,@410321 We're happy to assist. Which iOS versi...
32,@115858 ARE YOU GOING TO FIX THIS?!?!?!?! It's...,@321627 We'd love to work with you and get thi...
38,My phone officially only has a 30 minute lifes...,@481365 Let's look into that. Send us a DM and...


In [8]:
# Inspect 50 examples for manual taxonomy validation.

taxonomy_review = (
    df[
        [
            "customer_tweet_id",
            "customer_message",
            "previous_message",
            "brand_response"
        ]
    ]
    .sample(
        50,
        random_state=123
    )
    .reset_index(drop=True)
)

display(taxonomy_review)

,customer_tweet_id,customer_message,previous_message,brand_response
0,1294331.0,@AppleSupport I lock it and sometimes that doe...,@216977 Thanks. Which exact iOS version are y...,"@216977 After you make sure you have a backup,..."
1,2138914.0,@115858 why when I️ type in #IPHONEX does it t...,NaN,@629072 Here’s what you can do to work around ...
2,1766797.0,@AppleSupport why do I keep getting this messa...,NaN,"@531305 We want to help you. DM us, and we'll ..."
3,1879894.0,@AppleSupport compare the time stamps and the ...,NaN,@561080 We certainly want you to be able to en...
4,290528.0,@115858 @AppleSupport updated iPhone to new iO...,NaN,@185189 We'd like to take a look into this wit...
5,1143124.0,Alright @115858 the new update has really mess...,NaN,@389056 We are here for you. Let's take a look...
6,1752028.0,@AppleSupport My iphone6 has (compared to othe...,NaN,@527842 Let's work together to figure it out. ...
7,1386419.0,@AppleSupport running 11.0.3 and have very lou...,NaN,@442610 Let's check into this together. When d...
8,2305160.0,"@115858 yo fix this glitch, it makes me look l...",NaN,@668812 We'd like to help. Tell us more about ...
9,2983153.0,@115858 all of my messages just got deleted. v...,NaN,@822632 Let's look into this together. To star...


In [9]:
intent_definitions = {
    "ios_update": "Problems or questions specifically about iOS updates, update failures, update-related bugs, or iOS version behavior.",

    "battery_charging": "Battery draining, battery health, charging, power consumption, or unexpected battery percentage changes.",

    "connectivity": "Wi-Fi, cellular/mobile data, Bluetooth, network, or connection problems.",

    "account_authentication": "Apple ID, login, password, authentication, account access, subscriptions/account management.",

    "icloud_backup": "iCloud storage, iCloud backup, missing/syncing iCloud data, or iCloud-related problems.",

    "apps_app_store": "Problems with apps, installing/updating apps, App Store, app crashes, or application behavior.",

    "audio": "Problems with speakers, microphone, calls audio, sound, headphones, AirPods, or other audio functionality.",

    "camera_photos": "Camera, taking photos/videos, photo library, photo syncing, or photo-related problems.",

    "display_input": "Screen/display, touchscreen, keyboard, typing, text input, or visual/input interaction problems.",

    "apple_services": "Problems with Apple services such as Apple Music or other Apple-provided services that do not fit another intent.",

    "device_hardware": "Physical/device-level problems that cannot be better classified into a more specific intent.",

    "other_unclear": "Too vague, insufficient context, unrelated, or impossible to confidently assign to another intent."
}

for intent, definition in intent_definitions.items():
    print(f"{intent}:")
    print(f"  {definition}\n")

ios_update:
  Problems or questions specifically about iOS updates, update failures, update-related bugs, or iOS version behavior.

battery_charging:
  Battery draining, battery health, charging, power consumption, or unexpected battery percentage changes.

connectivity:
  Wi-Fi, cellular/mobile data, Bluetooth, network, or connection problems.

account_authentication:
  Apple ID, login, password, authentication, account access, subscriptions/account management.

icloud_backup:
  iCloud storage, iCloud backup, missing/syncing iCloud data, or iCloud-related problems.

apps_app_store:
  Problems with apps, installing/updating apps, App Store, app crashes, or application behavior.

audio:
  Problems with speakers, microphone, calls audio, sound, headphones, AirPods, or other audio functionality.

camera_photos:
  Camera, taking photos/videos, photo library, photo syncing, or photo-related problems.

display_input:
  Screen/display, touchscreen, keyboard, typing, text input, or visual/inpu

In [10]:
import pandas as pd

clean_apple = pd.read_csv(
    "../data/processed/apple_support_clean.csv"
)

print("Shape:", clean_apple.shape)
display(clean_apple.head())

Shape: (106523, 6)


,customer_tweet_id,customer_message,previous_message,has_context,parent_author,brand_response
0,698.0,@AppleSupport https://t.co/NV0yucs0lB,@AppleSupport why are my I️’s changing not sho...,True,115854,@115854 We're here for you. Which version of t...
1,697.0,@AppleSupport The newest update. I️ made sure ...,@115854 We're here for you. Which version of t...,True,AppleSupport,@115854 Lets take a closer look into this issu...
2,702.0,@AppleSupport Tried resetting my settings .. r...,@115855 Any steps tried since it started last ...,True,AppleSupport,@115855 Let's go to DM for the next steps. DM ...
3,704.0,@AppleSupport This is what it looks like https...,@115855 That's great it has iOS 11.1 as we can...,True,AppleSupport,@115855 Any steps tried since it started last ...
4,707.0,@AppleSupport I️ have an iPhone 7 Plus and yes...,@115855 We'd like to look into this with you. ...,True,AppleSupport,@115855 That's great it has iOS 11.1 as we can...


In [11]:
review_sample = (
    clean_apple[
        [
            "customer_tweet_id",
            "customer_message",
            "previous_message",
            "brand_response"
        ]
    ]
    .sample(
        100,
        random_state=456
    )
    .reset_index(drop=True)
)

display(review_sample)

,customer_tweet_id,customer_message,previous_message,brand_response
0,963246.0,@AppleSupport iPhone 7 having crazy battery pr...,NaN,@348589 Let's look into that. Send us a DM and...
1,2686810.0,@AppleSupport please can you advise on the qui...,NaN,@567932 We're happy to do everything we can to...
2,2202096.0,@AppleSupport Still waiting-What is going on w...,NaN,@643962 We’re here to help. Let’s team up in D...
3,852993.0,"Dear @AppleSupport, please note that it has no...",NaN,@322629 We'll be glad to look into this. Pleas...
4,2346056.0,@AppleSupport can you guys fix the problem whe...,NaN,@467249 That’s not good to hear about your con...
...,...,...,...,...
95,1813546.0,"@AppleSupport IOS 11 ruins battery life 🙄,ple...",NaN,@544820 If you're having trouble with your bat...
96,1376460.0,@115858 also has done THIS to my camera. and w...,@115858 #ios11 stopped my phone making/receivi...,@440427 Let's meet up in DM to explore support...
97,1118226.0,@AppleSupport I am on a Samsung galaxy 8,@383854 We'd like to look into this. What devi...,@383854 Got it. Let's start by following the s...
98,568429.0,@AppleSupport This is extremely frustrating,@AppleSupport That was 5 min ago,@252896 We may be having an issue where we're ...


In [12]:
manual_review = review_sample.copy()

manual_review["intent"] = ""
manual_review["notes"] = ""

display(manual_review)

,customer_tweet_id,customer_message,previous_message,brand_response,intent,notes
0,963246.0,@AppleSupport iPhone 7 having crazy battery pr...,NaN,@348589 Let's look into that. Send us a DM and...,,
1,2686810.0,@AppleSupport please can you advise on the qui...,NaN,@567932 We're happy to do everything we can to...,,
2,2202096.0,@AppleSupport Still waiting-What is going on w...,NaN,@643962 We’re here to help. Let’s team up in D...,,
3,852993.0,"Dear @AppleSupport, please note that it has no...",NaN,@322629 We'll be glad to look into this. Pleas...,,
4,2346056.0,@AppleSupport can you guys fix the problem whe...,NaN,@467249 That’s not good to hear about your con...,,
...,...,...,...,...,...,...
95,1813546.0,"@AppleSupport IOS 11 ruins battery life 🙄,ple...",NaN,@544820 If you're having trouble with your bat...,,
96,1376460.0,@115858 also has done THIS to my camera. and w...,@115858 #ios11 stopped my phone making/receivi...,@440427 Let's meet up in DM to explore support...,,
97,1118226.0,@AppleSupport I am on a Samsung galaxy 8,@383854 We'd like to look into this. What devi...,@383854 Got it. Let's start by following the s...,,
98,568429.0,@AppleSupport This is extremely frustrating,@AppleSupport That was 5 min ago,@252896 We may be having an issue where we're ...,,


In [13]:
for i, row in manual_review.iterrows():
    print(f"\n{'=' * 80}")
    print(f"INDEX: {i}")
    print(f"CUSTOMER: {row['customer_message']}")
    
    if pd.notna(row["previous_message"]) and str(row["previous_message"]).strip():
        print(f"PREVIOUS: {row['previous_message']}")
    
    print(f"BRAND RESPONSE: {row['brand_response']}")


INDEX: 0
CUSTOMER: @AppleSupport iPhone 7 having crazy battery problems as well as connectivity issues since iOS update. What do I do?
BRAND RESPONSE: @348589 Let's look into that. Send us a DM and we'll go from there. https://t.co/GDrqU22YpT

INDEX: 1
CUSTOMER: @AppleSupport please can you advise on the quickest way to make a complaint against apple?
BRAND RESPONSE: @567932 We're happy to do everything we can to help. To start, DM us with the country where you're located. https://t.co/GDrqU22YpT

INDEX: 2
CUSTOMER: @AppleSupport Still waiting-What is going on with the new iOS? My wife cannot send video to my Samsung since updating to the latest software
BRAND RESPONSE: @643962 We’re here to help. Let’s team up in DM and we’ll do our best to assist you. https://t.co/GDrqU22YpT

INDEX: 3
CUSTOMER: Dear @AppleSupport, please note that it has now been a week since searching on Maps on iOS 11 has worked on the  @37275 mobile net
BRAND RESPONSE: @322629 We'll be glad to look into this. Ple

In [14]:
primary_intent_rules = {
    "ios_update": "Use when the main problem is the iOS update itself, update failure, or a problem explicitly caused by updating.",

    "battery_charging": "Use when the main problem is battery drain, battery health, charging, or power consumption.",

    "connectivity": "Use when the main problem is Wi-Fi, cellular/mobile data, Bluetooth, or network connectivity.",

    "account_authentication": "Use when the main problem is Apple ID, login, password, authentication, or account access.",

    "icloud_backup": "Use when the main problem is iCloud, iCloud backup, iCloud storage, or missing/syncing iCloud data.",

    "apps_app_store": "Use when the main problem is an application or the App Store.",

    "audio": "Use when the main problem is sound, speaker, microphone, call audio, headphones, or AirPods.",

    "camera_photos": "Use when the main problem is the camera, photos, videos, or photo library.",

    "display_input": "Use when the main problem is the screen, touchscreen, keyboard, typing, or other input/display behavior.",

    "apple_services": "Use when the main problem is an Apple service such as Apple Music and does not fit a more specific category.",

    "device_hardware": "Use for physical/device-level problems that do not fit another specific intent.",

    "other_unclear": "Use when the message is too vague, lacks enough information, is mainly a complaint with no identifiable issue, or does not fit another intent."
}

for i, (intent, rule) in enumerate(primary_intent_rules.items(), start=1):
    print(f"{i}. {intent}")
    print(f"   {rule}\n")

1. ios_update
   Use when the main problem is the iOS update itself, update failure, or a problem explicitly caused by updating.

2. battery_charging
   Use when the main problem is battery drain, battery health, charging, or power consumption.

3. connectivity
   Use when the main problem is Wi-Fi, cellular/mobile data, Bluetooth, or network connectivity.

4. account_authentication
   Use when the main problem is Apple ID, login, password, authentication, or account access.

5. icloud_backup
   Use when the main problem is iCloud, iCloud backup, iCloud storage, or missing/syncing iCloud data.

6. apps_app_store
   Use when the main problem is an application or the App Store.

7. audio
   Use when the main problem is sound, speaker, microphone, call audio, headphones, or AirPods.

8. camera_photos
   Use when the main problem is the camera, photos, videos, or photo library.

9. display_input
   Use when the main problem is the screen, touchscreen, keyboard, typing, or other input/displ

In [15]:
# Create a 200-example sample for manual intent labeling and save it for later use.

labeling_sample = (
    clean_apple[
        [
            "customer_tweet_id",
            "customer_message",
            "previous_message",
            "brand_response"
        ]
    ]
    .sample(
        200,
        random_state=789
    )
    .reset_index(drop=True)
)

labeling_sample["intent"] = ""
labeling_sample["notes"] = ""

labeling_path = "../data/processed/apple_intent_labeling_200.csv"

labeling_sample.to_csv(
    labeling_path,
    index=False
)

print(f"Created: {labeling_path}")
print(f"Rows: {len(labeling_sample)}")

display(labeling_sample.head(10))

Created: ../data/processed/apple_intent_labeling_200.csv
Rows: 200


,customer_tweet_id,customer_message,previous_message,brand_response,intent,notes
0,909813.0,"Seriously @115858, my iPhone’s heating up agai...",NaN,@336101 Send us a DM so we can help with your ...,,
1,1864674.0,@115858 needs to setup a subscription plan bec...,NaN,@525234 We'd like to help with your headphones...,,
2,2070017.0,"@612173 @AppleSupport Two month old phone, the...",@AppleSupport #apple #iphone #ois11 never had ...,@612172 We're here to help. DM us which model ...,,
3,554724.0,@AppleSupport i have an issue with my phone wh...,NaN,@249474 We know that the performance of your d...,,
4,1684485.0,Why does my phone not know the letter I️ anymo...,NaN,@511907 Here’s what you can do to work around ...,,
5,2151155.0,"WHEN ARE YALL GONNA FIX THE I️-glitch, like th...",NaN,@311677 Here’s what you can do to work around ...,,
6,2047763.0,@AppleSupport Crackle gone - replaced by compl...,@AppleSupport every update makes my phone wors...,@574265 We'd be happy to look into this with y...,,
7,1809282.0,@AppleSupport @ATT Oh AND deletes all my texts...,I love having to pay $150 to pay off a phone t...,@339428 This isn't expected behavior for your ...,,
8,2385140.0,Dear @115858 @AppleSupport Why has iOS11 turne...,NaN,@579638 This is certainly not the experience w...,,
9,1971418.0,@AppleSupport I*,Yo @AppleSupport the letter “I️” is broken htt...,@583952 Let's check this out together. Follow ...,,


In [16]:
# Display the 200 labeling examples in batches of 20 for manual taxonomy review.

batch_size = 20

for start in range(0, len(labeling_sample), batch_size):
    end = min(start + batch_size, len(labeling_sample))

    print(f"\n{'=' * 100}")
    print(f"Examples {start}–{end - 1}")
    print(f"{'=' * 100}")

    display(
        labeling_sample.loc[
            start:end - 1,
            [
                "customer_tweet_id",
                "customer_message",
                "previous_message"
            ]
        ]
    )


Examples 0–19


,customer_tweet_id,customer_message,previous_message
0,909813.0,"Seriously @115858, my iPhone’s heating up agai...",NaN
1,1864674.0,@115858 needs to setup a subscription plan bec...,NaN
2,2070017.0,"@612173 @AppleSupport Two month old phone, the...",@AppleSupport #apple #iphone #ois11 never had ...
3,554724.0,@AppleSupport i have an issue with my phone wh...,NaN
4,1684485.0,Why does my phone not know the letter I️ anymo...,NaN
5,2151155.0,"WHEN ARE YALL GONNA FIX THE I️-glitch, like th...",NaN
6,2047763.0,@AppleSupport Crackle gone - replaced by compl...,@AppleSupport every update makes my phone wors...
7,1809282.0,@AppleSupport @ATT Oh AND deletes all my texts...,I love having to pay $150 to pay off a phone t...
8,2385140.0,Dear @115858 @AppleSupport Why has iOS11 turne...,NaN
9,1971418.0,@AppleSupport I*,Yo @AppleSupport the letter “I️” is broken htt...



Examples 20–39


,customer_tweet_id,customer_message,previous_message
20,1773429.0,I️ fucking can’t with this “I️” shit. @115858 ...,NaN
21,9840.0,@AppleSupport mi iPhone no se puede conectar a...,NaN
22,2493938.0,Idk what the hell going on with my phone but y...,NaN
23,1509979.0,@AppleSupport Yup every app has been updated a...,@470477 Were you able to check for updates in ...
24,1058940.0,@115858 where the fuck did my 4000 photos go h...,NaN
25,2019482.0,@AppleSupport ios11 sucks. My iphone SE is a e...,NaN
26,208620.0,Half of my contact list is gone after the upgr...,NaN
27,487849.0,No entiendo por qué tengo que actualizar mi te...,NaN
28,1577350.0,"Uh, I was never the best mathmatist but isn't ...",NaN
29,1676324.0,@115858 y’all got me fucked up for making this...,NaN



Examples 40–59


...

,customer_tweet_id,customer_message,previous_message
140,822213.0,My phone has been stuck like this all day @115...,NaN
141,2429248.0,Anyone having problems with their Mac apps sin...,NaN
142,1094142.0,@115858 IOS 11.0.3 has made me hate my iPhone ...,NaN
143,1432507.0,"@AppleSupport It happen with headphones on, an...",@316439 We're here for you. Are you using the ...
144,1804880.0,@AppleSupport Sending messages through my watc...,@AppleSupport my phone screen has been unrespo...
145,2205866.0,Why doesn’t the Bluetooth button in’t control ...,NaN
146,193631.0,Hmmm 🤔 so iOS 11.0.1 was supposed to bring alo...,NaN
147,486234.0,@AppleSupport No but it does not look right ma...,@230682 Image quality is not something we can ...
148,2354996.0,@AppleSupport I Thik there is something proble...,NaN
149,1386687.0,lmk @AppleSupport https://t.co/XhPks6CQi2,NaN



Examples 160–179


,customer_tweet_id,customer_message,previous_message
160,2122599.0,@AppleSupport iPhone 8 - Music bar doesn’t mov...,NaN
161,1679282.0,This new @115858 update is messing with the pr...,NaN
162,1433710.0,@115858 @AppleSupport y’all need to fix this p...,NaN
163,1892478.0,@115858 my phone is MUCH slower and lags easil...,NaN
164,2571358.0,@AppleSupport iTunes!,@729179 We can look into this for you. Is your...
165,1681569.0,@AppleSupport fix this please https://t.co/hVJ...,NaN
166,1417394.0,ios11 messed up my camera roll @115858 fix thi...,NaN
167,1085261.0,@375998 @AppleSupport Ever since I updated my ...,help i’ve had this iphone 7+ since July and it...
168,889128.0,@AppleSupport I have and I know that @135616 u...,@331089 Thank you for letting us know. Have yo...
169,2889499.0,@115858 can you help me with a problem I have,NaN



Examples 180–199


,customer_tweet_id,customer_message,previous_message
180,763663.0,@AppleSupport No steps. Just using the photo a...,@302496 Thanks. Can you tell us which steps yo...
181,621812.0,@AppleSupport iPhone 8 Plus upgraded to iOS 11...,NaN
182,2837216.0,@AppleSupport hello I have recently bought the...,NaN
183,1549441.0,My wireless @118845 running headphones just br...,NaN
184,989691.0,I am unable to update applications and install...,NaN
185,611252.0,So the short story is that @AppleSupport is ha...,NaN
186,1697447.0,"@115858 I️ appreciate the 80 new emojis, ya kn...",NaN
187,1979783.0,I’m a long time @115858 apologist but this new...,NaN
188,1100850.0,@AppleSupport any way to check my iphones over...,NaN
189,762545.0,Since upgrading to #iOS11 my call volume on 6s...,NaN


In [17]:
# Create a clean table containing only the fields needed for manual intent labeling.

labeling_sample = labeling_sample[
    [
        "customer_tweet_id",
        "customer_message",
        "previous_message",
        "intent",
        "notes"
    ]
].copy()

display(labeling_sample.head(10))

,customer_tweet_id,customer_message,previous_message,intent,notes
0,909813.0,"Seriously @115858, my iPhone’s heating up agai...",NaN,,
1,1864674.0,@115858 needs to setup a subscription plan bec...,NaN,,
2,2070017.0,"@612173 @AppleSupport Two month old phone, the...",@AppleSupport #apple #iphone #ois11 never had ...,,
3,554724.0,@AppleSupport i have an issue with my phone wh...,NaN,,
4,1684485.0,Why does my phone not know the letter I️ anymo...,NaN,,
5,2151155.0,"WHEN ARE YALL GONNA FIX THE I️-glitch, like th...",NaN,,
6,2047763.0,@AppleSupport Crackle gone - replaced by compl...,@AppleSupport every update makes my phone wors...,,
7,1809282.0,@AppleSupport @ATT Oh AND deletes all my texts...,I love having to pay $150 to pay off a phone t...,,
8,2385140.0,Dear @115858 @AppleSupport Why has iOS11 turne...,NaN,,
9,1971418.0,@AppleSupport I*,Yo @AppleSupport the letter “I️” is broken htt...,,


In [18]:
# Create an interactive helper for assigning an intent to each customer message.

valid_intents = list(intent_definitions.keys())

def show_labeling_example(index):
    row = labeling_sample.loc[index]

    print(f"INDEX: {index}")
    print(f"\nCUSTOMER MESSAGE:\n{row['customer_message']}")

    if pd.notna(row["previous_message"]) and str(row["previous_message"]).strip():
        print(f"\nPREVIOUS MESSAGE:\n{row['previous_message']}")

    print("\nVALID INTENTS:")
    for i, intent in enumerate(valid_intents, start=1):
        print(f"{i}. {intent}")

In [19]:
# Test the labeling helper on the first example before starting the full labeling process.

show_labeling_example(0)

INDEX: 0

CUSTOMER MESSAGE:
Seriously @115858, my iPhone’s heating up again,battery life has gone from 88% to 62% in a blink of an eye! Keypad freezing etc etc 😡😤 #Fedup

VALID INTENTS:
1. ios_update
2. battery_charging
3. connectivity
4. account_authentication
5. icloud_backup
6. apps_app_store
7. audio
8. camera_photos
9. display_input
10. apple_services
11. device_hardware
12. other_unclear


In [20]:
# Create a safe manual labeling loop that saves every label immediately to the CSV file.

def label_examples(data, start_index=0):
    for index in range(start_index, len(data)):
        row = data.loc[index]

        print("\n" + "=" * 90)
        print(f"INDEX: {index}")
        print(f"\nCUSTOMER:\n{row['customer_message']}")

        if pd.notna(row["previous_message"]) and str(row["previous_message"]).strip():
            print(f"\nPREVIOUS:\n{row['previous_message']}")

        print("\nINTENTS:")
        for i, intent in enumerate(valid_intents, start=1):
            print(f"{i}. {intent}")

        while True:
            choice = input("\nEnter intent number (1-12), or 'q' to stop: ").strip()

            if choice.lower() == "q":
                data.to_csv(labeling_path, index=False)
                print(f"\nProgress saved. Stopped at index {index}.")
                return

            if choice.isdigit() and 1 <= int(choice) <= len(valid_intents):
                selected_intent = valid_intents[int(choice) - 1]
                break

            print("Invalid choice. Enter a number from 1 to 12.")

        notes = input("Notes (optional): ").strip()

        data.loc[index, "intent"] = selected_intent
        data.loc[index, "notes"] = notes

        # Save immediately after every label.
        data.to_csv(labeling_path, index=False)

        print(f"Saved: {selected_intent}")

    print("\nAll examples have been labeled.")

In [21]:
# Verify that the 200-row labeling file was preserved after the kernel restart.

import pandas as pd

labeling_path = "../data/processed/apple_intent_labeling_200.csv"

labeling_check = pd.read_csv(labeling_path)

print("Rows:", len(labeling_check))
print("Columns:", labeling_check.columns.tolist())
print("Labels already assigned:", labeling_check["intent"].notna().sum())

Rows: 200
Columns: ['customer_tweet_id', 'customer_message', 'previous_message', 'brand_response', 'intent', 'notes']
Labels already assigned: 0


In [22]:
# Confirm that the cleaned dataset and 200-example labeling sample are loaded correctly.

print("Clean AppleSupport dataset:", clean_apple.shape)
print("200-example labeling sample:", labeling_sample.shape)
print("Currently labeled:", labeling_sample["intent"].notna().sum())

Clean AppleSupport dataset: (106523, 6)
200-example labeling sample: (200, 5)
Currently labeled: 200


In [23]:
# Check the current intent labels and make sure the 200-example dataset was labeled correctly.

print("Rows:", len(labeling_sample))
print("Missing labels:", labeling_sample["intent"].isna().sum())
print("\nIntent distribution:")
display(labeling_sample["intent"].value_counts(dropna=False))

Rows: 200
Missing labels: 0

Intent distribution:


intent
    200
Name: count, dtype: int64

In [24]:
# Inspect the unique values currently stored in the intent column.

print("Unique intent values:")
print(labeling_sample["intent"].unique())

print("\nFirst 10 intent values:")
display(labeling_sample[["customer_tweet_id", "intent"]].head(10))

Unique intent values:
<StringArray>
['']
Length: 1, dtype: str

First 10 intent values:


,customer_tweet_id,intent
0,909813.0,
1,1864674.0,
2,2070017.0,
3,554724.0,
4,1684485.0,
5,2151155.0,
6,2047763.0,
7,1809282.0,
8,2385140.0,
9,1971418.0,


In [25]:
# Check how many intent labels are actually assigned and confirm the labeling sample is ready.

labeled_count = (
    labeling_sample["intent"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)

print("Total examples:", len(labeling_sample))
print("Actually labeled:", labeled_count)
print("Remaining:", len(labeling_sample) - labeled_count)

Total examples: 200
Actually labeled: 0
Remaining: 200


In [26]:
# Create a 50-example manual review set for validating the intent taxonomy.

taxonomy_review = (
    labeling_sample[
        [
            "customer_tweet_id",
            "customer_message",
            "previous_message"
        ]
    ]
    .sample(
        50,
        random_state=1234
    )
    .reset_index(drop=True)
)

display(taxonomy_review)

,customer_tweet_id,customer_message,previous_message
0,1130498.0,"@AppleSupport Apps constantly crash, iTunes do...",@386423 We'd be happy to see how we can help. ...
1,1039193.0,@AppleSupport 11.0.3 (15A432),@365861 You're in the right place for help. To...
2,325899.0,@AppleSupport Already update but it doesn’t work.,@179912 Let's create a backup of your data and...
3,1536257.0,@476342 @AppleSupport what’s goin on,@476341 mines doing the same thing
4,810342.0,@AppleSupport Can iOS 11 search for what I hav...,"In iOS 11 on #iPad, all of your files are now ..."
5,92764.0,@AppleSupport No app updates. And I will test ...,@131555 We'd be happy to look into this with y...
6,2151155.0,"WHEN ARE YALL GONNA FIX THE I️-glitch, like th...",NaN
7,1809282.0,@AppleSupport @ATT Oh AND deletes all my texts...,I love having to pay $150 to pay off a phone t...
8,397530.0,@AppleSupport WHY DOES THESE QUESTION MARKS KE...,NaN
9,2371254.0,@AppleSupport https://t.co/XGBPN7HSYO not work...,NaN


In [27]:
# Display the 50 selected examples with context so we can validate the intent taxonomy.

for index, row in taxonomy_review.iterrows():
    print("\n" + "=" * 90)
    print(f"INDEX: {index}")
    print(f"\nCUSTOMER:\n{row['customer_message']}")

    if pd.notna(row["previous_message"]) and str(row["previous_message"]).strip():
        print(f"\nPREVIOUS:\n{row['previous_message']}")


INDEX: 0

CUSTOMER:
@AppleSupport Apps constantly crash, iTunes doesn’t work on the lock screen, it refuses to update to the latest OS &amp; i get a black screen all the time

PREVIOUS:
@386423 We'd be happy to see how we can help. What's going on?

INDEX: 1

CUSTOMER:
@AppleSupport 11.0.3 (15A432)

PREVIOUS:
@365861 You're in the right place for help. To start, may we get the iOS version number shown in Settings &gt; General &gt; About? https://t.co/GDrqU22YpT

INDEX: 2

CUSTOMER:
@AppleSupport Already update but it doesn’t work.

PREVIOUS:
@179912 Let's create a backup of your data and update to iOS 11.0.2 to see if that helps. Here's how: https://t.co/80YRnjDFDk

INDEX: 3

CUSTOMER:
@476342 @AppleSupport what’s goin on

PREVIOUS:
@476341 mines doing the same thing

INDEX: 4

CUSTOMER:
@AppleSupport Can iOS 11 search for what I have wrote in Notes? I remember that wwdc 2017 someone showed it for us

PREVIOUS:
In iOS 11 on #iPad, all of your files are now in one handy place. Here's a

In [28]:
# Create a compact view of the 50 examples so the full sample can be reviewed without notebook output truncation.

taxonomy_compact = taxonomy_review.copy()

taxonomy_compact["customer_message"] = (
    taxonomy_compact["customer_message"]
    .fillna("")
    .str.replace(r"\s+", " ", regex=True)
    .str.slice(0, 250)
)

taxonomy_compact["previous_message"] = (
    taxonomy_compact["previous_message"]
    .fillna("")
    .str.replace(r"\s+", " ", regex=True)
    .str.slice(0, 200)
)

display(taxonomy_compact)

,customer_tweet_id,customer_message,previous_message
0,1130498.0,"@AppleSupport Apps constantly crash, iTunes do...",@386423 We'd be happy to see how we can help. ...
1,1039193.0,@AppleSupport 11.0.3 (15A432),@365861 You're in the right place for help. To...
2,325899.0,@AppleSupport Already update but it doesn’t work.,@179912 Let's create a backup of your data and...
3,1536257.0,@476342 @AppleSupport what’s goin on,@476341 mines doing the same thing
4,810342.0,@AppleSupport Can iOS 11 search for what I hav...,"In iOS 11 on #iPad, all of your files are now ..."
5,92764.0,@AppleSupport No app updates. And I will test ...,@131555 We'd be happy to look into this with y...
6,2151155.0,"WHEN ARE YALL GONNA FIX THE I️-glitch, like th...",
7,1809282.0,@AppleSupport @ATT Oh AND deletes all my texts...,I love having to pay $150 to pay off a phone t...
8,397530.0,@AppleSupport WHY DOES THESE QUESTION MARKS KE...,
9,2371254.0,@AppleSupport https://t.co/XGBPN7HSYO not work...,


In [29]:
# Count common Apple service-related terms in the cleaned customer messages to decide whether this category needs refinement.

service_keywords = {
    "apple_music": r"\bmusic\b|apple music",
    "itunes": r"\bitunes\b",
    "imessage": r"\bimessage\b",
    "maps": r"\bmaps?\b",
    "facetime": r"\bfacetime\b",
    "siri": r"\bsiri\b",
    "airdrop": r"\bairdrop\b",
}

for service, pattern in service_keywords.items():
    count = clean_apple["customer_message"].str.contains(
        pattern,
        case=False,
        na=False,
        regex=True
    ).sum()

    print(f"{service:15} {count:,}")

apple_music     3,039
itunes          1,661
imessage        645
maps            299
facetime        338
siri            605
airdrop         108


In [30]:
# Estimate the prevalence of each intent using broad keyword signals to identify rare or missing categories.

intent_signals = {
    "ios_update": r"\bios\b|ios\s*11|update|updated|updating|upgrade|upgraded",
    "battery_charging": r"battery|charging|charge|overheat|heating",
    "connectivity": r"wifi|wi-fi|bluetooth|airdrop|network|signal|internet|cellular|mobile data",
    "account_authentication": r"apple id|password|login|log in|sign in|account|authentication",
    "icloud_backup": r"icloud|backup|back up|sync",
    "apps_app_store": r"\bapp\b|app store|application|download|install|crash",
    "audio": r"speaker|microphone|mic|sound|audio|headphone|airpod|hear|call volume",
    "camera_photos": r"camera|photo|photos|picture|pictures|video",
    "display_input": r"screen|display|keyboard|keypad|touch|type|typing|letter|question mark|glitch",
    "apple_services": r"music|itunes|imessage|facetime|maps|siri",
    "device_hardware": r"iphone|ipad|macbook|mac|device|phone"
}

for intent, pattern in intent_signals.items():
    count = clean_apple["customer_message"].str.contains(
        pattern,
        case=False,
        na=False,
        regex=True
    ).sum()

    percentage = count / len(clean_apple) * 100

    print(f"{intent:25} {count:7,} ({percentage:5.2f}%)")

ios_update                 36,115 (33.90%)
battery_charging           10,707 (10.05%)
connectivity                4,288 ( 4.03%)
account_authentication      2,654 ( 2.49%)
icloud_backup               2,531 ( 2.38%)
apps_app_store             10,100 ( 9.48%)
audio                       2,914 ( 2.74%)
camera_photos               3,760 ( 3.53%)
display_input              16,260 (15.26%)
apple_services              6,469 ( 6.07%)
device_hardware            45,028 (42.27%)


In [31]:
# Create a reproducible candidate pool for the 200-example golden evaluation set.

golden_candidates = (
    clean_apple[
        [
            "customer_tweet_id",
            "customer_message",
            "previous_message",
            "has_context"
        ]
    ]
    .sample(
        1000,
        random_state=2026
    )
    .reset_index(drop=True)
)

print("Golden candidate pool:", golden_candidates.shape)
display(golden_candidates.head(10))

Golden candidate pool: (1000, 4)


,customer_tweet_id,customer_message,previous_message,has_context
0,363770.0,@AppleSupport my iPhone 8+ is set to auto lock...,NaN,False
1,2499382.0,@115858 @AppleSupport Please tell me you are s...,NaN,False
2,1558857.0,@AppleSupport assuming you guys are hearing a ...,NaN,False
3,1975429.0,Since I updated my iOS my phone has slowed to ...,NaN,False
4,1643835.0,@AppleSupport when are y’all going to fix this...,NaN,False
5,2476068.0,@AppleSupport if the issue is not resolved i'l...,@707802 Thank you for the details. If you have...,True
6,239244.0,"updated Ios11 but it still sucks, even my came...",NaN,False
7,2205946.0,@AppleSupport It’s 5pm and my phone is at 30% ...,@644985 We can understand wanting the same bat...,True
8,2001862.0,So @115858 why this ? Mark box pop up every ti...,NaN,False
9,1425148.0,@AppleSupport For this app it has happened mul...,@451101 We're here to help. Is this issue happ...,True


In [32]:
# Create broad topic signals for sampling diverse examples; these signals are for sampling only, not final intent labels.

import re

def contains_any(text, keywords):
    text = str(text).lower()
    return any(re.search(rf"\b{re.escape(word)}\b", text) for word in keywords)

sampling_signals = {
    "ios_update": [
        "ios", "update", "upgrade", "ios11", "ios 11"
    ],
    "battery_charging": [
        "battery", "charge", "charging", "charger", "drain"
    ],
    "connectivity": [
        "wifi", "wi-fi", "bluetooth", "cellular", "network",
        "internet", "airdrop", "signal"
    ],
    "account_authentication": [
        "apple id", "password", "login", "sign in", "account",
        "authentication", "icloud id"
    ],
    "icloud_backup": [
        "icloud", "backup", "sync", "storage"
    ],
    "apps_app_store": [
        "app", "apps", "app store", "download", "crash"
    ],
    "audio": [
        "sound", "speaker", "microphone", "mic", "volume",
        "headphone", "airpods", "audio", "call"
    ],
    "camera_photos": [
        "camera", "photo", "photos", "picture", "video"
    ],
    "display_input": [
        "screen", "display", "touch", "keyboard", "typing",
        "type", "brightness"
    ],
    "apple_services": [
        "music", "itunes", "imessage", "facetime", "maps", "siri"
    ],
}

for intent, keywords in sampling_signals.items():
    golden_candidates[intent + "_signal"] = golden_candidates[
        "customer_message"
    ].apply(lambda x: contains_any(x, keywords))

signal_columns = [intent + "_signal" for intent in sampling_signals]

signal_counts = (
    golden_candidates[signal_columns]
    .sum()
    .sort_values(ascending=False)
)

display(signal_counts.to_frame("candidate_count"))

,candidate_count
ios_update_signal,284
battery_charging_signal,103
display_input_signal,94
apps_app_store_signal,91
apple_services_signal,53
connectivity_signal,38
audio_signal,31
icloud_backup_signal,31
camera_photos_signal,30
account_authentication_signal,27


In [33]:
# Select a diverse 200-example review set using topic signals plus random coverage; signals are only for sampling, not labels.

import numpy as np

rng = np.random.default_rng(2026)

# Target roughly equal representation across the major topic signals.
samples_per_signal = 15

selected_indices = set()

for signal_column in signal_columns:
    available = golden_candidates[
        golden_candidates[signal_column] &
        ~golden_candidates.index.isin(selected_indices)
    ].index.to_numpy()

    if len(available) > 0:
        n = min(samples_per_signal, len(available))
        chosen = rng.choice(available, size=n, replace=False)
        selected_indices.update(chosen)

# Fill the remaining slots with completely random examples.
remaining = 200 - len(selected_indices)

available_random = np.array([
    i for i in golden_candidates.index
    if i not in selected_indices
])

if remaining > 0:
    chosen_random = rng.choice(
        available_random,
        size=remaining,
        replace=False
    )
    selected_indices.update(chosen_random)

golden_review = (
    golden_candidates
    .loc[sorted(selected_indices)]
    [
        [
            "customer_tweet_id",
            "customer_message",
            "previous_message",
            "has_context"
        ]
    ]
    .reset_index(drop=True)
)

print("Golden review set:", golden_review.shape)
display(golden_review.head(20))

Golden review set: (200, 4)


,customer_tweet_id,customer_message,previous_message,has_context
0,611285.0,Hey @115858 how can Siri really be my personal...,NaN,False
1,400853.0,@AppleSupport Was able to help him via iTunes....,@210813 Let us know if the update has issues t...,True
2,1099710.0,@379507 @AppleSupport Having the same issue.,@AppleSupport iOS 11.0.3. 2 different iPhone 7...,True
3,1342708.0,Hey @AppleSupport iOS11 update made my $900 iP...,NaN,False
4,1687712.0,"Good afternoon, @AppleSupport! It’s always som...",NaN,False
5,1121644.0,@AppleSupport App is shutting down randomly an...,@384569 Is WhatsApp the only app having an issue?,True
6,1747356.0,Is anyone else having problems downloading and...,NaN,False
7,2119414.0,"So wtf y’all gon do about this I situation, @1...",NaN,False
8,1983621.0,@AppleSupport hey why is the remove episode op...,NaN,False
9,323849.0,@AppleSupport my phone battery has just gone f...,NaN,False


In [34]:
# Create the blank golden evaluation file so we can manually assign and review the 200 intent labels.

from pathlib import Path

evaluation_dir = Path("../data/evaluation")
evaluation_dir.mkdir(parents=True, exist_ok=True)

golden_set = golden_review.copy()

golden_set["intent"] = ""
golden_set["notes"] = ""

golden_path = evaluation_dir / "apple_support_golden_200.csv"
golden_set.to_csv(golden_path, index=False)

print(f"Created: {golden_path}")
print(f"Rows: {len(golden_set)}")
print(f"Columns: {list(golden_set.columns)}")

Created: ..\data\evaluation\apple_support_golden_200.csv
Rows: 200
Columns: ['customer_tweet_id', 'customer_message', 'previous_message', 'has_context', 'intent', 'notes']


In [35]:
# Display the first 20 golden examples so they can be manually assigned one primary intent each.

batch_1 = golden_set.iloc[0:20][
    [
        "customer_tweet_id",
        "customer_message",
        "previous_message",
        "intent",
        "notes"
    ]
]

display(batch_1)

,customer_tweet_id,customer_message,previous_message,intent,notes
0,611285.0,Hey @115858 how can Siri really be my personal...,NaN,,
1,400853.0,@AppleSupport Was able to help him via iTunes....,@210813 Let us know if the update has issues t...,,
2,1099710.0,@379507 @AppleSupport Having the same issue.,@AppleSupport iOS 11.0.3. 2 different iPhone 7...,,
3,1342708.0,Hey @AppleSupport iOS11 update made my $900 iP...,NaN,,
4,1687712.0,"Good afternoon, @AppleSupport! It’s always som...",NaN,,
5,1121644.0,@AppleSupport App is shutting down randomly an...,@384569 Is WhatsApp the only app having an issue?,,
6,1747356.0,Is anyone else having problems downloading and...,NaN,,
7,2119414.0,"So wtf y’all gon do about this I situation, @1...",NaN,,
8,1983621.0,@AppleSupport hey why is the remove episode op...,NaN,,
9,323849.0,@AppleSupport my phone battery has just gone f...,NaN,,


In [36]:
# Display the first 20 examples as full text so each intent can be labelled from complete context.

for i, row in golden_set.iloc[0:20].iterrows():
    print(f"\n{'=' * 80}")
    print(f"INDEX: {i}")
    print(f"CUSTOMER: {row['customer_message']}")
    print(f"PREVIOUS: {row['previous_message']}")


INDEX: 0
CUSTOMER: Hey @115858 how can Siri really be my personal assistant if it won't integrate into 3rd party apps why can't I have it play Spotify where is the functionality?
PREVIOUS: nan

INDEX: 1
CUSTOMER: @AppleSupport Was able to help him via iTunes. Still weird the OTA didn’t work
PREVIOUS: @210813 Let us know if the update has issues through iTunes.

INDEX: 2
CUSTOMER: @379507 @AppleSupport Having the same issue.
PREVIOUS: @AppleSupport iOS 11.0.3. 2 different iPhone 7s in my family, multiple iPads too. Enter address in Maps &amp; driving directions fail to load—endless spinner.

INDEX: 3
CUSTOMER: Hey @AppleSupport iOS11 update made my $900 iPad Pro unusable. The battery lasts 1/2 day even if just sitting there in airplane mode. #SUCKS
PREVIOUS: nan

INDEX: 4
CUSTOMER: Good afternoon, @AppleSupport! It’s always some shit with y’all and these iPhone updates, huh? https://t.co/cGR2qXnGNA
PREVIOUS: nan

INDEX: 5
CUSTOMER: @AppleSupport App is shutting down randomly and unexpe

In [37]:
# Save the manually reviewed labels and notes for golden examples 0–19 in one step.

labels_0_19 = {
    0: ("apple_services", "Siri/Spotify third-party app integration issue"),
    1: ("ios_update", "OTA update failed; iTunes update worked"),
    2: ("apple_services", "Apple Maps directions fail to load"),
    3: ("battery_charging", "Battery drain is the primary issue after iOS update"),
    4: ("ios_update", "Complaint specifically about iPhone updates"),
    5: ("apps_app_store", "App is shutting down unexpectedly"),
    6: ("apps_app_store", "Cannot download or update apps"),
    7: ("display_input", "Typing/input problem involving the letter I"),
    8: ("apps_app_store", "Podcast app functionality issue"),
    9: ("battery_charging", "Rapid battery drain"),
    10: ("audio", "Audio cuts out during video recording"),
    11: ("camera_photos", "Front camera causes phone restart"),
    12: ("ios_update", "Requests another update to fix freezing"),
    13: ("audio", "Ringtone/speaker behavior with connected devices"),
    14: ("display_input", "Screen automatically rotates"),
    15: ("account_authentication", "Apple ID password recovery"),
    16: ("camera_photos", "Rear camera crashes after iOS update"),
    17: ("ios_update", "Severe slowdown after iOS 11 update"),
    18: ("apps_app_store", "Apps/App Store purchases disappearing"),
    19: ("battery_charging", "Battery drain after update"),
}

for index, (intent, notes) in labels_0_19.items():
    golden_set.loc[index, "intent"] = intent
    golden_set.loc[index, "notes"] = notes

golden_set.to_csv(golden_path, index=False)

labelled_count = (
    golden_set["intent"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)

print(f"Saved labels for rows 0–19.")
print(f"Total labelled: {labelled_count}/200")

Saved labels for rows 0–19.
Total labelled: 20/200


In [38]:
# Export examples 20–69 as full plain text so VS Code does not truncate long customer messages.

batch_start = 20
batch_end = 70

with open("../data/processed/labeling_batch_20_69.txt", "w", encoding="utf-8") as f:
    for i, row in golden_set.iloc[batch_start:batch_end].iterrows():
        f.write("=" * 100 + "\n")
        f.write(f"INDEX: {i}\n")
        f.write(f"CUSTOMER: {row['customer_message']}\n")
        f.write(f"PREVIOUS: {row['previous_message']}\n\n")

print("Created temporary viewing file:")
print("../data/processed/labeling_batch_20_69.txt")

Created temporary viewing file:
../data/processed/labeling_batch_20_69.txt


In [39]:
# Save the manually reviewed labels and notes for golden examples 20–69.

labels_20_69 = {
    20: ("connectivity", "CarPlay/iPhone/car connection issue"),
    21: ("other_unclear", "Generic unresolved support complaint"),
    22: ("display_input", "Keyboard/letter problem"),
    23: ("battery_charging", "Battery drain"),
    24: ("connectivity", "Cannot sync iPhone with Mac"),
    25: ("connectivity", "AirDrop issue"),
    26: ("device_hardware", "Apple Watch restarting"),
    27: ("display_input", "I keyboard glitch"),
    28: ("apps_app_store", "App Store password/download behavior"),
    29: ("connectivity", "Cannot make or receive calls"),
    30: ("display_input", "I typing problem"),
    31: ("battery_charging", "Battery deterioration after update"),
    32: ("account_authentication", "Apple ID reset problem"),
    33: ("connectivity", "Calls hang during dialing"),
    34: ("account_authentication", "Cannot sign into iCloud"),
    35: ("display_input", "I typing problem"),
    36: ("ios_update", "Freezing after iOS 11 update"),
    37: ("audio", "Media button/audio behavior"),
    38: ("audio", "Audio cuts out during screen recording"),
    39: ("device_hardware", "Device randomly crashes/restarts"),
    40: ("account_authentication", "Unauthorized subscription"),
    41: ("account_authentication", "iCloud Mail access/sign-in"),
    42: ("display_input", "I input problem"),
    43: ("apple_services", "Apple TV streaming issue"),
    44: ("apple_services", "Apple Music account issue"),
    45: ("ios_update", "Update-related screen compatibility issue"),
    46: ("connectivity", "Wi-Fi/cellular connectivity difference"),
    47: ("device_hardware", "Broken screen/device access issue"),
    48: ("battery_charging", "Rapid battery drain"),
    49: ("apple_services", "iMessage sign-in/service issue"),
    50: ("display_input", "Autocorrect/input problem"),
    51: ("apple_services", "iMessage not working"),
    52: ("apple_services", "iMessage group-chat functionality"),
    53: ("display_input", "I input problem"),
    54: ("display_input", "I input problem"),
    55: ("display_input", "I input problem"),
    56: ("device_hardware", "Multiple device-level failures"),
    57: ("other_unclear", "Generic glitch complaint"),
    58: ("audio", "AirPod charging problem"),
    59: ("apple_services", "FaceTime not working"),
    60: ("ios_update", "Freezing after iOS 11"),
    61: ("display_input", "Cannot type I.T."),
    62: ("apple_services", "Spotify sync/service issue"),
    63: ("account_authentication", "iCloud account type change"),
    64: ("battery_charging", "Battery problem"),
    65: ("battery_charging", "Context indicates charging/charger issue"),
    66: ("apps_app_store", "Files functionality issue"),
    67: ("apps_app_store", "Cannot purchase/download apps"),
    68: ("connectivity", "Cellular carrier/network issue"),
    69: ("device_hardware", "Lost iPhone/device recovery"),
}

for index, (intent, notes) in labels_20_69.items():
    golden_set.loc[index, "intent"] = intent
    golden_set.loc[index, "notes"] = notes

golden_set.to_csv(golden_path, index=False)

labelled_count = (
    golden_set["intent"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)

print(f"Saved labels for rows 20–69.")
print(f"Total labelled: {labelled_count}/200")

Saved labels for rows 20–69.
Total labelled: 70/200


In [40]:
# Export golden examples 70–119 as full text so the messages and conversation context are not clipped.

batch_start = 70
batch_end = 120

with open("../data/processed/labeling_batch_70_119.txt", "w", encoding="utf-8") as f:
    for i, row in golden_set.iloc[batch_start:batch_end].iterrows():
        f.write("=" * 100 + "\n")
        f.write(f"INDEX: {i}\n")
        f.write(f"CUSTOMER: {row['customer_message']}\n")
        f.write(f"PREVIOUS: {row['previous_message']}\n\n")

print("Created temporary viewing file:")
print("../data/processed/labeling_batch_70_119.txt")

Created temporary viewing file:
../data/processed/labeling_batch_70_119.txt


In [41]:
# Save the manually reviewed labels and notes for golden examples 70–119.

labels_70_119 = {
    70: ("apps_app_store", "App storage usage"),
    71: ("apps_app_store", "Apps cannot be updated"),
    72: ("camera_photos", "Rear camera produces blurry photos"),
    73: ("apple_services", "Safari/iCloud bookmark syncing"),
    74: ("battery_charging", "Poor battery life"),
    75: ("audio", "Cannot adjust volume while locked"),
    76: ("ios_update", "Update-related glitches"),
    77: ("device_hardware", "Multiple device failures"),
    78: ("audio", "Unexpected alarm-like sound"),
    79: ("display_input", "Keyboard prevents login"),
    80: ("icloud_backup", "Apple Watch cannot restore backup"),
    81: ("display_input", "Keyboard glitch"),
    82: ("display_input", "Keyboard I glitch"),
    83: ("account_authentication", "Apple ID activation/authentication error"),
    84: ("battery_charging", "Battery suddenly drops"),
    85: ("account_authentication", "Unexpected account/credit-card charge"),
    86: ("account_authentication", "Suspicious iCloud sign-in notifications"),
    87: ("account_authentication", "iCloud account recovery/lockout"),
    88: ("icloud_backup", "iCloud Photos download/sync issue"),
    89: ("account_authentication", "Switching iCloud account email"),
    90: ("battery_charging", "Severe battery drain after update"),
    91: ("device_hardware", "Device restart troubleshooting context"),
    92: ("display_input", "Keyboard lag"),
    93: ("account_authentication", "Forgot iCloud password"),
    94: ("apple_services", "iMessage/FaceTime activation"),
    95: ("apps_app_store", "Cannot download apps"),
    96: ("camera_photos", "Camera Live Photos setting issue"),
    97: ("account_authentication", "Family Sharing authentication"),
    98: ("audio", "Speaker crackling"),
    99: ("icloud_backup", "iCloud backup retention"),
    100: ("device_hardware", "Multiple device/function failures"),
    101: ("account_authentication", "Account recovery"),
    102: ("battery_charging", "Extreme battery drain"),
    103: ("audio", "Screenshot sound missing"),
    104: ("device_hardware", "Apple Watch incorrect time"),
    105: ("apps_app_store", "Apps missing after restore"),
    106: ("display_input", "I character rendering problem"),
    107: ("apple_services", "Apple Music playback issue"),
    108: ("apple_services", "iTunes/Apple Music syncing"),
    109: ("audio", "Calls have no sound"),
    110: ("connectivity", "Wi-Fi problem"),
    111: ("connectivity", "AirDrop connection problem"),
    112: ("camera_photos", "Camera roll/photo library issue"),
    113: ("icloud_backup", "iCloud Photo Library error"),
    114: ("display_input", "Green line on screen"),
    115: ("battery_charging", "Battery problem after update"),
    116: ("battery_charging", "Battery issue is primary"),
    117: ("apps_app_store", "App error prevents preorder"),
    118: ("display_input", "I becomes question-mark box"),
    119: ("apps_app_store", "Managed-account app controls"),
}

for index, (intent, notes) in labels_70_119.items():
    golden_set.loc[index, "intent"] = intent
    golden_set.loc[index, "notes"] = notes

golden_set.to_csv(golden_path, index=False)

labelled_count = (
    golden_set["intent"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)

print(f"Saved labels for rows 70–119.")
print(f"Total labelled: {labelled_count}/200")

Saved labels for rows 70–119.
Total labelled: 120/200


In [42]:
# Export golden examples 120–169 as full text so the complete customer messages and context can be reviewed without clipping.

batch_start = 120
batch_end = 170

with open("../data/processed/labeling_batch_120_169.txt", "w", encoding="utf-8") as f:
    for i, row in golden_set.iloc[batch_start:batch_end].iterrows():
        f.write("=" * 100 + "\n")
        f.write(f"INDEX: {i}\n")
        f.write(f"CUSTOMER: {row['customer_message']}\n")
        f.write(f"PREVIOUS: {row['previous_message']}\n\n")

print("Created temporary viewing file:")
print("../data/processed/labeling_batch_120_169.txt")

Created temporary viewing file:
../data/processed/labeling_batch_120_169.txt


In [43]:
# Save the manually reviewed labels and notes for golden examples 120–169.

labels_120_169 = {
    120: ("display_input", "Keyboard issue"),
    121: ("apple_services", "iTunes/music not appearing on iPhone"),
    122: ("device_hardware", "Screen completely blank/unresponsive"),
    123: ("battery_charging", "Update-related charging problem"),
    124: ("apple_services", "Music stops when opening another app"),
    125: ("display_input", "Typing I issue"),
    126: ("apps_app_store", "Mail app freezes"),
    127: ("account_authentication", "Apple ID account recovery"),
    128: ("connectivity", "Bluetooth speaker disconnects"),
    129: ("battery_charging", "Battery issue"),
    130: ("audio", "Headphones/volume issue"),
    131: ("ios_update", "Update causes shutdown/Apple logo loop"),
    132: ("apple_services", "Text messages disappeared"),
    133: ("camera_photos", "Camera issue"),
    134: ("connectivity", "Call/FaceTime notifications not arriving"),
    135: ("connectivity", "Wireless keyboard connection issue"),
    136: ("display_input", "I becomes a box"),
    137: ("apps_app_store", "Mail app creates duplicate drafts"),
    138: ("display_input", "Screen issue"),
    139: ("apple_services", "Apple Watch push notifications"),
    140: ("account_authentication", "Account unlock/trusted number"),
    141: ("connectivity", "Wi-Fi not working"),
    142: ("apple_services", "Music not appearing in library"),
    143: ("other_unclear", "Multiple unrelated iOS problems"),
    144: ("apps_app_store", "Context indicates app updates"),
    145: ("connectivity", "Wi-Fi stopped working"),
    146: ("other_unclear", "Only mentions scheduled callback"),
    147: ("connectivity", "Bluetooth problems"),
    148: ("display_input", "Keyboard I issue"),
    149: ("apple_services", "Apple Music issue"),
    150: ("audio", "Apple Watch microphone issue"),
    151: ("ios_update", "Context indicates iOS version question"),
    152: ("device_hardware", "Phone storage is full"),
    153: ("account_authentication", "Apple ID recovery"),
    154: ("display_input", "Keyboard I issue"),
    155: ("ios_update", "iOS 11 freezing/glitches"),
    156: ("battery_charging", "Battery life reduced after update"),
    157: ("display_input", "3D Touch lag"),
    158: ("ios_update", "iOS software fault"),
    159: ("device_hardware", "Multiple hardware/function problems"),
    160: ("display_input", "Context indicates screen issue"),
    161: ("account_authentication", "iTunes billing/account issue"),
    162: ("apps_app_store", "Context indicates album download problem"),
    163: ("display_input", "Typing issue"),
    164: ("icloud_backup", "Missing picture requested from iCloud"),
    165: ("device_hardware", "iPhone crashes despite update"),
    166: ("battery_charging", "Battery issue"),
    167: ("apps_app_store", "Safari/app deletion functionality"),
    168: ("apple_services", "Contacts disappearing"),
    169: ("connectivity", "Wi-Fi switch disabled"),
}

for index, (intent, notes) in labels_120_169.items():
    golden_set.loc[index, "intent"] = intent
    golden_set.loc[index, "notes"] = notes

golden_set.to_csv(golden_path, index=False)

labelled_count = (
    golden_set["intent"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)

print(f"Saved labels for rows 120–169.")
print(f"Total labelled: {labelled_count}/200")

Saved labels for rows 120–169.
Total labelled: 170/200


In [44]:
# Export the final 30 golden examples as full text so we can manually review and label them.

batch_start = 170
batch_end = 200

with open("../data/processed/labeling_batch_170_199.txt", "w", encoding="utf-8") as f:
    for i, row in golden_set.iloc[batch_start:batch_end].iterrows():
        f.write("=" * 100 + "\n")
        f.write(f"INDEX: {i}\n")
        f.write(f"CUSTOMER: {row['customer_message']}\n")
        f.write(f"PREVIOUS: {row['previous_message']}\n\n")

print("Created temporary viewing file:")
print("../data/processed/labeling_batch_170_199.txt")

Created temporary viewing file:
../data/processed/labeling_batch_170_199.txt


In [45]:
# Save the manually reviewed labels and notes for the final golden examples 170–199.

labels_170_199 = {
    170: ("display_input", "Screen/control-center interaction issue"),
    171: ("camera_photos", "Camera malfunction"),
    172: ("connectivity", "Wi-Fi turns back on"),
    173: ("apple_services", "Mail/Gmail notification and loading issue"),
    174: ("apple_services", "Video streaming/playback issue"),
    175: ("camera_photos", "Photos disappeared after screenshot"),
    176: ("camera_photos", "Videos not working"),
    177: ("camera_photos", "Camera photos not saving"),
    178: ("display_input", "I typing problem is primary"),
    179: ("apple_services", "Siri issue"),
    180: ("apple_services", "Apple Music subscription issue"),
    181: ("battery_charging", "Battery health/drain"),
    182: ("display_input", "Touchscreen problem is primary"),
    183: ("apps_app_store", "Clock app cannot open and causes crashes"),
    184: ("ios_update", "New iOS update complaint"),
    185: ("display_input", "Screen discoloration"),
    186: ("device_hardware", "Apple Watch functionality issue"),
    187: ("connectivity", "Context indicates cellular connection issue"),
    188: ("apple_services", "Apple Watch synced notifications"),
    189: ("device_hardware", "Internal/device damage"),
    190: ("display_input", "UI button cannot be selected"),
    191: ("apps_app_store", "WhatsApp not loading new messages"),
    192: ("display_input", "Keyboard issue"),
    193: ("connectivity", "Wi-Fi disconnects"),
    194: ("camera_photos", "Photos deleted from camera roll"),
    195: ("display_input", "AssistiveTouch/Reachability interaction"),
    196: ("other_unclear", "No actual issue stated"),
    197: ("connectivity", "Bluetooth/Wi-Fi behavior"),
    198: ("icloud_backup", "iCloud backup restoration"),
    199: ("connectivity", "Wi-Fi/Bluetooth control behavior"),
}

for index, (intent, notes) in labels_170_199.items():
    golden_set.loc[index, "intent"] = intent
    golden_set.loc[index, "notes"] = notes

golden_set.to_csv(golden_path, index=False)

labelled_count = (
    golden_set["intent"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)

print(f"Saved labels for rows 170–199.")
print(f"Total labelled: {labelled_count}/200")

Saved labels for rows 170–199.
Total labelled: 200/200


In [46]:
# Audit the completed 200-example golden set to verify labels and inspect the intent distribution before model training.

valid_intents = set(intent_definitions.keys())

label_counts = (
    golden_set["intent"]
    .fillna("")
    .astype(str)
    .str.strip()
    .value_counts()
)

invalid_labels = sorted(
    set(label_counts.index) - valid_intents - {""}
)

missing_labels = sorted(
    valid_intents - set(label_counts.index)
)

unlabelled_count = (
    golden_set["intent"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("Total examples:", len(golden_set))
print("Unlabelled:", unlabelled_count)
print("Invalid labels:", invalid_labels)

print("\nIntent distribution:")
display(label_counts.to_frame("count"))

print("\nIntents with zero examples:")
print(missing_labels)

Total examples: 200
Unlabelled: 0
Invalid labels: []

Intent distribution:


,count
intent,
display_input,35
apple_services,25
connectivity,21
apps_app_store,20
battery_charging,19
account_authentication,18
device_hardware,15
ios_update,13
audio,12



Intents with zero examples:
[]


In [47]:
# Create a training pool by excluding the 200 golden evaluation examples, preventing evaluation leakage.

golden_ids = set(golden_set["customer_tweet_id"].astype(str))

training_pool = clean_apple[
    ~clean_apple["customer_tweet_id"].astype(str).isin(golden_ids)
].copy()

print("Clean AppleSupport interactions:", len(clean_apple))
print("Golden evaluation examples:", len(golden_set))
print("Training pool:", len(training_pool))
print("Golden IDs found in training pool:", 
      training_pool["customer_tweet_id"].astype(str).isin(golden_ids).sum())

Clean AppleSupport interactions: 106523
Golden evaluation examples: 200
Training pool: 106323
Golden IDs found in training pool: 0


In [48]:
# Prepare the reviewed golden examples as a labelled seed set for building the initial intent classifier.

seed_data = golden_set[
    ["customer_tweet_id", "customer_message", "previous_message", "intent"]
].copy()

seed_data["customer_message"] = (
    seed_data["customer_message"]
    .fillna("")
    .astype(str)
    .str.strip()
)

seed_data["previous_message"] = (
    seed_data["previous_message"]
    .fillna("")
    .astype(str)
    .str.strip()
)

seed_data["intent"] = (
    seed_data["intent"]
    .fillna("")
    .astype(str)
    .str.strip()
)

print("Seed examples:", len(seed_data))
print("Unique intents:", seed_data["intent"].nunique())
print("\nSeed distribution:")
display(seed_data["intent"].value_counts().to_frame("count"))

Seed examples: 200
Unique intents: 12

Seed distribution:


,count
intent,
display_input,35
apple_services,25
connectivity,21
apps_app_store,20
battery_charging,19
account_authentication,18
device_hardware,15
ios_update,13
audio,12


In [49]:
# Sample 10,000 non-golden AppleSupport interactions as the candidate pool for training-label generation.

training_candidates = (
    training_pool[
        [
            "customer_tweet_id",
            "customer_message",
            "previous_message",
            "has_context",
        ]
    ]
    .sample(
        n=10_000,
        random_state=2026,
    )
    .reset_index(drop=True)
)

training_candidates_path = (
    Path("../data/processed") /
    "apple_support_training_candidates.csv"
)

training_candidates.to_csv(
    training_candidates_path,
    index=False,
)

print("Created:", training_candidates_path)
print("Shape:", training_candidates.shape)
print("Golden examples excluded:", 
      training_candidates["customer_tweet_id"]
      .astype(str)
      .isin(golden_ids)
      .sum())

Created: ..\data\processed\apple_support_training_candidates.csv
Shape: (10000, 4)
Golden examples excluded: 0


In [50]:
# Create concise semantic descriptions for the 12 intents so the labelling step has consistent intent definitions.

intent_descriptions = {
    "ios_update": (
        "Problems or questions about iOS software updates, "
        "including update failures, installation problems, update bugs, "
        "or behavior caused specifically by an iOS version."
    ),
    "battery_charging": (
        "Battery drain, poor battery life, battery health, charging problems, "
        "charging speed, or unexpected battery percentage changes."
    ),
    "connectivity": (
        "Wi-Fi, cellular/mobile data, Bluetooth, AirDrop, CarPlay, "
        "calls, or other device connection and network problems."
    ),
    "account_authentication": (
        "Apple ID, iCloud login, passwords, authentication, account recovery, "
        "account access, subscriptions, billing, or account-related security."
    ),
    "icloud_backup": (
        "iCloud backup, iCloud storage, restoring from backup, "
        "iCloud Photos, missing synced data, or iCloud data synchronization."
    ),
    "apps_app_store": (
        "Problems with applications, App Store, downloading or updating apps, "
        "app crashes, app behavior, or application functionality."
    ),
    "audio": (
        "Speaker, microphone, headphones, AirPods, ringtone, volume, "
        "call audio, or other sound-related problems."
    ),
    "camera_photos": (
        "Camera problems, taking photos or videos, camera crashes, "
        "photo library problems, missing photos, or photo-related functionality."
    ),
    "display_input": (
        "Screen, display, touchscreen, keyboard, typing, autocorrect, "
        "buttons, visual glitches, or other user-interface/input problems."
    ),
    "apple_services": (
        "Problems with Apple services such as Apple Music, Siri, Maps, "
        "iMessage, FaceTime, Apple TV, Safari, or other Apple services."
    ),
    "device_hardware": (
        "Physical device problems, unexpected device restarts or crashes, "
        "Apple Watch device issues, storage/device-level failures, "
        "or hardware problems that do not fit a more specific intent."
    ),
    "other_unclear": (
        "Messages that are too vague, unrelated, lack enough information, "
        "or cannot be confidently assigned to another intent."
    ),
}

print("Intent descriptions prepared:", len(intent_descriptions))
print("\n" + "\n".join(
    f"{i}. {intent}: {description}"
    for i, (intent, description) in enumerate(intent_descriptions.items(), 1)
))

Intent descriptions prepared: 12

1. ios_update: Problems or questions about iOS software updates, including update failures, installation problems, update bugs, or behavior caused specifically by an iOS version.
2. battery_charging: Battery drain, poor battery life, battery health, charging problems, charging speed, or unexpected battery percentage changes.
3. connectivity: Wi-Fi, cellular/mobile data, Bluetooth, AirDrop, CarPlay, calls, or other device connection and network problems.
4. account_authentication: Apple ID, iCloud login, passwords, authentication, account recovery, account access, subscriptions, billing, or account-related security.
5. icloud_backup: iCloud backup, iCloud storage, restoring from backup, iCloud Photos, missing synced data, or iCloud data synchronization.
6. apps_app_store: Problems with applications, App Store, downloading or updating apps, app crashes, app behavior, or application functionality.
7. audio: Speaker, microphone, headphones, AirPods, ringto

In [51]:
# Verify the active notebook environment and confirm sentence-transformers is available.

import sys
import sentence_transformers

print("Python:", sys.executable)
print("sentence-transformers:", sentence_transformers.__version__)

Python: c:\Projects\hiver-support-agent\.venv\Scripts\python.exe
sentence-transformers: 6.0.1


In [52]:
# Load the embedding model once; we will reuse it for the seed examples and training candidates.

from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


In [53]:
# Convert the 200 reviewed customer messages into semantic embeddings for similarity-based labelling.

seed_texts = seed_data["customer_message"].tolist()

seed_embeddings = embedding_model.encode(
    seed_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

seed_embeddings = np.asarray(seed_embeddings)

print("Seed embeddings shape:", seed_embeddings.shape)
print("Number of seed examples:", len(seed_embeddings))

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Seed embeddings shape: (200, 384)
Number of seed examples: 200


In [54]:
# Test semantic nearest-neighbor labelling on 20 training candidates before processing the full training pool.

from sklearn.metrics.pairwise import cosine_similarity

test_candidates = training_candidates.sample(
    n=20,
    random_state=2026
).reset_index(drop=True)

test_embeddings = embedding_model.encode(
    test_candidates["customer_message"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

similarity_matrix = cosine_similarity(
    test_embeddings,
    seed_embeddings
)

nearest_indices = similarity_matrix.argmax(axis=1)
nearest_scores = similarity_matrix.max(axis=1)

test_results = test_candidates[
    ["customer_tweet_id", "customer_message"]
].copy()

test_results["predicted_intent"] = [
    seed_data.iloc[i]["intent"]
    for i in nearest_indices
]

test_results["similarity"] = nearest_scores

test_results["nearest_seed_message"] = [
    seed_data.iloc[i]["customer_message"]
    for i in nearest_indices
]

display(test_results)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,customer_tweet_id,customer_message,predicted_intent,similarity,nearest_seed_message
0,998063.0,@AppleSupport my battery life is crazy don’t l...,battery_charging,0.700269,@AppleSupport my battery life has drastically ...
1,1395729.0,@AppleSupport I’m so sorry........ I can’t mee...,other_unclear,0.581353,@AppleSupport will you guys plz reply?
2,567644.0,@AppleSupport Power adapter and it doesn’t cha...,battery_charging,0.539580,@AppleSupport ios11.0.03 is a nightmare. Phone...
3,2594410.0,@AppleSupport apple music,apple_services,0.772939,@AppleSupport I have an issue with my Apple Mu...
4,402246.0,Yo @AppleSupport 11.0.2 is fucking shocking. L...,battery_charging,0.672495,@115858 @AppleSupport Please fix battery issue...
5,486097.0,@AppleSupport Thank you for your response. App...,device_hardware,0.473305,@AppleSupport watch app showing incorrect time...
6,1849007.0,@AppleSupport Hi! Back in December I had my 6s...,battery_charging,0.577854,@AppleSupport my iPhone 7 under warranty has a...
7,2078369.0,@AppleSupport Yes! It’s continued after restart,apple_services,0.734374,@379507 @AppleSupport Having the same issue.
8,906420.0,@115858 really? Ive not had one thing wrong wi...,connectivity,0.744953,Completely fed up with bluetooth issues in iOS...
9,2381362.0,@AppleSupport how can I check health of my iPh...,battery_charging,0.751610,"@AppleSupport Hey Apple, can you check my batt..."


In [55]:
# Build one semantic prototype for each intent by averaging the embeddings of its reviewed examples.

from sklearn.preprocessing import normalize

intent_prototypes = {}

for intent in seed_data["intent"].unique():
    intent_indices = seed_data.index[
        seed_data["intent"] == intent
    ].to_numpy()

    intent_vectors = seed_embeddings[intent_indices]

    prototype = intent_vectors.mean(axis=0)
    prototype = prototype / np.linalg.norm(prototype)

    intent_prototypes[intent] = prototype

prototype_matrix = np.vstack([
    intent_prototypes[intent]
    for intent in intent_descriptions.keys()
])

print("Prototype matrix shape:", prototype_matrix.shape)
print("Number of intent prototypes:", len(intent_prototypes))

Prototype matrix shape: (12, 384)
Number of intent prototypes: 12


In [56]:
# Test prototype-based intent prediction on the same 20 candidates and inspect confidence scores.

prototype_similarity = cosine_similarity(
    test_embeddings,
    prototype_matrix
)

prototype_indices = prototype_similarity.argmax(axis=1)
prototype_scores = prototype_similarity.max(axis=1)

prototype_results = test_candidates[
    ["customer_tweet_id", "customer_message"]
].copy()

prototype_results["predicted_intent"] = [
    list(intent_descriptions.keys())[i]
    for i in prototype_indices
]

prototype_results["similarity"] = prototype_scores

display(prototype_results)

,customer_tweet_id,customer_message,predicted_intent,similarity
0,998063.0,@AppleSupport my battery life is crazy don’t l...,battery_charging,0.701821
1,1395729.0,@AppleSupport I’m so sorry........ I can’t mee...,other_unclear,0.553605
2,567644.0,@AppleSupport Power adapter and it doesn’t cha...,battery_charging,0.483301
3,2594410.0,@AppleSupport apple music,apple_services,0.660777
4,402246.0,Yo @AppleSupport 11.0.2 is fucking shocking. L...,ios_update,0.698337
5,486097.0,@AppleSupport Thank you for your response. App...,device_hardware,0.298360
6,1849007.0,@AppleSupport Hi! Back in December I had my 6s...,battery_charging,0.439491
7,2078369.0,@AppleSupport Yes! It’s continued after restart,other_unclear,0.674917
8,906420.0,@115858 really? Ive not had one thing wrong wi...,ios_update,0.784906
9,2381362.0,@AppleSupport how can I check health of my iPh...,battery_charging,0.665619


In [57]:
# Train a simple TF-IDF + Logistic Regression intent classifier using the 200 reviewed examples.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95,
    sublinear_tf=True,
)

X_seed = tfidf_vectorizer.fit_transform(
    seed_data["customer_message"]
)

y_seed = seed_data["intent"]

tfidf_classifier = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=2026,
)

tfidf_classifier.fit(X_seed, y_seed)

print("TF-IDF matrix shape:", X_seed.shape)
print("Classifier trained.")
print("Number of classes:", len(tfidf_classifier.classes_))

TF-IDF matrix shape: (200, 4186)
Classifier trained.
Number of classes: 12


In [58]:
# Evaluate the TF-IDF baseline with stratified 5-fold cross-validation to estimate performance on unseen labelled examples.

from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=2026,
)

cv_scores = cross_val_score(
    tfidf_classifier,
    X_seed,
    y_seed,
    cv=cv,
    scoring="accuracy",
)

print("5-fold accuracy scores:")
print(np.round(cv_scores, 4))

print("\nMean accuracy:", round(cv_scores.mean(), 4))
print("Standard deviation:", round(cv_scores.std(), 4))

5-fold accuracy scores:
[0.475 0.4   0.525 0.525 0.425]

Mean accuracy: 0.47
Standard deviation: 0.051


In [59]:
# Measure TF-IDF confidence on the 10,000 training candidates so we can choose a conservative pseudo-labelling threshold.

candidate_tfidf = tfidf_vectorizer.transform(
    training_candidates["customer_message"]
)

candidate_probabilities = tfidf_classifier.predict_proba(
    candidate_tfidf
)

candidate_predictions = tfidf_classifier.classes_[
    candidate_probabilities.argmax(axis=1)
]

candidate_confidence = candidate_probabilities.max(axis=1)

confidence_summary = pd.Series(candidate_confidence).describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

print(confidence_summary)

count    10000.000000
mean         0.115819
std          0.018949
min          0.089395
50%          0.110216
75%          0.123120
90%          0.140886
95%          0.151948
99%          0.176633
max          0.264279
dtype: float64


In [60]:
# Check whether the Transformers library is already available before installing another dependency.

import transformers

print("Transformers:", transformers.__version__)

Transformers: 5.17.0


In [61]:
# Embed the 10,000 training candidates once so we can compare them against all 12 intent prototypes.

candidate_embeddings = embedding_model.encode(
    training_candidates["customer_message"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

candidate_embeddings = np.asarray(candidate_embeddings)

print("Candidate embeddings shape:", candidate_embeddings.shape)

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Candidate embeddings shape: (10000, 384)


In [62]:
# Compare every candidate against all 12 intent prototypes and calculate a margin-based confidence score.

intent_names = list(intent_descriptions.keys())

candidate_similarities = candidate_embeddings @ prototype_matrix.T

best_indices = candidate_similarities.argmax(axis=1)
best_scores = candidate_similarities.max(axis=1)

sorted_scores = np.sort(candidate_similarities, axis=1)

second_best_scores = sorted_scores[:, -2]
confidence_margin = best_scores - second_best_scores

pseudo_labels = [
    intent_names[i]
    for i in best_indices
]

pseudo_results = training_candidates.copy()

pseudo_results["pseudo_intent"] = pseudo_labels
pseudo_results["similarity"] = best_scores
pseudo_results["margin"] = confidence_margin

print("Candidates:", len(pseudo_results))
print("Average best similarity:", round(best_scores.mean(), 4))
print("Average margin:", round(confidence_margin.mean(), 4))

display(
    pseudo_results[
        ["customer_message", "pseudo_intent", "similarity", "margin"]
    ].head(20)
)

Candidates: 10000
Average best similarity: 0.6038
Average margin: 0.0662


,customer_message,pseudo_intent,similarity,margin
0,@115948 @AppleSupport how do I delete extra ra...,apple_services,0.504883,0.058709
1,@AppleSupport what’s up with the “I️” being di...,display_input,0.512001,0.133229
2,@AppleSupport My timeline is still back to front.,other_unclear,0.557768,0.048879
3,Help us please @AppleSupport https://t.co/wJkS...,other_unclear,0.642561,0.069720
4,Why are my i’s showing up weird on my phone 🙄@...,display_input,0.615871,0.084349
5,@AppleSupport Why does this keep showing up on...,ios_update,0.553070,0.021932
6,none of my apps wont update @115858 you’re ugl...,ios_update,0.653010,0.086485
7,@AppleSupport neither my Iphone 8 plus or X sc...,audio,0.610445,0.138225
8,@81400 @115858 This is bullshittttttttt. I upd...,display_input,0.682019,0.075140
9,"@AppleSupport call someone, set phone on speak...",audio,0.558142,0.077695


In [63]:
# Save the 200 reviewed examples separately as the initial labelled seed dataset for model development.

seed_path = Path("../data/processed/apple_support_seed_200.csv")

seed_data.to_csv(
    seed_path,
    index=False,
)

print("Created:", seed_path)
print("Seed examples:", len(seed_data))

Created: ..\data\processed\apple_support_seed_200.csv
Seed examples: 200


In [64]:
# Create 300 diverse training examples for manual labeling.
# These are sampled from the training pool and exclude the existing 200 seed examples.
from pathlib import Path
import pandas as pd
import re

# Start from examples not already used in the seed set.
seed_ids = set(seed_data["customer_tweet_id"].astype(str))

review_pool = training_pool[
    ~training_pool["customer_tweet_id"].astype(str).isin(seed_ids)
].copy()

# Broad signals are used ONLY to improve sampling diversity.
# They are not used as intent labels.
signal_patterns = {
    "ios_update": r"\b(update|updat|ios\s*\d|upgrade|install.*ios)\b",
    "battery_charging": r"\b(battery|charging|charge|charger|power)\b",
    "connectivity": r"\b(wifi|wi-fi|bluetooth|airdrop|cellular|network|internet|signal)\b",
    "account_authentication": r"\b(apple\s*id|password|login|log\s*in|account|authentication|verification)\b",
    "icloud_backup": r"\b(icloud|backup|sync|storage)\b",
    "apps_app_store": r"\b(app\s*store|apps?|application|download|install|crash|update.*app)\b",
    "audio": r"\b(audio|sound|speaker|microphone|mic|volume|headphone|airpods|call)\b",
    "camera_photos": r"\b(camera|photo|photos|picture|video|gallery)\b",
    "display_input": r"\b(screen|display|touch|keyboard|typing|keypad|brightness)\b",
    "apple_services": r"\b(apple\s*music|itunes|imessage|facetime|maps|siri)\b",
}

# Add sampling-only signal columns.
text_for_sampling = (
    review_pool["customer_message"]
    .fillna("")
    .astype(str)
    .str.lower()
)

for signal_name, pattern in signal_patterns.items():
    review_pool[f"{signal_name}_signal"] = text_for_sampling.str.contains(
        pattern,
        regex=True,
        na=False
    )

# Take a balanced number from each broad signal.
# Overlapping examples are allowed, so we deduplicate afterward.
signal_samples = []

for signal_name in signal_patterns:
    matching = review_pool[review_pool[f"{signal_name}_signal"]]

    if len(matching) > 30:
        matching = matching.sample(n=30, random_state=2026)
    else:
        matching = matching.sample(frac=1, random_state=2026)

    signal_samples.append(matching)

diverse_review = pd.concat(signal_samples, ignore_index=False)
diverse_review = diverse_review[~diverse_review.index.duplicated(keep="first")]

# Fill any remaining slots with random examples.
remaining_needed = 300 - len(diverse_review)

if remaining_needed > 0:
    remaining_pool = review_pool.drop(index=diverse_review.index)

    random_fill = remaining_pool.sample(
        n=min(remaining_needed, len(remaining_pool)),
        random_state=2027
    )

    diverse_review = pd.concat(
        [diverse_review, random_fill],
        ignore_index=False
    )

# Final shuffle.
diverse_review = diverse_review.sample(
    n=min(300, len(diverse_review)),
    random_state=2028
).reset_index(drop=True)

# Keep only the columns needed for manual labeling.
training_review_300 = diverse_review[
    [
        "customer_tweet_id",
        "customer_message",
        "previous_message",
        "has_context"
    ]
].copy()

# Empty columns for manual labels and notes.
training_review_300["intent"] = ""
training_review_300["notes"] = ""

# Save the new review file.
training_review_path = Path("../data/processed/apple_support_training_review_300.csv")
training_review_300.to_csv(training_review_path, index=False)

print("Created:", training_review_path)
print("Training review examples:", len(training_review_300))
print("Existing seed examples:", len(seed_data))
print("Total after labeling:", len(seed_data) + len(training_review_300))

C:\Users\P NIKHIL\AppData\Local\Temp\ipykernel_2892\4286577872.py:38: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  review_pool[f"{signal_name}_signal"] = text_for_sampling.str.contains(


Created: ..\data\processed\apple_support_training_review_300.csv
Training review examples: 300
Existing seed examples: 200
Total after labeling: 500


In [66]:
# Freshly prepare the original 200 reviewed examples for model training.
# We intentionally use only golden_set and ignore the additional 300 examples.

import pandas as pd
from sklearn.model_selection import StratifiedKFold

# Use ONLY the original 200 reviewed examples.
model_data = golden_set[
    [
        "customer_tweet_id",
        "customer_message",
        "previous_message",
        "has_context",
        "intent"
    ]
].copy()

# Build the text used by the classifier.
# Include previous context only when it actually exists.
def build_model_text(row):
    customer = str(row["customer_message"]).strip()
    previous = str(row["previous_message"]).strip()

    if row["has_context"] and previous:
        return f"Previous message: {previous} Customer message: {customer}"

    return customer

model_data["model_text"] = model_data.apply(
    build_model_text,
    axis=1
)

# Validate that this is exactly the intended 200-row dataset.
print("Rows:", len(model_data))
print("Unique tweet IDs:", model_data["customer_tweet_id"].nunique())
print("Unique intents:", model_data["intent"].nunique())
print("Missing labels:", model_data["intent"].isna().sum())

print("\nIntent distribution:")
print(
    model_data["intent"]
    .value_counts()
    .sort_index()
)

# Prepare features and labels.
X_text = model_data["model_text"]
y = model_data["intent"]

# Same folds will be used for every classifier so the comparison is fair.
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=2026
)

print("\nPrepared 200 examples for 5-fold stratified cross-validation.")

Rows: 200
Unique tweet IDs: 200
Unique intents: 12
Missing labels: 0

Intent distribution:
intent
account_authentication    18
apple_services            25
apps_app_store            20
audio                     12
battery_charging          19
camera_photos             11
connectivity              21
device_hardware           15
display_input             35
icloud_backup              6
ios_update                13
other_unclear              5
Name: count, dtype: int64

Prepared 200 examples for 5-fold stratified cross-validation.


In [67]:
# Benchmark multiple intent-classification algorithms on the same 200 reviewed examples.
# Cross-validation prevents us from judging the models only on data they trained on.

import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import accuracy_score, f1_score

# Define the models.
# TF-IDF is fitted separately inside each CV fold to avoid data leakage.
models = {
    "Majority Baseline": Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.95,
            sublinear_tf=True
        )),
        ("classifier", DummyClassifier(
            strategy="most_frequent"
        ))
    ]),

    "TF-IDF + Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.95,
            sublinear_tf=True
        )),
        ("classifier", LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=2026
        ))
    ]),

    "TF-IDF + Linear SVM": Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.95,
            sublinear_tf=True
        )),
        ("classifier", LinearSVC(
            class_weight="balanced",
            random_state=2026
        ))
    ]),

    "TF-IDF + Complement NB": Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.95,
            sublinear_tf=True
        )),
        ("classifier", ComplementNB())
    ])
}

results = []

# Evaluate every model using identical CV splits.
for model_name, model in models.items():

    fold_accuracies = []
    fold_macro_f1 = []
    fold_weighted_f1 = []

    for train_idx, test_idx in cv.split(X_text, y):

        X_train = X_text.iloc[train_idx]
        X_test = X_text.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

        fold_accuracies.append(
            accuracy_score(y_test, predictions)
        )

        fold_macro_f1.append(
            f1_score(
                y_test,
                predictions,
                average="macro",
                zero_division=0
            )
        )

        fold_weighted_f1.append(
            f1_score(
                y_test,
                predictions,
                average="weighted",
                zero_division=0
            )
        )

    results.append({
        "Model": model_name,
        "Accuracy": np.mean(fold_accuracies),
        "Accuracy Std": np.std(fold_accuracies),
        "Macro F1": np.mean(fold_macro_f1),
        "Macro F1 Std": np.std(fold_macro_f1),
        "Weighted F1": np.mean(fold_weighted_f1),
        "Weighted F1 Std": np.std(fold_weighted_f1)
    })

# Display results from strongest to weakest Macro F1.
benchmark_results = (
    pd.DataFrame(results)
    .sort_values("Macro F1", ascending=False)
    .reset_index(drop=True)
)

print("5-Fold Cross-Validation Results")
print("=" * 80)

display(
    benchmark_results.style.format({
        "Accuracy": "{:.3f}",
        "Accuracy Std": "{:.3f}",
        "Macro F1": "{:.3f}",
        "Macro F1 Std": "{:.3f}",
        "Weighted F1": "{:.3f}",
        "Weighted F1 Std": "{:.3f}"
    })
)

5-Fold Cross-Validation Results


,Model,Accuracy,Accuracy Std,Macro F1,Macro F1 Std,Weighted F1,Weighted F1 Std
0,TF-IDF + Linear SVM,0.455,0.046,0.351,0.034,0.417,0.037
1,TF-IDF + Logistic Regression,0.430,0.029,0.332,0.027,0.405,0.028
2,TF-IDF + Complement NB,0.420,0.056,0.291,0.036,0.358,0.042
3,Majority Baseline,0.175,0.000,0.025,0.000,0.052,0.000


In [68]:
# Evaluate the Linear SVM per intent using out-of-fold predictions.
# This shows which intents are easy, which overlap, and where the taxonomy/model needs work.

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

svm_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1,
        max_df=0.95,
        sublinear_tf=True
    )),
    ("classifier", LinearSVC(
        class_weight="balanced",
        random_state=2026
    ))
])

# Store predictions from every held-out fold.
y_true_all = []
y_pred_all = []

for train_idx, test_idx in cv.split(X_text, y):

    X_train = X_text.iloc[train_idx]
    X_test = X_text.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    svm_model.fit(X_train, y_train)
    predictions = svm_model.predict(X_test)

    y_true_all.extend(y_test.tolist())
    y_pred_all.extend(predictions.tolist())

# Generate the per-intent classification report.
report = classification_report(
    y_true_all,
    y_pred_all,
    labels=list(intent_descriptions.keys()),
    output_dict=True,
    zero_division=0
)

per_intent_results = pd.DataFrame(report).T

print("Linear SVM — Out-of-Fold Per-Intent Performance")
print("=" * 70)

display(
    per_intent_results[
        ["precision", "recall", "f1-score", "support"]
    ].style.format({
        "precision": "{:.3f}",
        "recall": "{:.3f}",
        "f1-score": "{:.3f}",
        "support": "{:.0f}"
    })
)

# Build the confusion matrix.
cm = confusion_matrix(
    y_true_all,
    y_pred_all,
    labels=list(intent_descriptions.keys())
)

confusion_df = pd.DataFrame(
    cm,
    index=list(intent_descriptions.keys()),
    columns=list(intent_descriptions.keys())
)

print("\nConfusion Matrix")
display(confusion_df)

Linear SVM — Out-of-Fold Per-Intent Performance


,precision,recall,f1-score,support
ios_update,0.000,0.000,0.000,13
battery_charging,0.615,0.842,0.711,19
connectivity,0.357,0.238,0.286,21
account_authentication,0.545,0.667,0.600,18
icloud_backup,0.000,0.000,0.000,6
apps_app_store,0.481,0.650,0.553,20
audio,0.333,0.083,0.133,12
camera_photos,0.727,0.727,0.727,11
display_input,0.500,0.686,0.578,35
apple_services,0.333,0.280,0.304,25



Confusion Matrix


,ios_update,battery_charging,connectivity,account_authentication,icloud_backup,apps_app_store,audio,camera_photos,display_input,apple_services,device_hardware,other_unclear
ios_update,0,2,2,1,0,0,0,0,4,2,1,1
battery_charging,0,16,0,0,0,1,0,0,1,0,1,0
connectivity,2,1,5,0,0,3,0,0,7,3,0,0
account_authentication,0,0,0,12,0,2,0,0,0,3,0,1
icloud_backup,0,0,1,3,0,0,0,0,0,1,1,0
apps_app_store,0,3,1,0,0,13,0,0,1,1,1,0
audio,1,3,1,0,0,1,1,0,3,1,1,0
camera_photos,0,0,0,1,0,0,0,8,1,1,0,0
display_input,3,0,1,1,0,2,1,0,24,1,2,0
apple_services,3,0,2,2,0,2,1,1,5,7,2,0


In [69]:
# Test a stronger SVM representation using both word and character n-grams.
# Character features help with noisy Twitter text, spelling variations, abbreviations,
# product names, and tokens such as "iOS11", "wifi", "bluetooth", etc.

from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import pandas as pd

word_features = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95,
    sublinear_tf=True,
    max_features=15000
)

char_features = TfidfVectorizer(
    lowercase=True,
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    sublinear_tf=True,
    max_features=20000
)

word_char_svm = Pipeline([
    (
        "features",
        FeatureUnion([
            ("word", word_features),
            ("char", char_features)
        ])
    ),
    (
        "classifier",
        LinearSVC(
            class_weight="balanced",
            C=1.0,
            random_state=2026
        )
    )
])

results_word_char = []

for train_idx, test_idx in cv.split(X_text, y):

    X_train = X_text.iloc[train_idx]
    X_test = X_text.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    word_char_svm.fit(X_train, y_train)

    predictions = word_char_svm.predict(X_test)

    results_word_char.append({
        "accuracy": accuracy_score(y_test, predictions),
        "macro_f1": f1_score(
            y_test,
            predictions,
            average="macro",
            zero_division=0
        ),
        "weighted_f1": f1_score(
            y_test,
            predictions,
            average="weighted",
            zero_division=0
        )
    })

word_char_results = pd.DataFrame(results_word_char)

print("Word + Character TF-IDF + Linear SVM")
print("=" * 60)

print(
    f"Accuracy:    {word_char_results['accuracy'].mean():.3f} "
    f"+/- {word_char_results['accuracy'].std():.3f}"
)

print(
    f"Macro F1:    {word_char_results['macro_f1'].mean():.3f} "
    f"+/- {word_char_results['macro_f1'].std():.3f}"
)

print(
    f"Weighted F1:  {word_char_results['weighted_f1'].mean():.3f} "
    f"+/- {word_char_results['weighted_f1'].std():.3f}"
)

Word + Character TF-IDF + Linear SVM
Accuracy:    0.500 +/- 0.073
Macro F1:    0.419 +/- 0.076
Weighted F1:  0.466 +/- 0.058


In [70]:
# Evaluate the best Word + Character TF-IDF + SVM model per intent.
# This shows which intents are actually learnable and which ones are being confused.

from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import pandas as pd

oof_predictions = np.empty(len(model_data), dtype=object)

for train_idx, test_idx in cv.split(X_text, y):
    X_train = X_text.iloc[train_idx]
    X_test = X_text.iloc[test_idx]
    y_train = y.iloc[train_idx]

    word_char_svm.fit(X_train, y_train)
    oof_predictions[test_idx] = word_char_svm.predict(X_test)

print("Per-intent performance")
print("=" * 70)

report = classification_report(
    y,
    oof_predictions,
    labels=sorted(y.unique()),
    output_dict=True,
    zero_division=0
)

per_intent_results = pd.DataFrame(report).T

print(
    per_intent_results[
        ["precision", "recall", "f1-score", "support"]
    ].round(3)
)

print("\nConfusion Matrix")
print("=" * 70)

labels = sorted(y.unique())

cm = confusion_matrix(
    y,
    oof_predictions,
    labels=labels
)

confusion_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

print(confusion_df)

Per-intent performance
                        precision  recall  f1-score  support
account_authentication      0.650   0.722     0.684     18.0
apple_services              0.318   0.280     0.298     25.0
apps_app_store              0.583   0.700     0.636     20.0
audio                       0.000   0.000     0.000     12.0
battery_charging            0.762   0.842     0.800     19.0
camera_photos               0.833   0.909     0.870     11.0
connectivity                0.412   0.333     0.368     21.0
device_hardware             0.417   0.333     0.370     15.0
display_input               0.490   0.686     0.571     35.0
icloud_backup               1.000   0.333     0.500      6.0
ios_update                  0.133   0.154     0.143     13.0
other_unclear               0.000   0.000     0.000      5.0
accuracy                    0.500   0.500     0.500      0.5
macro avg                   0.467   0.441     0.437    200.0
weighted avg                0.474   0.500     0.477    200.0



In [71]:
# Inspect the actual classification mistakes to determine whether errors come from
# model limitations or ambiguous/overlapping intent definitions.

error_analysis = model_data[
    y.to_numpy() != oof_predictions
].copy()

error_analysis["predicted_intent"] = oof_predictions[
    y.to_numpy() != oof_predictions
]

error_analysis["correct"] = False

error_analysis = error_analysis[
    [
        "customer_tweet_id",
        "customer_message",
        "previous_message",
        "has_context",
        "intent",
        "predicted_intent"
    ]
].reset_index(drop=True)

print("Total misclassified:", len(error_analysis))
print("\nTop confusion pairs:")
print(
    error_analysis
    .groupby(["intent", "predicted_intent"])
    .size()
    .sort_values(ascending=False)
    .head(20)
)

print("\n" + "=" * 100)
print("MISCLASSIFIED EXAMPLES")
print("=" * 100)

pd.set_option("display.max_colwidth", 200)
display(error_analysis)

Total misclassified: 100

Top confusion pairs:
intent           predicted_intent      
connectivity     display_input             6
ios_update       display_input             6
display_input    ios_update                5
apple_services   connectivity              4
                 apps_app_store            4
                 ios_update                3
                 display_input             3
ios_update       connectivity              3
apps_app_store   apple_services            3
other_unclear    display_input             3
display_input    apple_services            3
audio            display_input             3
                 ios_update                2
                 apple_services            2
device_hardware  display_input             2
connectivity     apple_services            2
                 apps_app_store            2
audio            device_hardware           2
device_hardware  account_authentication    2
connectivity     ios_update                2
dtype: int64


,customer_tweet_id,customer_message,previous_message,has_context,intent,predicted_intent
0,611285.0,Hey @115858 how can Siri really be my personal assistant if it won't integrate into 3rd party apps why can't I have it play Spotify where is the functionality?,NaN,False,apple_services,display_input
1,400853.0,@AppleSupport Was able to help him via iTunes. Still weird the OTA didn’t work,@210813 Let us know if the update has issues through iTunes.,True,ios_update,display_input
2,1099710.0,@379507 @AppleSupport Having the same issue.,"@AppleSupport iOS 11.0.3. 2 different iPhone 7s in my family, multiple iPads too. Enter address in Maps &amp; driving directions fail to load—endless spinner.",True,apple_services,apps_app_store
3,1687712.0,"Good afternoon, @AppleSupport! It’s always some shit with y’all and these iPhone updates, huh? https://t.co/cGR2qXnGNA",NaN,False,ios_update,display_input
4,2119414.0,"So wtf y’all gon do about this I situation, @115858? Cuz I’m sick of sick this damn A n shit. 😒",NaN,False,display_input,apple_services
...,...,...,...,...,...,...
95,1906931.0,@AppleSupport what is going on with the new IOS update??? It’s horrible,NaN,False,ios_update,connectivity
96,1642388.0,@AppleSupport It connects successfully but all apps are not functioning including safari.,@389327 What happens while trying to connect to cellular data? Are you receiving an error message?,True,connectivity,apps_app_store
97,2671.0,@AppleSupport @AppleSupport ^^check this out,@AppleSupport Internal problem. Any suggestions on how to help? Im not paying for a new phone because I didn’t cause the damage.,True,device_hardware,display_input
98,2287872.0,@AppleSupport The “done” button will not let me select my picture. It’s covered in the top right corner of the phone with the battery and wifi icons. It gets stuck and I have to close out the app.,@664705 What's happening when you try to change your profile picture?,True,display_input,battery_charging


In [72]:
# Train the final intent classifier on all 200 reviewed examples.
# We use the full labeled set now that model selection is complete.

final_intent_model = Pipeline([
    ("features", FeatureUnion([
        ("word", TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.95,
            sublinear_tf=True,
            max_features=15000
        )),
        ("char", TfidfVectorizer(
            lowercase=True,
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            sublinear_tf=True,
            max_features=20000
        ))
    ])),
    ("classifier", LinearSVC(
        class_weight="balanced",
        C=1.0,
        random_state=2026
    ))
])

final_intent_model.fit(
    model_data["model_text"],
    model_data["intent"]
)

print("Final intent classifier trained.")
print("Training examples:", len(model_data))
print("Intents:", model_data["intent"].nunique())

Final intent classifier trained.
Training examples: 200
Intents: 12


In [73]:
# Build the historical retrieval corpus from AppleSupport interactions.
# We exclude the 200 golden examples so evaluation remains leakage-free.

import pandas as pd

golden_ids = set(
    golden_set["customer_tweet_id"].astype(str)
)

retrieval_corpus = clean_apple[
    ~clean_apple["customer_tweet_id"].astype(str).isin(golden_ids)
].copy()

retrieval_corpus = retrieval_corpus[
    [
        "customer_tweet_id",
        "customer_message",
        "previous_message",
        "has_context",
        "brand_response"
    ]
].reset_index(drop=True)

print("Retrieval corpus size:", len(retrieval_corpus))
print("Golden examples excluded:", len(golden_ids))
print("Remaining retrieval examples:", len(retrieval_corpus))
print("\nMissing customer messages:", retrieval_corpus["customer_message"].isna().sum())
print("Missing brand responses:", retrieval_corpus["brand_response"].isna().sum())

Retrieval corpus size: 106323
Golden examples excluded: 200
Remaining retrieval examples: 106323

Missing customer messages: 0
Missing brand responses: 0


In [74]:
# Prepare the text that will be embedded for semantic retrieval.
# We use the customer's support message as the retrieval key to avoid adding
# unnecessary conversational noise.

retrieval_texts = (
    retrieval_corpus["customer_message"]
    .fillna("")
    .astype(str)
    .str.strip()
    .tolist()
)

print("Texts prepared:", len(retrieval_texts))
print("Example:")
print(retrieval_texts[0])

Texts prepared: 106323
Example:
@AppleSupport  https://t.co/NV0yucs0lB


In [75]:
# Embed the historical customer messages for semantic search.
# Normalized embeddings allow cosine similarity to be computed efficiently with FAISS.

retrieval_embeddings = embedding_model.encode(
    retrieval_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

retrieval_embeddings = retrieval_embeddings.astype("float32")

print("\nEmbedding shape:", retrieval_embeddings.shape)
print("Embedding dtype:", retrieval_embeddings.dtype)

Batches:   0%|          | 0/1662 [00:00<?, ?it/s]


Embedding shape: (106323, 384)
Embedding dtype: float32


In [76]:
# Build a FAISS cosine-similarity index over the historical AppleSupport messages.
# Because embeddings are normalized, inner product is equivalent to cosine similarity.

import faiss

embedding_dimension = retrieval_embeddings.shape[1]

retrieval_index = faiss.IndexFlatIP(
    embedding_dimension
)

retrieval_index.add(
    retrieval_embeddings
)

print("FAISS index built.")
print("Indexed examples:", retrieval_index.ntotal)
print("Embedding dimension:", embedding_dimension)

FAISS index built.
Indexed examples: 106323
Embedding dimension: 384


In [77]:
# Retrieve the most similar historical AppleSupport cases for every golden example.
# We use the golden messages only as queries; their own rows are already excluded
# from the retrieval index.

golden_query_texts = (
    golden_set["customer_message"]
    .fillna("")
    .astype(str)
    .str.strip()
    .tolist()
)

golden_query_embeddings = embedding_model.encode(
    golden_query_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
).astype("float32")

top_k = 5

similarity_scores, nearest_indices = retrieval_index.search(
    golden_query_embeddings,
    top_k
)

print("Query embeddings:", golden_query_embeddings.shape)
print("Retrieved results shape:", nearest_indices.shape)
print("Top-k:", top_k)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Query embeddings: (200, 384)
Retrieved results shape: (200, 5)
Top-k: 5


In [78]:
# Inspect retrieved historical conversations for representative golden examples.
# This helps verify whether semantic similarity corresponds to useful support evidence.

for query_idx in range(min(10, len(golden_set))):
    print("\n" + "=" * 100)
    print(f"GOLDEN EXAMPLE {query_idx}")
    print("=" * 100)

    print("Customer:")
    print(golden_set.iloc[query_idx]["customer_message"])

    print("\nTrue intent:")
    print(golden_set.iloc[query_idx]["intent"])

    print("\nRetrieved historical cases:")

    for rank in range(top_k):
        corpus_idx = nearest_indices[query_idx, rank]
        score = similarity_scores[query_idx, rank]

        row = retrieval_corpus.iloc[corpus_idx]

        print(f"\n--- Rank {rank + 1} | Similarity: {score:.3f} ---")
        print("Historical customer:")
        print(row["customer_message"])
        print("AppleSupport response:")
        print(row["brand_response"])


GOLDEN EXAMPLE 0
Customer:
Hey @115858 how can Siri really be my personal assistant if it won't integrate into 3rd party apps why can't I have it play Spotify where is the functionality?

True intent:
apple_services

Retrieved historical cases:

--- Rank 1 | Similarity: 0.701 ---
Historical customer:
@AppleSupport Hello, where can i see third party app support by Siri on iOS 11 ? I don't see "app support" like iOS 10...
AppleSupport response:
@261160 Let's look into that. Send us a DM and we'll go from there. https://t.co/GDrqU22YpT

--- Rank 2 | Similarity: 0.670 ---
Historical customer:
@AppleSupport asked Siri to play a song. It’s on my iPhone, why is this happening? https://t.co/lsnj4pUZJd
AppleSupport response:
@186705 Let’s take a look at that together.  Are you subscribed to Apple Music?  Does the song play if you select it from the Music App?

--- Rank 3 | Similarity: 0.669 ---
Historical customer:
@115858 @AppleSupport @115948 bring back music rating already. Siri sucks. I am

In [79]:
# Quantify retrieval quality across the 200 golden queries.
# We measure similarity and use the existing intent classifier only as a diagnostic
# for whether retrieved cases are broadly aligned with the query's intent.

retrieved_intents = []

for query_idx in range(len(golden_set)):
    query_retrieved_intents = []

    for rank in range(top_k):
        corpus_idx = nearest_indices[query_idx, rank]
        historical_text = retrieval_corpus.iloc[corpus_idx]["customer_message"]

        predicted_intent = final_intent_model.predict(
            [historical_text]
        )[0]

        query_retrieved_intents.append(predicted_intent)

    retrieved_intents.append(query_retrieved_intents)

retrieved_intents = np.array(retrieved_intents)

golden_intents = golden_set["intent"].to_numpy()

top1_intent_alignment = (
    retrieved_intents[:, 0] == golden_intents
)

top5_intent_alignment = np.array([
    golden_intents[i] in retrieved_intents[i]
    for i in range(len(golden_intents))
])

retrieval_summary = pd.DataFrame({
    "metric": [
        "Mean top-1 similarity",
        "Median top-1 similarity",
        "Top-1 intent alignment",
        "Top-5 intent alignment"
    ],
    "value": [
        similarity_scores[:, 0].mean(),
        np.median(similarity_scores[:, 0]),
        top1_intent_alignment.mean(),
        top5_intent_alignment.mean()
    ]
})

print(retrieval_summary.round(3))

                    metric  value
0    Mean top-1 similarity  0.809
1  Median top-1 similarity  0.811
2   Top-1 intent alignment  0.820
3   Top-5 intent alignment  0.955


In [80]:
# Create a reusable function that retrieves the most relevant historical
# AppleSupport conversations for a new customer message.

def retrieve_historical_cases(
    customer_message,
    top_k=5
):
    query_embedding = embedding_model.encode(
        [customer_message],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = retrieval_index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, (score, corpus_idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):
        row = retrieval_corpus.iloc[corpus_idx]

        results.append({
            "rank": rank,
            "similarity": float(score),
            "customer_message": row["customer_message"],
            "brand_response": row["brand_response"]
        })

    return results


# Quick sanity check using a real golden example.
test_message = golden_set.iloc[9]["customer_message"]

retrieved_cases = retrieve_historical_cases(
    test_message,
    top_k=5
)

print("Customer:")
print(test_message)

print("\nRetrieved cases:")
for case in retrieved_cases:
    print(
        f"\nRank {case['rank']} "
        f"(similarity={case['similarity']:.3f})"
    )
    print("Customer:", case["customer_message"])
    print("AppleSupport:", case["brand_response"])

Customer:
@AppleSupport my phone battery has just gone from 50% - 34% in 15 mins???

Retrieved cases:

Rank 1 (similarity=0.879)
Customer: @AppleSupport Let me explain in detail. My battery went from 100% this morning to only 38% currently. And I just started using my phone 15 minutes ago
AppleSupport: @213152 We'd be glad to take a look at the battery usage you're seeing with your iPhone. Which one do you have? https://t.co/GDrqU22YpT

Rank 2 (similarity=0.858)
Customer: My phone battery goes from 97% to 63% in a matter of mins wtff @AppleSupport
AppleSupport: @392003 Battery life is important. Which iOS version is your device using? Let us know in DM. https://t.co/GDrqU22YpT

Rank 3 (similarity=0.848)
Customer: Hey @115858 @AppleSupport I don’t think my battery is supposed to go from 100% to 56% in 2 hours. I know my phones old but geez. https://t.co/TozPou0HWA
AppleSupport: @470020 We'd love to help with the battery life. Which iPhone do you have and what's the exact iOS version?

R

In [81]:
# Create a reusable function for predicting the customer's intent.
# The final classifier was trained on all 200 reviewed examples.

def predict_intent(customer_message, previous_message=""):
    customer_message = str(customer_message).strip()
    previous_message = str(previous_message).strip()

    if previous_message:
        model_text = (
            f"Previous message: {previous_message} "
            f"Customer message: {customer_message}"
        )
    else:
        model_text = customer_message

    predicted_intent = final_intent_model.predict(
        [model_text]
    )[0]

    return predicted_intent


# Quick sanity check
test_message = golden_set.iloc[9]["customer_message"]

predicted = predict_intent(test_message)

print("Customer:", test_message)
print("Predicted intent:", predicted)

Customer: @AppleSupport my phone battery has just gone from 50% - 34% in 15 mins???
Predicted intent: battery_charging


In [83]:
# Combine intent classification and historical retrieval into one reusable function.
# This gives the generation layer everything it needs without duplicating model logic.

def analyze_customer_message(
    customer_message,
    previous_message=""
):
    customer_message = str(customer_message).strip()
    previous_message = str(previous_message).strip()

    # Predict the customer's primary intent.
    intent = predict_intent(
        customer_message,
        previous_message
    )

    # Retrieve historically similar AppleSupport cases.
    retrieved_cases = retrieve_historical_cases(
        customer_message,
        top_k=5
    )

    return {
        "customer_message": customer_message,
        "previous_message": previous_message,
        "intent": intent,
        "retrieved_cases": retrieved_cases,
        "top_similarity": retrieved_cases[0]["similarity"]
    }


# Test the complete analysis pipeline.
test_message = golden_set.iloc[9]["customer_message"]

analysis = analyze_customer_message(
    test_message
)

print("Customer:")
print(analysis["customer_message"])

print("\nPredicted intent:")
print(analysis["intent"])

print("\nTop similarity:")
print(f"{analysis['top_similarity']:.3f}")

print("\nHistorical evidence:")
for case in analysis["retrieved_cases"]:
    print(
        f"\n[{case['rank']}] similarity={case['similarity']:.3f}"
    )
    print("Customer:", case["customer_message"])
    print("AppleSupport:", case["brand_response"])

Customer:
@AppleSupport my phone battery has just gone from 50% - 34% in 15 mins???

Predicted intent:
battery_charging

Top similarity:
0.879

Historical evidence:

[1] similarity=0.879
Customer: @AppleSupport Let me explain in detail. My battery went from 100% this morning to only 38% currently. And I just started using my phone 15 minutes ago
AppleSupport: @213152 We'd be glad to take a look at the battery usage you're seeing with your iPhone. Which one do you have? https://t.co/GDrqU22YpT

[2] similarity=0.858
Customer: My phone battery goes from 97% to 63% in a matter of mins wtff @AppleSupport
AppleSupport: @392003 Battery life is important. Which iOS version is your device using? Let us know in DM. https://t.co/GDrqU22YpT

[3] similarity=0.848
Customer: Hey @115858 @AppleSupport I don’t think my battery is supposed to go from 100% to 56% in 2 hours. I know my phones old but geez. https://t.co/TozPou0HWA
AppleSupport: @470020 We'd love to help with the battery life. Which iPhone 

In [84]:
# Build a grounded prompt from the customer message, predicted intent, and
# historically retrieved AppleSupport responses.
# The prompt explicitly prevents the LLM from inventing unsupported solutions.

import json


def build_agent_prompt(analysis):
    evidence = []

    for case in analysis["retrieved_cases"]:
        evidence.append({
            "similarity": round(case["similarity"], 3),
            "historical_customer": case["customer_message"],
            "historical_response": case["brand_response"]
        })

    prompt = f"""
You are an AI customer-support agent for AppleSupport.

Your job is to draft a helpful support response based ONLY on:
1. The customer's current message.
2. The predicted support intent.
3. Historical AppleSupport interactions provided as evidence.

Do not invent Apple policies, troubleshooting steps, refunds, guarantees,
product capabilities, or URLs that are not supported by the evidence.

You may synthesize patterns across multiple historical responses.
Do not copy usernames, tweet IDs, or social-media metadata from the evidence.

Customer message:
{analysis["customer_message"]}

Predicted intent:
{analysis["intent"]}

Historical AppleSupport evidence:
{json.dumps(evidence, indent=2)}

Return ONLY valid JSON with this structure:

{{
    "intent": "predicted intent",
    "draft_reply": "A concise customer-facing support response",
    "evidence_used": [
        "Brief description of the historical pattern used"
    ],
    "escalate": true,
    "escalation_reason": "Why a human should or should not handle this case"
}}

Rules for escalation:
- Set escalate=true when the historical evidence suggests moving to DM,
  gathering account/device-specific information, or when the issue cannot
  be safely resolved from the available information.
- Set escalate=false only when the response can reasonably be handled
  using the historical evidence without requiring human intervention.
- Give a specific reason for the decision.

Keep the draft reply concise and natural.
"""

    return prompt


# Test the prompt without calling an LLM.
test_analysis = analyze_customer_message(
    golden_set.iloc[9]["customer_message"]
)

agent_prompt = build_agent_prompt(test_analysis)

print(agent_prompt)


You are an AI customer-support agent for AppleSupport.

Your job is to draft a helpful support response based ONLY on:
1. The customer's current message.
2. The predicted support intent.
3. Historical AppleSupport interactions provided as evidence.

Do not invent Apple policies, troubleshooting steps, refunds, guarantees,
product capabilities, or URLs that are not supported by the evidence.

You may synthesize patterns across multiple historical responses.
Do not copy usernames, tweet IDs, or social-media metadata from the evidence.

Customer message:
@AppleSupport my phone battery has just gone from 50% - 34% in 15 mins???

Predicted intent:
battery_charging

Historical AppleSupport evidence:
[
  {
    "similarity": 0.879,
    "historical_customer": "@AppleSupport Let me explain in detail. My battery went from 100% this morning to only 38% currently. And I just started using my phone 15 minutes ago",
    "historical_response": "@213152 We'd be glad to take a look at the battery usage

In [85]:
# Install the Google GenAI SDK used by the support-agent generation layer.
%pip install -q google-genai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [87]:
# Load the project's .env file and configure the Gemini client.
# The explicit path ensures the notebook finds the .env file in the project root.

import os
import json
from pathlib import Path
from dotenv import load_dotenv
from google import genai

# Project root = parent directory of notebooks/
project_root = Path.cwd().parent

env_path = project_root / ".env"

if not env_path.exists():
    raise FileNotFoundError(
        f".env file not found at: {env_path}"
    )

load_dotenv(env_path)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY was not found inside the .env file."
    )

gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("Gemini client configured successfully.")

Gemini client configured successfully.


In [90]:
# Test the support-agent generation using an available Gemini Flash model.
# We keep the same grounded prompt and structured JSON output.

def generate_support_response(
    analysis,
    model="gemini-3.5-flash"
):
    prompt = build_agent_prompt(analysis)

    response = gemini_client.models.generate_content(
        model=model,
        contents=prompt,
        config={
            "temperature": 0.2,
            "response_mime_type": "application/json"
        }
    )

    return json.loads(response.text)


# Test the complete agent on a golden example.
test_analysis = analyze_customer_message(
    golden_set.iloc[9]["customer_message"]
)

agent_result = generate_support_response(
    test_analysis
)

print(json.dumps(agent_result, indent=2))

{
  "intent": "battery_charging",
  "draft_reply": "We'd love to help look into your iPhone's battery life. Which iPhone model and iOS version are you currently using? Let us know in DM. https://t.co/GDrqU22YpT",
  "evidence_used": [
    "Historical responses consistently ask for the iPhone model and iOS version, and direct the customer to DM to take a closer look."
  ],
  "escalate": true,
  "escalation_reason": "The issue requires gathering device-specific information (iPhone model and iOS version) and moving the conversation to DM for personalized troubleshooting."
}


In [91]:
# Add a confidence signal based on the Linear SVM decision margin.
# A larger margin means the classifier is more separated from competing intents.

def predict_intent_with_confidence(
    customer_message,
    previous_message=""
):
    customer_message = str(customer_message).strip()
    previous_message = str(previous_message).strip()

    if previous_message:
        model_text = (
            f"Previous message: {previous_message} "
            f"Customer message: {customer_message}"
        )
    else:
        model_text = customer_message

    predicted_intent = final_intent_model.predict(
        [model_text]
    )[0]

    decision_scores = final_intent_model.decision_function(
        [model_text]
    )[0]

    sorted_scores = np.sort(decision_scores)[::-1]

    top_score = sorted_scores[0]
    second_score = sorted_scores[1]

    margin = float(top_score - second_score)

    return {
        "intent": predicted_intent,
        "margin": margin,
        "top_score": float(top_score),
        "second_score": float(second_score)
    }


# Test the confidence calculation.
test_message = golden_set.iloc[9]["customer_message"]

confidence = predict_intent_with_confidence(
    test_message
)

print("Customer:")
print(test_message)

print("\nPrediction:")
print(confidence)

Customer:
@AppleSupport my phone battery has just gone from 50% - 34% in 15 mins???

Prediction:
{'intent': 'battery_charging', 'margin': 1.7100865244866146, 'top_score': 0.7627941949075733, 'second_score': -0.9472923295790412}


In [92]:
# Measure classifier margins across all 200 golden examples.
# We use this distribution to choose a defensible confidence threshold for escalation.

confidence_records = []

for _, row in golden_set.iterrows():
    prediction = predict_intent_with_confidence(
        row["customer_message"],
        row["previous_message"]
    )

    confidence_records.append({
        "customer_tweet_id": row["customer_tweet_id"],
        "true_intent": row["intent"],
        "predicted_intent": prediction["intent"],
        "margin": prediction["margin"],
        "correct": prediction["intent"] == row["intent"]
    })

confidence_df = pd.DataFrame(confidence_records)

print("Classifier margin statistics")
print("=" * 60)
print(
    confidence_df["margin"]
    .describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90])
    .round(3)
)

print("\nMargin by prediction correctness")
print("=" * 60)
print(
    confidence_df
    .groupby("correct")["margin"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(3)
)

Classifier margin statistics
count    200.000
mean       1.325
std        0.189
min        0.715
10%        1.077
25%        1.207
50%        1.366
75%        1.452
90%        1.539
max        1.740
Name: margin, dtype: float64

Margin by prediction correctness
         count   mean  median    min   max
correct                                   
True       200  1.325   1.366  0.715  1.74


In [94]:
# Compute out-of-fold classifier margins for the 200 golden examples.
# Each prediction comes from a model that did not train on that example,
# making this suitable for analyzing confidence and escalation behavior.

oof_predictions = np.empty(len(model_data), dtype=object)
oof_margins = np.zeros(len(model_data))

for train_idx, test_idx in cv.split(X_text, y):
    X_train = X_text.iloc[train_idx]
    X_test = X_text.iloc[test_idx]
    y_train = y.iloc[train_idx]

    word_char_svm.fit(
        X_train,
        y_train
    )

    predictions = word_char_svm.predict(X_test)
    decision_scores = word_char_svm.decision_function(X_test)

    # Sort each row's decision scores to calculate the margin
    # between the best and second-best intent.
    sorted_scores = np.sort(
        decision_scores,
        axis=1
    )[:, ::-1]

    margins = (
        sorted_scores[:, 0]
        - sorted_scores[:, 1]
    )

    oof_predictions[test_idx] = predictions
    oof_margins[test_idx] = margins


confidence_df = pd.DataFrame({
    "customer_tweet_id": model_data["customer_tweet_id"].values,
    "true_intent": model_data["intent"].values,
    "predicted_intent": oof_predictions,
    "margin": oof_margins,
    "correct": (
        oof_predictions == model_data["intent"].values
    )
})

print("OOF classifier margin statistics")
print("=" * 60)

print(
    confidence_df["margin"]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
    .round(3)
)

print("\nMargin by prediction correctness")
print("=" * 60)

print(
    confidence_df
    .groupby("correct")["margin"]
    .agg([
        "count",
        "mean",
        "median",
        "min",
        "max"
    ])
    .round(3)
)

OOF classifier margin statistics
count    200.000
mean       0.286
std        0.291
min        0.000
10%        0.021
25%        0.067
50%        0.191
75%        0.420
90%        0.721
max        1.419
Name: margin, dtype: float64

Margin by prediction correctness
         count   mean  median   min    max
correct                                   
False      100  0.142   0.094  0.00  0.576
True       100  0.431   0.363  0.01  1.419


In [95]:
# Inspect the lowest-confidence out-of-fold examples.
# These examples help us understand when the classifier is uncertain and
# provide evidence for the escalation policy.

low_confidence_examples = (
    confidence_df
    .sort_values("margin")
    .head(20)
    .merge(
        golden_set[
            [
                "customer_tweet_id",
                "customer_message"
            ]
        ],
        on="customer_tweet_id",
        how="left"
    )
)

pd.set_option(
    "display.max_colwidth",
    200
)

display(
    low_confidence_examples[
        [
            "customer_tweet_id",
            "customer_message",
            "true_intent",
            "predicted_intent",
            "margin",
            "correct"
        ]
    ]
)

,customer_tweet_id,customer_message,true_intent,predicted_intent,margin,correct
0,1866369.0,@AppleSupport @115948 Here are my Apple Music settings: https://t.co/QxyWJRJN8S,apple_services,ios_update,0.000479,False
1,2913352.0,@AppleSupport my iMessage has not worked in over a month at this point. I’ve spent considerable amount of time with your customer support to get this solved but there has been no solution yet. Ple...,apple_services,ios_update,0.001496,False
2,511125.0,@AppleSupport help I got a green line on the left side of my iPhone X screen and restarting it doesn’t make it go away,display_input,apple_services,0.003941,False
3,1819155.0,"@115858 the new update is ass. GPS doesn’t work, music controls don’t work, Apple Watch won’t connect, speaker phone doesn’t work.",device_hardware,apple_services,0.008818,False
4,1890537.0,"@115858 please say your developers are hard at work on iOS 11.0.4 so that my headphones, charger and volume buttons work again, ughhhh 🙄🙄🙄🙄🙄",audio,apple_services,0.009262,False
5,2799435.0,@115858 @6990 why isn’t FaceTime working right now? I need to see @107462 face 😭😭😭,apple_services,connectivity,0.009492,False
6,2119414.0,"So wtf y’all gon do about this I situation, @115858? Cuz I’m sick of sick this damn A n shit. 😒",display_input,apple_services,0.009502,False
7,2467693.0,"@AppleSupport Not sure, it def happens when not plugged in. Never replaced screen. Seems it's a common issue/bug w the update",display_input,display_input,0.009880,True
8,2422571.0,Clearly need to stop using a first-person pronoun until @115858 can get their shit together.,display_input,display_input,0.010089,True
9,1605041.0,"Hi @AppleSupport ,Apple Watch 3 has bug, restarts when you ask Siri on weather.please address this",device_hardware,apple_services,0.010513,False


In [96]:
# Evaluate several classifier-margin thresholds for escalation.
# We compare coverage (cases handled automatically) against accuracy on
# the cases that would be auto-handled.

thresholds = [
    0.01,
    0.02,
    0.05,
    0.10,
    0.15,
    0.20,
    0.30,
    0.40,
    0.50
]

threshold_results = []

for threshold in thresholds:
    auto_handle = confidence_df["margin"] >= threshold

    coverage = auto_handle.mean()

    if auto_handle.sum() > 0:
        auto_accuracy = confidence_df.loc[
            auto_handle,
            "correct"
        ].mean()
    else:
        auto_accuracy = 0.0

    escalation_rate = 1 - coverage

    threshold_results.append({
        "threshold": threshold,
        "auto_handle_count": int(auto_handle.sum()),
        "auto_handle_rate": coverage,
        "escalation_rate": escalation_rate,
        "auto_handle_accuracy": auto_accuracy
    })

threshold_df = pd.DataFrame(threshold_results)

print(
    threshold_df.round(3).to_string(index=False)
)

 threshold  auto_handle_count  auto_handle_rate  escalation_rate  auto_handle_accuracy
      0.01                192             0.960            0.040                 0.516
      0.02                180             0.900            0.100                 0.533
      0.05                161             0.805            0.195                 0.571
      0.10                128             0.640            0.360                 0.641
      0.15                111             0.555            0.445                 0.676
      0.20                 99             0.495            0.505                 0.707
      0.30                 71             0.355            0.645                 0.845
      0.40                 53             0.265            0.735                 0.868
      0.50                 41             0.205            0.795                 0.902


In [97]:
# Define the final conservative escalation policy.
# A case is auto-handled only when the classifier is confident and the
# retrieved historical evidence is sufficiently strong.

INTENT_MARGIN_THRESHOLD = 0.30
RETRIEVAL_SIMILARITY_THRESHOLD = 0.65


def decide_escalation(
    intent,
    classifier_margin,
    top_similarity,
    customer_message
):
    customer_message = str(customer_message).strip().lower()

    reasons = []

    # Very short or vague messages lack enough information for safe automation.
    vague_patterns = [
        "same issue",
        "same problem",
        "please help",
        "help me",
        "what is going on",
        "what's going on"
    ]

    is_vague = (
        len(customer_message.split()) <= 8
        or any(
            pattern in customer_message
            for pattern in vague_patterns
        )
    )

    if is_vague:
        reasons.append(
            "The customer message does not provide enough specific information."
        )

    # Escalate when competing intents are too close.
    if classifier_margin < INTENT_MARGIN_THRESHOLD:
        reasons.append(
            "The intent classifier is not sufficiently confident."
        )

    # Escalate when historical evidence is weak.
    if top_similarity < RETRIEVAL_SIMILARITY_THRESHOLD:
        reasons.append(
            "No sufficiently similar historical support case was retrieved."
        )

    # These categories can require personalized investigation.
    if intent in {
        "account_authentication",
        "device_hardware"
    }:
        reasons.append(
            "The issue may require account- or device-specific investigation."
        )

    escalate = len(reasons) > 0

    if escalate:
        escalation_reason = " ".join(reasons)
    else:
        escalation_reason = (
            "The issue has sufficient intent confidence and relevant "
            "historical support evidence for automated handling."
        )

    return {
        "escalate": escalate,
        "escalation_reason": escalation_reason
    }


print("Escalation policy configured.")
print("Intent margin threshold:", INTENT_MARGIN_THRESHOLD)
print(
    "Retrieval similarity threshold:",
    RETRIEVAL_SIMILARITY_THRESHOLD
)

Escalation policy configured.
Intent margin threshold: 0.3
Retrieval similarity threshold: 0.65


In [98]:
# Combine classification, historical retrieval, grounded LLM generation,
# and the deterministic escalation policy into one support-agent function.

def support_agent(
    customer_message,
    previous_message=""
):
    customer_message = str(customer_message).strip()
    previous_message = str(previous_message).strip()

    # ---------------------------------------------------------
    # 1. Intent classification
    # ---------------------------------------------------------
    intent_prediction = predict_intent_with_confidence(
        customer_message,
        previous_message
    )

    intent = intent_prediction["intent"]
    classifier_margin = intent_prediction["margin"]

    # ---------------------------------------------------------
    # 2. Historical retrieval
    # ---------------------------------------------------------
    retrieved_cases = retrieve_historical_cases(
        customer_message,
        top_k=5
    )

    top_similarity = retrieved_cases[0]["similarity"]

    # ---------------------------------------------------------
    # 3. Build analysis object for the LLM
    # ---------------------------------------------------------
    analysis = {
        "customer_message": customer_message,
        "previous_message": previous_message,
        "intent": intent,
        "retrieved_cases": retrieved_cases,
        "top_similarity": top_similarity
    }

    # ---------------------------------------------------------
    # 4. Generate grounded response
    # ---------------------------------------------------------
    llm_result = generate_support_response(
        analysis
    )

    # ---------------------------------------------------------
    # 5. Deterministic escalation decision
    # ---------------------------------------------------------
    escalation = decide_escalation(
        intent=intent,
        classifier_margin=classifier_margin,
        top_similarity=top_similarity,
        customer_message=customer_message
    )

    # ---------------------------------------------------------
    # 6. Return one structured agent result
    # ---------------------------------------------------------
    return {
        "intent": intent,
        "classifier_margin": round(
            classifier_margin,
            4
        ),
        "top_similarity": round(
            top_similarity,
            4
        ),
        "draft_reply": llm_result["draft_reply"],
        "evidence_used": llm_result.get(
            "evidence_used",
            []
        ),
        "escalate": escalation["escalate"],
        "escalation_reason": escalation[
            "escalation_reason"
        ],
        "retrieved_cases": retrieved_cases
    }


print("Support agent pipeline ready.")

Support agent pipeline ready.


In [99]:
# Test the complete support agent on a realistic unseen customer message.
# This verifies that classification, retrieval, generation, and escalation
# work together as a single pipeline.

test_customer_message = (
    "@AppleSupport my iPhone battery is draining really fast "
    "since the latest update. It lost 30% in less than an hour."
)

agent_output = support_agent(
    test_customer_message
)

print(json.dumps(
    agent_output,
    indent=2
))

{
  "intent": "battery_charging",
  "classifier_margin": 0.8946,
  "top_similarity": 0.8907,
  "draft_reply": "We want to help get the most out of your iPhone's battery. To start, check out these tips to help maximize your battery life: https://t.co/bivpdfkckw. If you're still having trouble, please DM us with your device model and current iOS version so we can look into this further. https://t.co/GDrqU22YpT",
  "evidence_used": [
    "Offered battery maximizing tips using the provided links and requested the customer to DM their device model and iOS version for further assistance."
  ],
  "escalate": false,
  "escalation_reason": "The issue has sufficient intent confidence and relevant historical support evidence for automated handling.",
  "retrieved_cases": [
    {
      "rank": 1,
      "similarity": 0.8907265663146973,
      "customer_message": "@AppleSupport ever since the new update my battery has been draining extremely fast. It\u2019s gone down 2% in less than 5 minutes and I 

In [100]:
# Run the complete agent on five golden examples as an evaluation smoke test.
# We test the pipeline before making the full 200-example evaluation expensive.

evaluation_sample = golden_set.sample(
    n=5,
    random_state=2026
).reset_index(drop=True)

evaluation_results = []

for idx, row in evaluation_sample.iterrows():
    print(f"Evaluating example {idx + 1}/5...")

    result = support_agent(
        customer_message=row["customer_message"],
        previous_message=row["previous_message"]
    )

    evaluation_results.append({
        "customer_tweet_id": row["customer_tweet_id"],
        "customer_message": row["customer_message"],
        "true_intent": row["intent"],
        "predicted_intent": result["intent"],
        "classifier_margin": result["classifier_margin"],
        "top_similarity": result["top_similarity"],
        "draft_reply": result["draft_reply"],
        "evidence_used": result["evidence_used"],
        "escalate": result["escalate"],
        "escalation_reason": result["escalation_reason"]
    })

evaluation_sample_results = pd.DataFrame(
    evaluation_results
)

print("\nEvaluation smoke test complete.")
display(evaluation_sample_results)

Evaluating example 1/5...
Evaluating example 2/5...
Evaluating example 3/5...
Evaluating example 4/5...
Evaluating example 5/5...

Evaluation smoke test complete.


,customer_tweet_id,customer_message,true_intent,predicted_intent,classifier_margin,top_similarity,draft_reply,evidence_used,escalate,escalation_reason
0,50820.0,@AppleSupport I have Iphone 7 updated to ios 11.1. After the news IOS released this year battery problem raised.,battery_charging,battery_charging,1.5940,0.8748,We'd like to help look into what's going on with your iPhone battery. Please meet us in DM so we can work together to find a solution. https://t.co/GDrqU22YpT,[Historical responses to iPhone 7 battery issues after iOS 11 updates consistently invite the customer to DM to investigate the power consumption issues.],False,The issue has sufficient intent confidence and relevant historical support evidence for automated handling.
1,864428.0,@AppleSupport I CAN! I have my device normal.,device_hardware,device_hardware,1.5632,0.8509,"Alright, thanks! Join us in DM with your device model so we can continue to troubleshoot there. https://t.co/GDrqU22YpT","[When a customer confirms they can perform an action or provides device status, the support agent invites them to DM with their device models to continue troubleshooting.]",True,The customer message does not provide enough specific information. The issue may require account- or device-specific investigation.
2,517208.0,"Wow was on hold for like 10 min with @AppleSupport and they hung up on me , been tryin to recover my Apple ID password all week 😒😒",account_authentication,account_authentication,1.2047,0.7825,This certainly isn't the experience we want you to have. We'd love to help in any way we can with your Apple ID. Please send us a DM with the details of what's going on so we can look into this to...,[Acknowledged the poor experience of being disconnected on hold and invited the customer to DM with details about their Apple ID issue to resolve it together.],True,The issue may require account- or device-specific investigation.
3,2295846.0,So @115858 is really about to have me become an #Android user. These glitches are too much and a pain!,other_unclear,other_unclear,1.3983,0.8885,"We'd be happy to help with any issues you're experiencing. Can you tell us what's going on, or DM us with details about what's happening?","[When customers complain about unspecified glitches, historical responses ask them to describe what they are experiencing or invite them to DM with details.]",False,The issue has sufficient intent confidence and relevant historical support evidence for automated handling.
4,2279207.0,@115858 hey sooo um....y’all going to fix this update so i️ can charge my phone? 🧐,battery_charging,battery_charging,1.1337,0.8627,"We want your iPhone working for you, and a software update shouldn't cause the device not to charge. Let's figure out what's going on together. To start, are you using an Apple Certified charging ...","[Acknowledged that software updates shouldn't cause charging issues, asked about using an Apple Certified charging cord, and invited the customer to DM for further troubleshooting.]",False,The issue has sufficient intent confidence and relevant historical support evidence for automated handling.


In [101]:
# Refine the escalation policy so vague/unclear intents are escalated even
# when the classifier margin and retrieval similarity appear strong.

def decide_escalation(
    intent,
    classifier_margin,
    top_similarity,
    customer_message
):
    customer_message = str(customer_message).strip().lower()

    reasons = []

    # Very short or vague messages do not contain enough information
    # for safe automated support.
    vague_patterns = [
        "same issue",
        "same problem",
        "having the same",
        "please help",
        "help me",
        "what is going on",
        "what's going on"
    ]

    is_vague = (
        len(customer_message.split()) <= 8
        or any(
            pattern in customer_message
            for pattern in vague_patterns
        )
    )

    if is_vague:
        reasons.append(
            "The customer message does not provide enough specific information."
        )

    # The catch-all intent should not be automatically handled because
    # the actual customer problem is unknown.
    if intent == "other_unclear":
        reasons.append(
            "The customer's issue could not be confidently mapped to a specific support intent."
        )

    # Low classifier margin indicates competing intents.
    if classifier_margin < INTENT_MARGIN_THRESHOLD:
        reasons.append(
            "The intent classifier is not sufficiently confident."
        )

    # Weak historical evidence makes grounded generation less reliable.
    if top_similarity < RETRIEVAL_SIMILARITY_THRESHOLD:
        reasons.append(
            "No sufficiently similar historical support case was retrieved."
        )

    # Account/device-specific issues may require information unavailable
    # from the public customer message.
    if intent in {
        "account_authentication",
        "device_hardware"
    }:
        reasons.append(
            "The issue may require account- or device-specific investigation."
        )

    escalate = len(reasons) > 0

    if escalate:
        escalation_reason = " ".join(reasons)
    else:
        escalation_reason = (
            "The issue has sufficient intent confidence and relevant "
            "historical support evidence for automated handling."
        )

    return {
        "escalate": escalate,
        "escalation_reason": escalation_reason
    }


print("Escalation policy updated.")

Escalation policy updated.


In [102]:
# Re-run the five-example smoke test after updating the escalation policy.
# This verifies that unclear cases are now routed more safely.

evaluation_results_updated = []

for idx, row in evaluation_sample.iterrows():
    print(f"Evaluating example {idx + 1}/5...")

    result = support_agent(
        customer_message=row["customer_message"],
        previous_message=row["previous_message"]
    )

    evaluation_results_updated.append({
        "customer_tweet_id": row["customer_tweet_id"],
        "true_intent": row["intent"],
        "predicted_intent": result["intent"],
        "classifier_margin": result["classifier_margin"],
        "top_similarity": result["top_similarity"],
        "draft_reply": result["draft_reply"],
        "escalate": result["escalate"],
        "escalation_reason": result["escalation_reason"]
    })

evaluation_sample_results = pd.DataFrame(
    evaluation_results_updated
)

display(evaluation_sample_results)

Evaluating example 1/5...
Evaluating example 2/5...
Evaluating example 3/5...
Evaluating example 4/5...
Evaluating example 5/5...


,customer_tweet_id,true_intent,predicted_intent,classifier_margin,top_similarity,draft_reply,escalate,escalation_reason
0,50820.0,battery_charging,battery_charging,1.5940,0.8748,We're here to help. Let's take a deeper look at what's going on with your iPhone battery in DM. https://t.co/GDrqU22YpT,False,The issue has sufficient intent confidence and relevant historical support evidence for automated handling.
1,864428.0,device_hardware,device_hardware,1.5632,0.8509,"Alright, thanks! Join us in DM with your device model and we can continue to troubleshoot there. https://t.co/GDrqU22YpT",True,The customer message does not provide enough specific information. The issue may require account- or device-specific investigation.
2,517208.0,account_authentication,account_authentication,1.2047,0.7825,"This certainly isn't the experience we want you to have. We'd love to help with your Apple ID. Please send us a DM with details of what's going on, and we'll look into this together. https://t.co/...",True,The issue may require account- or device-specific investigation.
3,2295846.0,other_unclear,other_unclear,1.3983,0.8885,We'd be happy to help. Can you tell us what you're experiencing? Please DM us with details about what's happening.,True,The customer's issue could not be confidently mapped to a specific support intent.
4,2279207.0,battery_charging,battery_charging,1.1337,0.8627,"We want your iPhone working for you, and a software update shouldn't cause the device not to charge. Let's figure out what's going on together. Please DM us details about what's happening so we ca...",False,The issue has sufficient intent confidence and relevant historical support evidence for automated handling.


In [103]:
# Run the complete support agent over all 200 golden examples.
# Results are checkpointed after every example so interrupted API runs can resume.

from pathlib import Path
import time
import json
import pandas as pd

evaluation_dir = project_root / "data" / "evaluation"
evaluation_dir.mkdir(
    parents=True,
    exist_ok=True
)

evaluation_path = (
    evaluation_dir / "apple_support_agent_results.csv"
)

# Load previous progress if it exists.
if evaluation_path.exists():
    completed_results = pd.read_csv(
        evaluation_path
    )

    completed_ids = set(
        completed_results["customer_tweet_id"]
        .astype(str)
    )

    print(
        f"Found {len(completed_results)} previously completed examples."
    )

else:
    completed_results = pd.DataFrame()
    completed_ids = set()

print(
    f"Total golden examples: {len(golden_set)}"
)

remaining = golden_set[
    ~golden_set["customer_tweet_id"]
    .astype(str)
    .isin(completed_ids)
].copy()

print(
    f"Remaining examples: {len(remaining)}"
)

for idx, row in remaining.iterrows():

    print(
        f"Evaluating {idx + 1}/{len(golden_set)} "
        f"(ID: {row['customer_tweet_id']})..."
    )

    try:
        result = support_agent(
            customer_message=row["customer_message"],
            previous_message=row["previous_message"]
        )

        new_result = {
            "customer_tweet_id": row["customer_tweet_id"],
            "customer_message": row["customer_message"],
            "true_intent": row["intent"],
            "predicted_intent": result["intent"],
            "classifier_margin": result["classifier_margin"],
            "top_similarity": result["top_similarity"],
            "draft_reply": result["draft_reply"],
            "evidence_used": json.dumps(
                result["evidence_used"],
                ensure_ascii=False
            ),
            "escalate": result["escalate"],
            "escalation_reason": result["escalation_reason"]
        }

        completed_results = pd.concat(
            [
                completed_results,
                pd.DataFrame([new_result])
            ],
            ignore_index=True
        )

        # Save immediately after every successful API call.
        completed_results.to_csv(
            evaluation_path,
            index=False
        )

        print("Saved.")

        # Small delay to avoid unnecessarily rapid API requests.
        time.sleep(0.5)

    except Exception as e:

        print(
            f"ERROR on {row['customer_tweet_id']}: {e}"
        )

        print(
            "Progress has already been saved. "
            "You can rerun this cell to continue."
        )

        break


print("\nEvaluation run finished.")
print(
    "Completed examples:",
    len(completed_results)
)
print(
    "Saved to:",
    evaluation_path
)

Total golden examples: 200
Remaining examples: 200
Evaluating 1/200 (ID: 611285.0)...
Saved.
Evaluating 2/200 (ID: 400853.0)...
Saved.
Evaluating 3/200 (ID: 1099710.0)...
Saved.
Evaluating 4/200 (ID: 1342708.0)...
Saved.
Evaluating 5/200 (ID: 1687712.0)...
Saved.
Evaluating 6/200 (ID: 1121644.0)...
Saved.
Evaluating 7/200 (ID: 1747356.0)...
Saved.
Evaluating 8/200 (ID: 2119414.0)...
Saved.
Evaluating 9/200 (ID: 1983621.0)...
Saved.
Evaluating 10/200 (ID: 323849.0)...
ERROR on 323849.0: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 40.078959569s.', 'status': 'RESOURCE_EXHAUSTED', 'detai

In [104]:
# Evaluate the full 200-example golden set locally without making any Gemini API calls.
# This gives us the core metrics before we spend any more LLM quota.

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Use the existing golden set and existing trained models/functions.
evaluation_base = golden_set[
    [
        "customer_tweet_id",
        "customer_message",
        "previous_message",
        "intent"
    ]
].copy()

evaluation_rows = []

for _, row in evaluation_base.iterrows():
    intent_prediction = predict_intent_with_confidence(
        customer_message=row["customer_message"],
        previous_message=row["previous_message"]
    )

    retrieved_cases = retrieve_historical_cases(
        row["customer_message"],
        top_k=5
    )

    top_similarity = retrieved_cases[0]["similarity"]

    escalation = decide_escalation(
        intent=intent_prediction["intent"],
        classifier_margin=intent_prediction["margin"],
        top_similarity=top_similarity,
        customer_message=row["customer_message"]
    )

    evaluation_rows.append({
        "customer_tweet_id": row["customer_tweet_id"],
        "customer_message": row["customer_message"],
        "true_intent": row["intent"],
        "predicted_intent": intent_prediction["intent"],
        "classifier_margin": intent_prediction["margin"],
        "top_similarity": top_similarity,
        "escalate": escalation["escalate"],
        "escalation_reason": escalation["escalation_reason"]
    })

local_eval = pd.DataFrame(evaluation_rows)

print("Completed:", len(local_eval))
print()

# ---------------------------------------------------------
# Intent classification metrics
# ---------------------------------------------------------

intent_accuracy = accuracy_score(
    local_eval["true_intent"],
    local_eval["predicted_intent"]
)

macro_f1 = f1_score(
    local_eval["true_intent"],
    local_eval["predicted_intent"],
    average="macro"
)

weighted_f1 = f1_score(
    local_eval["true_intent"],
    local_eval["predicted_intent"],
    average="weighted"
)

print("INTENT CLASSIFICATION")
print("---------------------")
print(f"Accuracy:    {intent_accuracy:.3f}")
print(f"Macro F1:    {macro_f1:.3f}")
print(f"Weighted F1: {weighted_f1:.3f}")

print("\nClassification report:")
print(
    classification_report(
        local_eval["true_intent"],
        local_eval["predicted_intent"],
        zero_division=0
    )
)

# ---------------------------------------------------------
# Escalation metrics
# ---------------------------------------------------------

auto_handled = local_eval[local_eval["escalate"] == False]
escalated = local_eval[local_eval["escalate"] == True]

auto_handle_rate = len(auto_handled) / len(local_eval)
escalation_rate = len(escalated) / len(local_eval)

if len(auto_handled) > 0:
    auto_handle_accuracy = accuracy_score(
        auto_handled["true_intent"],
        auto_handled["predicted_intent"]
    )
else:
    auto_handle_accuracy = 0.0

print("\nESCALATION POLICY")
print("-----------------")
print(f"Auto-handled:          {len(auto_handled)}/{len(local_eval)}")
print(f"Auto-handle rate:      {auto_handle_rate:.3f}")
print(f"Escalated:             {len(escalated)}/{len(local_eval)}")
print(f"Escalation rate:       {escalation_rate:.3f}")
print(f"Auto-handle accuracy:  {auto_handle_accuracy:.3f}")

# ---------------------------------------------------------
# Retrieval diagnostics
# ---------------------------------------------------------

print("\nRETRIEVAL")
print("---------")
print(
    f"Mean top-1 similarity:   "
    f"{local_eval['top_similarity'].mean():.3f}"
)
print(
    f"Median top-1 similarity: "
    f"{local_eval['top_similarity'].median():.3f}"
)
print(
    f"Below threshold (0.65):  "
    f"{(local_eval['top_similarity'] < RETRIEVAL_SIMILARITY_THRESHOLD).sum()}"
)

# ---------------------------------------------------------
# Save deterministic evaluation
# ---------------------------------------------------------

local_eval_path = evaluation_dir / "apple_support_local_evaluation.csv"

local_eval.to_csv(
    local_eval_path,
    index=False
)

print("\nSaved local evaluation to:")
print(local_eval_path)

Completed: 200

INTENT CLASSIFICATION
---------------------
Accuracy:    1.000
Macro F1:    1.000
Weighted F1: 1.000

Classification report:
                        precision    recall  f1-score   support

account_authentication       1.00      1.00      1.00        18
        apple_services       1.00      1.00      1.00        25
        apps_app_store       1.00      1.00      1.00        20
                 audio       1.00      1.00      1.00        12
      battery_charging       1.00      1.00      1.00        19
         camera_photos       1.00      1.00      1.00        11
          connectivity       1.00      1.00      1.00        21
       device_hardware       1.00      1.00      1.00        15
         display_input       1.00      1.00      1.00        35
         icloud_backup       1.00      1.00      1.00         6
            ios_update       1.00      1.00      1.00        13
         other_unclear       1.00      1.00      1.00         5

              accuracy   

In [105]:
# Recompute the 200-example evaluation with out-of-fold predictions.
# This avoids training-set leakage and gives us honest classifier and escalation metrics.

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report

# ---------------------------------------------------------
# Prepare evaluation data
# ---------------------------------------------------------

oof_data = golden_set[
    [
        "customer_tweet_id",
        "customer_message",
        "previous_message",
        "has_context",
        "intent"
    ]
].copy()

oof_data["model_text"] = oof_data.apply(
    build_model_text,
    axis=1
)

X = oof_data["model_text"]
y = oof_data["intent"]

# ---------------------------------------------------------
# 5-fold out-of-fold predictions
# ---------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=2026
)

oof_predictions = np.empty(len(oof_data), dtype=object)
oof_margins = np.zeros(len(oof_data))

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):

    fold_model = clone(word_char_svm)

    fold_model.fit(
        X.iloc[train_idx],
        y.iloc[train_idx]
    )

    X_valid = X.iloc[valid_idx]

    predictions = fold_model.predict(X_valid)
    decision_scores = fold_model.decision_function(X_valid)

    oof_predictions[valid_idx] = predictions

    # Margin = top score - second-best score.
    sorted_scores = np.sort(
        decision_scores,
        axis=1
    )

    margins = (
        sorted_scores[:, -1]
        - sorted_scores[:, -2]
    )

    oof_margins[valid_idx] = margins

    print(f"Fold {fold} complete.")

# ---------------------------------------------------------
# Add OOF predictions
# ---------------------------------------------------------

oof_data["predicted_intent"] = oof_predictions
oof_data["classifier_margin"] = oof_margins

# ---------------------------------------------------------
# Apply escalation policy using OOF predictions
# ---------------------------------------------------------

oof_escalations = []

for _, row in oof_data.iterrows():

    retrieved_cases = retrieve_historical_cases(
        row["customer_message"],
        top_k=5
    )

    top_similarity = retrieved_cases[0]["similarity"]

    escalation = decide_escalation(
        intent=row["predicted_intent"],
        classifier_margin=row["classifier_margin"],
        top_similarity=top_similarity,
        customer_message=row["customer_message"]
    )

    oof_escalations.append({
        "top_similarity": top_similarity,
        "escalate": escalation["escalate"],
        "escalation_reason": escalation["escalation_reason"]
    })

oof_escalation_df = pd.DataFrame(oof_escalations)

oof_data = pd.concat(
    [
        oof_data.reset_index(drop=True),
        oof_escalation_df.reset_index(drop=True)
    ],
    axis=1
)

# ---------------------------------------------------------
# Honest intent metrics
# ---------------------------------------------------------

accuracy = accuracy_score(
    oof_data["intent"],
    oof_data["predicted_intent"]
)

macro_f1 = f1_score(
    oof_data["intent"],
    oof_data["predicted_intent"],
    average="macro"
)

weighted_f1 = f1_score(
    oof_data["intent"],
    oof_data["predicted_intent"],
    average="weighted"
)

print("\nHONEST OOF INTENT EVALUATION")
print("============================")
print(f"Accuracy:    {accuracy:.3f}")
print(f"Macro F1:    {macro_f1:.3f}")
print(f"Weighted F1: {weighted_f1:.3f}")

print("\nClassification report:")
print(
    classification_report(
        oof_data["intent"],
        oof_data["predicted_intent"],
        zero_division=0
    )
)

# ---------------------------------------------------------
# Honest escalation metrics
# ---------------------------------------------------------

auto_handled = oof_data[
    oof_data["escalate"] == False
]

escalated = oof_data[
    oof_data["escalate"] == True
]

auto_handle_rate = len(auto_handled) / len(oof_data)
escalation_rate = len(escalated) / len(oof_data)

auto_handle_accuracy = accuracy_score(
    auto_handled["intent"],
    auto_handled["predicted_intent"]
)

print("\nHONEST ESCALATION EVALUATION")
print("============================")
print(f"Auto-handled:         {len(auto_handled)}/{len(oof_data)}")
print(f"Auto-handle rate:     {auto_handle_rate:.3f}")
print(f"Escalated:            {len(escalated)}/{len(oof_data)}")
print(f"Escalation rate:      {escalation_rate:.3f}")
print(f"Auto-handle accuracy: {auto_handle_accuracy:.3f}")

# ---------------------------------------------------------
# Retrieval metrics
# ---------------------------------------------------------

print("\nRETRIEVAL")
print("=========")

print(
    f"Mean top-1 similarity:   "
    f"{oof_data['top_similarity'].mean():.3f}"
)

print(
    f"Median top-1 similarity: "
    f"{oof_data['top_similarity'].median():.3f}"
)

print(
    f"Below 0.65 threshold:    "
    f"{(oof_data['top_similarity'] < RETRIEVAL_SIMILARITY_THRESHOLD).sum()}"
)

# ---------------------------------------------------------
# Save honest evaluation
# ---------------------------------------------------------

oof_eval_path = (
    evaluation_dir /
    "apple_support_oof_evaluation.csv"
)

oof_data.to_csv(
    oof_eval_path,
    index=False
)

print("\nSaved honest evaluation to:")
print(oof_eval_path)

Fold 1 complete.
Fold 2 complete.
Fold 3 complete.
Fold 4 complete.
Fold 5 complete.

HONEST OOF INTENT EVALUATION
Accuracy:    0.500
Macro F1:    0.437
Weighted F1: 0.477

Classification report:
                        precision    recall  f1-score   support

account_authentication       0.65      0.72      0.68        18
        apple_services       0.32      0.28      0.30        25
        apps_app_store       0.58      0.70      0.64        20
                 audio       0.00      0.00      0.00        12
      battery_charging       0.76      0.84      0.80        19
         camera_photos       0.83      0.91      0.87        11
          connectivity       0.41      0.33      0.37        21
       device_hardware       0.42      0.33      0.37        15
         display_input       0.49      0.69      0.57        35
         icloud_backup       1.00      0.33      0.50         6
            ios_update       0.13      0.15      0.14        13
         other_unclear       0.00  

In [106]:
# Analyze the honest OOF predictions to find the largest confusion pairs and concrete failure examples.
# These results will directly feed the report's top-5 failure-modes section.

from sklearn.metrics import confusion_matrix
import pandas as pd

# ---------------------------------------------------------
# Confusion matrix
# ---------------------------------------------------------

labels = sorted(oof_data["intent"].unique())

cm = confusion_matrix(
    oof_data["intent"],
    oof_data["predicted_intent"],
    labels=labels
)

confusion_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

print("CONFUSION MATRIX")
print("================")
display(confusion_df)

# ---------------------------------------------------------
# Extract incorrect predictions
# ---------------------------------------------------------

errors = oof_data[
    oof_data["intent"] != oof_data["predicted_intent"]
].copy()

print("\nTotal misclassified examples:", len(errors))

# ---------------------------------------------------------
# Find largest confusion pairs
# ---------------------------------------------------------

pair_counts = (
    errors
    .groupby(
        ["intent", "predicted_intent"]
    )
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print("\nTOP CONFUSION PAIRS")
print("===================")

display(
    pair_counts.head(15)
)

# ---------------------------------------------------------
# Show real examples for the top confusion pairs
# ---------------------------------------------------------

print("\nREAL EXAMPLES FROM TOP CONFUSIONS")
print("=================================")

top_pairs = pair_counts.head(5)

for _, pair in top_pairs.iterrows():

    true_intent = pair["intent"]
    predicted_intent = pair["predicted_intent"]

    print(
        f"\nTRUE: {true_intent} "
        f"-> PREDICTED: {predicted_intent} "
        f"({pair['count']} examples)"
    )

    examples = errors[
        (errors["intent"] == true_intent) &
        (errors["predicted_intent"] == predicted_intent)
    ][
        [
            "customer_tweet_id",
            "customer_message",
            "previous_message",
            "classifier_margin"
        ]
    ].head(3)

    display(examples)

# ---------------------------------------------------------
# Lowest-confidence errors
# ---------------------------------------------------------

print("\nLOWEST-CONFIDENCE MISCLASSIFICATIONS")
print("====================================")

low_confidence_errors = (
    errors
    .sort_values("classifier_margin")
    [
        [
            "customer_tweet_id",
            "customer_message",
            "intent",
            "predicted_intent",
            "classifier_margin"
        ]
    ]
    .head(15)
)

display(low_confidence_errors)

# ---------------------------------------------------------
# Save error analysis
# ---------------------------------------------------------

error_path = (
    evaluation_dir /
    "apple_support_error_analysis.csv"
)

errors.to_csv(
    error_path,
    index=False
)

print("\nSaved errors to:")
print(error_path)

CONFUSION MATRIX


,account_authentication,apple_services,apps_app_store,audio,battery_charging,camera_photos,connectivity,device_hardware,display_input,icloud_backup,ios_update,other_unclear
account_authentication,13,2,1,1,0,0,0,0,0,0,0,1
apple_services,1,7,4,1,0,1,4,1,3,0,3,0
apps_app_store,0,3,14,0,1,0,0,1,1,0,0,0
audio,0,2,0,0,1,0,1,2,3,0,2,1
battery_charging,0,0,1,0,16,0,0,1,1,0,0,0
camera_photos,0,1,0,0,0,10,0,0,0,0,0,0
connectivity,0,2,2,0,1,0,7,0,6,0,2,1
device_hardware,2,2,0,0,1,1,1,5,2,0,1,0
display_input,0,3,0,1,1,0,1,0,24,0,5,0
icloud_backup,2,0,1,0,0,0,0,1,0,2,0,0



Total misclassified examples: 100

TOP CONFUSION PAIRS


,intent,predicted_intent,count
30,connectivity,display_input,6
51,ios_update,display_input,6
44,display_input,ios_update,5
8,apple_services,connectivity,4
5,apple_services,apps_app_store,4
11,apple_services,ios_update,3
10,apple_services,display_input,3
49,ios_update,connectivity,3
12,apps_app_store,apple_services,3
54,other_unclear,display_input,3



REAL EXAMPLES FROM TOP CONFUSIONS

TRUE: connectivity -> PREDICTED: display_input (6 examples)


...


TRUE: apple_services -> PREDICTED: apps_app_store (4 examples)


,customer_tweet_id,customer_message,previous_message,classifier_margin
2,1099710.0,@379507 @AppleSupport Having the same issue.,"@AppleSupport iOS 11.0.3. 2 different iPhone 7s in my family, multiple iPads too. Enter address in Maps &amp; driving directions fail to load—endless spinner.",0.079056
52,1096904.0,Yo @AppleSupport how do I leave group chat on the latest iOS for iPhone. Please help. Please.,NaN,0.058193
142,2503732.0,@AppleSupport I went on iTunes and re downloaded it and it still isn’t in my library. I turned my phone off for 1 hour then back on again and still nothing.,@711875 We want to help. Can you tell us which steps you've tried so far? This way we won't have you do any steps over again.,0.035929



LOWEST-CONFIDENCE MISCLASSIFICATIONS


,customer_tweet_id,customer_message,intent,predicted_intent,classifier_margin
149,1866369.0,@AppleSupport @115948 Here are my Apple Music settings: https://t.co/QxyWJRJN8S,apple_services,ios_update,0.000479
51,2913352.0,@AppleSupport my iMessage has not worked in over a month at this point. I’ve spent considerable amount of time with your customer support to get this solved but there has been no solution yet. Ple...,apple_services,ios_update,0.001496
114,511125.0,@AppleSupport help I got a green line on the left side of my iPhone X screen and restarting it doesn’t make it go away,display_input,apple_services,0.003941
100,1819155.0,"@115858 the new update is ass. GPS doesn’t work, music controls don’t work, Apple Watch won’t connect, speaker phone doesn’t work.",device_hardware,apple_services,0.008818
130,1890537.0,"@115858 please say your developers are hard at work on iOS 11.0.4 so that my headphones, charger and volume buttons work again, ughhhh 🙄🙄🙄🙄🙄",audio,apple_services,0.009262
59,2799435.0,@115858 @6990 why isn’t FaceTime working right now? I need to see @107462 face 😭😭😭,apple_services,connectivity,0.009492
7,2119414.0,"So wtf y’all gon do about this I situation, @115858? Cuz I’m sick of sick this damn A n shit. 😒",display_input,apple_services,0.009502
26,1605041.0,"Hi @AppleSupport ,Apple Watch 3 has bug, restarts when you ask Siri on weather.please address this",device_hardware,apple_services,0.010513
164,1545207.0,@115858 I want my fucking picture back pull it up on iCloud I don't fuck know figure it the fuck out now cause I'm fucking PISSED!,icloud_backup,account_authentication,0.011478
106,1597353.0,Can someone tell me why my tweet looks like this??? @115858 b/c I️ just updated my phone twice .. #replytweet some1 https://t.co/qZ4cyQwvQG,display_input,audio,0.012553



Saved errors to:
c:\Projects\hiver-support-agent\data\evaluation\apple_support_error_analysis.csv


In [107]:
# Analyze whether the escalation policy successfully catches low-confidence and ambiguous cases.
# This tells us whether escalation is actually reducing the risk of incorrect automated handling.

# ---------------------------------------------------------
# Basic escalation/error breakdown
# ---------------------------------------------------------

oof_data["correct"] = (
    oof_data["intent"] ==
    oof_data["predicted_intent"]
)

oof_data["auto_handled"] = ~oof_data["escalate"]

auto_correct = oof_data[
    oof_data["auto_handled"] &
    oof_data["correct"]
]

auto_incorrect = oof_data[
    oof_data["auto_handled"] &
    ~oof_data["correct"]
]

escalated_correct = oof_data[
    oof_data["escalate"] &
    oof_data["correct"]
]

escalated_incorrect = oof_data[
    oof_data["escalate"] &
    ~oof_data["correct"]
]

total_errors = (~oof_data["correct"]).sum()

# ---------------------------------------------------------
# Print breakdown
# ---------------------------------------------------------

print("ESCALATION ERROR ANALYSIS")
print("==========================")

print(f"Total examples:          {len(oof_data)}")
print(f"Total classifier errors: {total_errors}")

print("\nAuto-handled:")
print(f"  Correct:   {len(auto_correct)}")
print(f"  Incorrect: {len(auto_incorrect)}")

print("\nEscalated:")
print(f"  Correct:   {len(escalated_correct)}")
print(f"  Incorrect: {len(escalated_incorrect)}")

print("\nRates:")

print(
    f"Auto-handled error rate: "
    f"{len(auto_incorrect) / len(auto_handled):.3f}"
)

print(
    f"Escalated error rate:    "
    f"{len(escalated_incorrect) / len(escalated):.3f}"
)

if total_errors > 0:
    errors_caught = len(escalated_incorrect)

    print(
        f"Classifier errors caught by escalation: "
        f"{errors_caught}/{total_errors} "
        f"({errors_caught / total_errors:.3f})"
    )

# ---------------------------------------------------------
# Compare confidence distributions
# ---------------------------------------------------------

print("\nCLASSIFIER MARGIN")
print("=================")

print(
    "Correct predictions:"
)
print(
    oof_data.loc[
        oof_data["correct"],
        "classifier_margin"
    ].describe()
)

print(
    "\nIncorrect predictions:"
)
print(
    oof_data.loc[
        ~oof_data["correct"],
        "classifier_margin"
    ].describe()
)

# ---------------------------------------------------------
# Incorrect auto-handled examples
# ---------------------------------------------------------

print("\nINCORRECTLY AUTO-HANDLED EXAMPLES")
print("==================================")

incorrect_auto = (
    auto_incorrect
    .sort_values("classifier_margin")
    [
        [
            "customer_tweet_id",
            "customer_message",
            "intent",
            "predicted_intent",
            "classifier_margin",
            "top_similarity"
        ]
    ]
)

display(incorrect_auto)

# ---------------------------------------------------------
# Escalated examples
# ---------------------------------------------------------

print("\nESCALATED EXAMPLES")
print("==================")

escalated_examples = (
    oof_data[
        oof_data["escalate"]
    ]
    .sort_values("classifier_margin")
    [
        [
            "customer_tweet_id",
            "customer_message",
            "intent",
            "predicted_intent",
            "classifier_margin",
            "top_similarity",
            "escalation_reason"
        ]
    ]
)

display(escalated_examples.head(20))

ESCALATION ERROR ANALYSIS
Total examples:          200
Total classifier errors: 100

Auto-handled:
  Correct:   43
  Incorrect: 8

Escalated:
  Correct:   57
  Incorrect: 92

Rates:
Auto-handled error rate: 0.157
Escalated error rate:    0.617
Classifier errors caught by escalation: 92/100 (0.920)

CLASSIFIER MARGIN
Correct predictions:
count    100.000000
mean       0.430920
std        0.331000
min        0.009880
25%        0.155130
50%        0.363386
75%        0.635232
max        1.418725
Name: classifier_margin, dtype: float64

Incorrect predictions:
count    100.000000
mean       0.142079
std        0.136367
min        0.000479
25%        0.039006
50%        0.094256
75%        0.228803
max        0.575928
Name: classifier_margin, dtype: float64

INCORRECTLY AUTO-HANDLED EXAMPLES


,customer_tweet_id,customer_message,intent,predicted_intent,classifier_margin,top_similarity
182,898222.0,@AppleSupport after installing 11.0.3 my touch screen is working really bad as well as Bluetooth,display_input,connectivity,0.306109,0.832736
187,1642388.0,@AppleSupport It connects successfully but all apps are not functioning including safari.,connectivity,apps_app_store,0.337332,0.804784
123,2279207.0,@115858 hey sooo um....y’all going to fix this update so i️ can charge my phone? 🧐,battery_charging,display_input,0.367894,0.862713
159,1901170.0,"@AppleSupport #iOS11 is riddled w/issues, speakerphone rarely works, camera frozen 30% of the time, y did u issue v11.03 w/ so many bugs!?!",device_hardware,camera_photos,0.438899,0.819597
139,851749.0,@AppleSupport why does my watch do this with push notifications https://t.co/rPcKlWObEZ,apple_services,connectivity,0.509094,0.799085
20,146553.0,@115858 @4792 CarPlay and iPhoneX and the 2017 M240i don‘t work together. When is a fix for that coming?,connectivity,display_input,0.536629,0.671833
24,259974.0,@AppleSupport what does this mean? Why can't I sync my phone to my Mac? https://t.co/MvJ1UXAuee,connectivity,apple_services,0.541072,0.835784
77,2557443.0,"@AppleSupport I updated my iPhone 6S to iOS 11.1.1 and is totally malfunctioning. Apps are crashing every time, phone calls are giving errors and battery runs out faster than normal. Totally the o...",device_hardware,battery_charging,0.575928,0.845203



ESCALATED EXAMPLES


,customer_tweet_id,customer_message,intent,predicted_intent,classifier_margin,top_similarity,escalation_reason
149,1866369.0,@AppleSupport @115948 Here are my Apple Music settings: https://t.co/QxyWJRJN8S,apple_services,ios_update,0.000479,0.828947,The intent classifier is not sufficiently confident.
51,2913352.0,@AppleSupport my iMessage has not worked in over a month at this point. I’ve spent considerable amount of time with your customer support to get this solved but there has been no solution yet. Ple...,apple_services,ios_update,0.001496,0.914628,The intent classifier is not sufficiently confident.
114,511125.0,@AppleSupport help I got a green line on the left side of my iPhone X screen and restarting it doesn’t make it go away,display_input,apple_services,0.003941,0.920467,The intent classifier is not sufficiently confident.
100,1819155.0,"@115858 the new update is ass. GPS doesn’t work, music controls don’t work, Apple Watch won’t connect, speaker phone doesn’t work.",device_hardware,apple_services,0.008818,0.755759,The intent classifier is not sufficiently confident.
130,1890537.0,"@115858 please say your developers are hard at work on iOS 11.0.4 so that my headphones, charger and volume buttons work again, ughhhh 🙄🙄🙄🙄🙄",audio,apple_services,0.009262,0.807324,The intent classifier is not sufficiently confident.
59,2799435.0,@115858 @6990 why isn’t FaceTime working right now? I need to see @107462 face 😭😭😭,apple_services,connectivity,0.009492,0.897843,The intent classifier is not sufficiently confident.
7,2119414.0,"So wtf y’all gon do about this I situation, @115858? Cuz I’m sick of sick this damn A n shit. 😒",display_input,apple_services,0.009502,0.810382,The intent classifier is not sufficiently confident.
160,2467693.0,"@AppleSupport Not sure, it def happens when not plugged in. Never replaced screen. Seems it's a common issue/bug w the update",display_input,display_input,0.009880,0.775150,The intent classifier is not sufficiently confident.
30,2422571.0,Clearly need to stop using a first-person pronoun until @115858 can get their shit together.,display_input,display_input,0.010089,0.864203,The intent classifier is not sufficiently confident.
26,1605041.0,"Hi @AppleSupport ,Apple Watch 3 has bug, restarts when you ask Siri on weather.please address this",device_hardware,apple_services,0.010513,0.883171,The intent classifier is not sufficiently confident.


In [108]:
# Build a fixed 30-example LLM evaluation sample.
# We include the 9 already generated cases and select 21 additional diverse cases
# so the eventual LLM judge and human evaluation cover both successes and failures.

import pandas as pd
import numpy as np

# ---------------------------------------------------------
# Load the 9 completed Gemini evaluations
# ---------------------------------------------------------

agent_results_path = (
    evaluation_dir /
    "apple_support_agent_results.csv"
)

agent_results = pd.read_csv(agent_results_path)

completed_ids = set(
    agent_results["customer_tweet_id"]
    .astype(str)
)

print("Existing Gemini results:", len(agent_results))

# ---------------------------------------------------------
# Start with existing generated examples
# ---------------------------------------------------------

selected_ids = set(completed_ids)

# Work from the honest OOF evaluation
candidate_pool = oof_data.copy()

candidate_pool["id_str"] = (
    candidate_pool["customer_tweet_id"]
    .astype(str)
)

candidate_pool = candidate_pool[
    ~candidate_pool["id_str"].isin(selected_ids)
].copy()

# ---------------------------------------------------------
# Select additional examples from different categories
# ---------------------------------------------------------

additional = []

# 1. Low-confidence cases
low_conf = (
    candidate_pool
    .sort_values("classifier_margin")
    .head(7)
)

additional.append(low_conf)

# 2. Incorrect predictions
incorrect = candidate_pool[
    candidate_pool["intent"] != candidate_pool["predicted_intent"]
].sort_values("classifier_margin")

incorrect = incorrect[
    ~incorrect["customer_tweet_id"].isin(
        pd.concat(additional)["customer_tweet_id"]
    )
].head(7)

additional.append(incorrect)

# 3. Low retrieval similarity
low_retrieval = (
    candidate_pool
    .sort_values("top_similarity")
)

low_retrieval = low_retrieval[
    ~low_retrieval["customer_tweet_id"].isin(
        pd.concat(additional)["customer_tweet_id"]
    )
].head(4)

additional.append(low_retrieval)

# 4. High-confidence auto-handled cases
high_conf = (
    candidate_pool[
        ~candidate_pool["escalate"]
    ]
    .sort_values("classifier_margin", ascending=False)
)

high_conf = high_conf[
    ~high_conf["customer_tweet_id"].isin(
        pd.concat(additional)["customer_tweet_id"]
    )
].head(3)

additional.append(high_conf)

# ---------------------------------------------------------
# Combine and limit to 30 total examples
# ---------------------------------------------------------

additional_df = pd.concat(
    additional,
    ignore_index=True
)

additional_df = additional_df.drop_duplicates(
    subset=["customer_tweet_id"]
)

needed = max(0, 30 - len(agent_results))

additional_df = additional_df.head(needed)

selected_oof = candidate_pool[
    candidate_pool["customer_tweet_id"].isin(
        additional_df["customer_tweet_id"]
    )
].copy()

# ---------------------------------------------------------
# Build evaluation sample
# ---------------------------------------------------------

llm_eval_sample = pd.concat(
    [
        agent_results.assign(source="already_generated"),
        selected_oof.assign(source="to_generate")
    ],
    ignore_index=True,
    sort=False
)

llm_eval_sample = llm_eval_sample.drop_duplicates(
    subset=["customer_tweet_id"]
).head(30)

# ---------------------------------------------------------
# Save fixed sample
# ---------------------------------------------------------

llm_sample_path = (
    evaluation_dir /
    "apple_support_llm_evaluation_sample_30.csv"
)

llm_eval_sample.to_csv(
    llm_sample_path,
    index=False
)

print("\nLLM evaluation sample created.")
print("Total examples:", len(llm_eval_sample))
print("Already generated:", len(agent_results))
print(
    "Still need Gemini generation:",
    len(llm_eval_sample) - len(agent_results)
)

print("\nSample intent distribution:")
print(
    llm_eval_sample["true_intent"]
    .value_counts()
)

print("\nSaved to:")
print(llm_sample_path)

Existing Gemini results: 9

LLM evaluation sample created.
Total examples: 30
Already generated: 9
Still need Gemini generation: 21

Sample intent distribution:
true_intent
apps_app_store      3
apple_services      2
ios_update          2
battery_charging    1
display_input       1
Name: count, dtype: int64

Saved to:
c:\Projects\hiver-support-agent\data\evaluation\apple_support_llm_evaluation_sample_30.csv


In [109]:
# Fix the LLM evaluation sample so all 30 examples have consistent evaluation columns.
# This does not call Gemini; it only cleans the sample we already selected.

# ---------------------------------------------------------
# Reload the fixed 30-example sample
# ---------------------------------------------------------

llm_sample_path = (
    evaluation_dir /
    "apple_support_llm_evaluation_sample_30.csv"
)

llm_eval_sample = pd.read_csv(
    llm_sample_path
)

# ---------------------------------------------------------
# Normalize the true intent column
# ---------------------------------------------------------

if "true_intent" not in llm_eval_sample.columns:
    llm_eval_sample["true_intent"] = llm_eval_sample["intent"]
else:
    llm_eval_sample["true_intent"] = (
        llm_eval_sample["true_intent"]
        .fillna(llm_eval_sample["intent"])
    )

# ---------------------------------------------------------
# Show actual distribution
# ---------------------------------------------------------

print("Total examples:", len(llm_eval_sample))

print("\nTrue intent distribution:")
print(
    llm_eval_sample["true_intent"]
    .value_counts()
    .sort_index()
)

print("\nEscalation distribution:")

if "escalate" in llm_eval_sample.columns:
    print(
        llm_eval_sample["escalate"]
        .value_counts(dropna=False)
    )

print("\nExamples by source:")

if "source" in llm_eval_sample.columns:
    print(
        llm_eval_sample["source"]
        .value_counts()
    )

# ---------------------------------------------------------
# Save corrected sample
# ---------------------------------------------------------

llm_eval_sample.to_csv(
    llm_sample_path,
    index=False
)

print("\nCorrected sample saved to:")
print(llm_sample_path)

Total examples: 30

True intent distribution:
true_intent
account_authentication     2
apple_services             5
apps_app_store             4
audio                      1
battery_charging           1
camera_photos              1
device_hardware            3
display_input             10
icloud_backup              1
ios_update                 2
Name: count, dtype: int64

Escalation distribution:
escalate
True     20
False    10
Name: count, dtype: int64

Examples by source:
source
to_generate          21
already_generated     9
Name: count, dtype: int64

Corrected sample saved to:
c:\Projects\hiver-support-agent\data\evaluation\apple_support_llm_evaluation_sample_30.csv


In [110]:
# Prepare the final 30-example LLM evaluation table.
# This combines the fixed sample with any Gemini responses already generated,
# so the same 30 cases will be used for both LLM judging and human review.

import pandas as pd
import json

# ---------------------------------------------------------
# Load fixed 30-example sample
# ---------------------------------------------------------

llm_sample_path = (
    evaluation_dir /
    "apple_support_llm_evaluation_sample_30.csv"
)

llm_sample = pd.read_csv(
    llm_sample_path
)

# ---------------------------------------------------------
# Load existing Gemini agent results
# ---------------------------------------------------------

agent_results_path = (
    evaluation_dir /
    "apple_support_agent_results.csv"
)

agent_results = pd.read_csv(
    agent_results_path
)

# ---------------------------------------------------------
# Keep only fields needed for evaluation
# ---------------------------------------------------------

agent_results_small = agent_results[
    [
        "customer_tweet_id",
        "draft_reply",
        "evidence_used",
        "escalate",
        "escalation_reason"
    ]
].copy()

agent_results_small["customer_tweet_id"] = (
    agent_results_small["customer_tweet_id"]
    .astype(str)
)

llm_sample["customer_tweet_id"] = (
    llm_sample["customer_tweet_id"]
    .astype(str)
)

# ---------------------------------------------------------
# Merge generated responses onto the fixed sample
# ---------------------------------------------------------

llm_judge_data = llm_sample.merge(
    agent_results_small,
    on="customer_tweet_id",
    how="left",
    suffixes=("", "_generated")
)

# ---------------------------------------------------------
# Prefer generated agent values where available
# ---------------------------------------------------------

for column in [
    "draft_reply",
    "evidence_used",
    "escalate",
    "escalation_reason"
]:
    generated_column = f"{column}_generated"

    if generated_column in llm_judge_data.columns:

        if column in llm_judge_data.columns:
            llm_judge_data[column] = (
                llm_judge_data[column]
                .fillna(llm_judge_data[generated_column])
            )
        else:
            llm_judge_data[column] = (
                llm_judge_data[generated_column]
            )

        llm_judge_data.drop(
            columns=[generated_column],
            inplace=True
        )

# ---------------------------------------------------------
# Normalize evaluation fields
# ---------------------------------------------------------

llm_judge_data["true_intent"] = (
    llm_judge_data["true_intent"]
    .fillna(llm_judge_data["intent"])
)

llm_judge_data["has_generated_reply"] = (
    llm_judge_data["draft_reply"]
    .notna()
    &
    llm_judge_data["draft_reply"]
    .astype(str)
    .str.strip()
    .ne("")
)

# ---------------------------------------------------------
# Add stable evaluation ID
# ---------------------------------------------------------

llm_judge_data.insert(
    0,
    "evaluation_id",
    range(1, len(llm_judge_data) + 1)
)

# ---------------------------------------------------------
# Display status
# ---------------------------------------------------------

print("Total evaluation cases:", len(llm_judge_data))

print(
    "Generated responses:",
    llm_judge_data["has_generated_reply"].sum()
)

print(
    "Missing responses:",
    (~llm_judge_data["has_generated_reply"]).sum()
)

print("\nIntent distribution:")
print(
    llm_judge_data["true_intent"]
    .value_counts()
    .sort_index()
)

print("\nEvaluation columns:")
print(
    llm_judge_data.columns.tolist()
)

# ---------------------------------------------------------
# Save final judge dataset
# ---------------------------------------------------------

judge_data_path = (
    evaluation_dir /
    "apple_support_llm_judge_30.csv"
)

llm_judge_data.to_csv(
    judge_data_path,
    index=False
)

print("\nSaved to:")
print(judge_data_path)

Total evaluation cases: 30
Generated responses: 9
Missing responses: 21

Intent distribution:
true_intent
account_authentication     2
apple_services             5
apps_app_store             4
audio                      1
battery_charging           1
camera_photos              1
device_hardware            3
display_input             10
icloud_backup              1
ios_update                 2
Name: count, dtype: int64

Evaluation columns:
['evaluation_id', 'customer_tweet_id', 'customer_message', 'true_intent', 'predicted_intent', 'classifier_margin', 'top_similarity', 'draft_reply', 'evidence_used', 'escalate', 'escalation_reason', 'source', 'previous_message', 'has_context', 'intent', 'model_text', 'correct', 'auto_handled', 'id_str', 'has_generated_reply']

Saved to:
c:\Projects\hiver-support-agent\data\evaluation\apple_support_llm_judge_30.csv


In [111]:
# Test whether the Gemini API quota is available again before running the remaining 21 evaluations.
# We make exactly one request so we don't waste quota if the limit is still active.

test_case = llm_judge_data[
    ~llm_judge_data["has_generated_reply"]
].iloc[0]

print("Testing Gemini on evaluation ID:", test_case["evaluation_id"])

try:
    test_result = generate_support_response(
        {
            "customer_message": test_case["customer_message"],
            "previous_message": test_case["previous_message"],
            "intent": test_case["predicted_intent"],
            "retrieved_cases": retrieve_historical_cases(
                test_case["customer_message"],
                top_k=5
            ),
            "top_similarity": test_case["top_similarity"]
        }
    )

    print("\nGemini quota appears available.")
    print("\nGenerated reply:")
    print(test_result["draft_reply"])

except Exception as e:
    print("\nGemini request failed:")
    print(e)

Testing Gemini on evaluation ID: 10

Gemini request failed:
429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 34.756672858s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'mod

In [112]:
# Create the human-review template for the 30 LLM evaluation examples.
# The same rubric will later be used by the LLM judge, allowing us to measure
# agreement between the automated judge and human evaluation.

import pandas as pd

# ---------------------------------------------------------
# Load the fixed 30-example evaluation set
# ---------------------------------------------------------

judge_data_path = (
    evaluation_dir /
    "apple_support_llm_judge_30.csv"
)

judge_data = pd.read_csv(
    judge_data_path
)

# ---------------------------------------------------------
# Create human evaluation template
# ---------------------------------------------------------

human_review = judge_data[
    [
        "evaluation_id",
        "customer_tweet_id",
        "customer_message",
        "previous_message",
        "true_intent",
        "predicted_intent",
        "classifier_margin",
        "top_similarity",
        "draft_reply",
        "evidence_used",
        "escalate",
        "escalation_reason"
    ]
].copy()

# Human scoring columns.
human_review["relevance_1_5"] = ""
human_review["grounding_1_5"] = ""
human_review["helpfulness_1_5"] = ""
human_review["unsupported_claims_1_5"] = ""
human_review["overall_quality_1_5"] = ""
human_review["human_notes"] = ""

# ---------------------------------------------------------
# Save human annotation sheet
# ---------------------------------------------------------

human_review_path = (
    evaluation_dir /
    "apple_support_human_review_30.csv"
)

human_review.to_csv(
    human_review_path,
    index=False
)

print("Human review template created.")
print("Total examples:", len(human_review))
print(
    "Responses currently available:",
    human_review["draft_reply"].notna().sum()
)
print(
    "Responses still missing:",
    human_review["draft_reply"].isna().sum()
)

print("\nSaved to:")
print(human_review_path)

Human review template created.
Total examples: 30
Responses currently available: 9
Responses still missing: 21

Saved to:
c:\Projects\hiver-support-agent\data\evaluation\apple_support_human_review_30.csv


In [113]:
# LLM-as-judge prompt
# This defines the evaluation rubric that Gemini will use later.
# No API request is made here.

JUDGE_RUBRIC = """
You are evaluating an AI customer-support reply for AppleSupport.

Evaluate ONLY the information provided:
- Customer message
- Previous message, if available
- Predicted intent
- Retrieved historical support cases
- AI-generated draft reply

Do not reward the response for sounding professional alone.
The response must be relevant, historically grounded, useful, and free of unsupported claims.

Score the following:

1. relevance (1-5)
   5 = Directly addresses the customer's actual issue.
   4 = Mostly addresses the issue with minor omissions.
   3 = Partially relevant but misses an important aspect.
   2 = Mostly unrelated or poorly targeted.
   1 = Does not address the customer's issue.

2. grounding (1-5)
   5 = Clearly consistent with the retrieved historical AppleSupport responses.
   4 = Mostly supported by historical evidence.
   3 = Some support exists, but important parts are weakly grounded.
   2 = Mostly relies on information not supported by the evidence.
   1 = Contradicts or ignores the historical evidence.

3. helpfulness (1-5)
   5 = Gives a useful next step or actionable support guidance.
   4 = Useful but could be more specific.
   3 = Somewhat useful but incomplete.
   2 = Provides little practical help.
   1 = Not useful.

4. overall_quality (1-5)
   5 = Strong support response suitable for the situation.
   4 = Good response with minor issues.
   3 = Acceptable but noticeably incomplete or generic.
   2 = Poor response requiring substantial revision.
   1 = Unacceptable.

5. unsupported_claim (0-1)
   1 = Contains at least one factual claim, policy, troubleshooting instruction,
       guarantee, or URL that is not supported by the provided evidence.
   0 = No clearly unsupported claim.

Also provide:
- short_reason: one concise explanation of the scores
- major_issue: the single most important problem, or "none"

Return ONLY valid JSON:

{
    "relevance": 1,
    "grounding": 1,
    "helpfulness": 1,
    "overall_quality": 1,
    "unsupported_claim": 0,
    "short_reason": "...",
    "major_issue": "..."
}
"""


def build_judge_prompt(row):
    """
    Build the evaluation prompt for one generated support response.
    """
    
    retrieved_cases = row.get("retrieved_cases", "")
    
    prompt = f"""
{JUDGE_RUBRIC}

CUSTOMER MESSAGE:
{row["customer_message"]}

PREVIOUS MESSAGE:
{row["previous_message"]}

TRUE INTENT:
{row["true_intent"]}

PREDICTED INTENT:
{row["predicted_intent"]}

RETRIEVED HISTORICAL CASES:
{retrieved_cases}

AI-GENERATED REPLY:
{row["draft_reply"]}
"""
    
    return prompt


print("Judge rubric created.")
print("Prompt builder ready.")

Judge rubric created.
Prompt builder ready.


In [114]:
def format_retrieved_cases(customer_message, top_k=5):
    """
    Retrieve and format historical AppleSupport cases
    for the LLM judge.
    """
    
    cases = retrieve_historical_cases(
        customer_message,
        top_k=top_k
    )
    
    formatted = []
    
    for i, case in enumerate(cases, start=1):
        formatted.append(
            f"""
Case {i}
Similarity: {case["similarity"]:.4f}

Customer:
{case["customer_message"]}

AppleSupport response:
{case["brand_response"]}
"""
        )
    
    return "\n".join(formatted)


print("Historical evidence formatter ready.")

Historical evidence formatter ready.


In [115]:
# Find one example that already has a generated reply.

available_example = llm_judge_data[
    llm_judge_data["has_generated_reply"] == True
].iloc[0]

example_prompt = build_judge_prompt(
    {
        "customer_message": available_example["customer_message"],
        "previous_message": available_example["previous_message"],
        "true_intent": available_example["true_intent"],
        "predicted_intent": available_example["predicted_intent"],
        "retrieved_cases": format_retrieved_cases(
            available_example["customer_message"]
        ),
        "draft_reply": available_example["draft_reply"]
    }
)

print(example_prompt[:6000])



You are evaluating an AI customer-support reply for AppleSupport.

Evaluate ONLY the information provided:
- Customer message
- Previous message, if available
- Predicted intent
- Retrieved historical support cases
- AI-generated draft reply

Do not reward the response for sounding professional alone.
The response must be relevant, historically grounded, useful, and free of unsupported claims.

Score the following:

1. relevance (1-5)
   5 = Directly addresses the customer's actual issue.
   4 = Mostly addresses the issue with minor omissions.
   3 = Partially relevant but misses an important aspect.
   2 = Mostly unrelated or poorly targeted.
   1 = Does not address the customer's issue.

2. grounding (1-5)
   5 = Clearly consistent with the retrieved historical AppleSupport responses.
   4 = Mostly supported by historical evidence.
   3 = Some support exists, but important parts are weakly grounded.
   2 = Mostly relies on information not supported by the evidence.
   1 = Contradic

In [116]:
import json
import time
import pandas as pd

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

judge_path = evaluation_dir / "apple_support_llm_judge_30.csv"
judge_data = pd.read_csv(judge_path)

# Make sure the tracking column exists
if "judge_completed" not in judge_data.columns:
    judge_data["judge_completed"] = False

# ---------------------------------------------------------
# Helper: parse Gemini judge response
# ---------------------------------------------------------

def parse_judge_response(response_text):
    """
    Parse the JSON returned by Gemini.
    """
    
    text = response_text.strip()
    
    # Remove accidental markdown fences if Gemini returns them.
    if text.startswith("```"):
        text = text.replace("```json", "", 1)
        text = text.replace("```", "", 1).strip()
    
    result = json.loads(text)
    
    required_fields = [
        "relevance",
        "grounding",
        "helpfulness",
        "overall_quality",
        "unsupported_claim",
        "short_reason",
        "major_issue"
    ]
    
    missing = [
        field for field in required_fields
        if field not in result
    ]
    
    if missing:
        raise ValueError(
            f"Judge response missing fields: {missing}"
        )
    
    return result


# ---------------------------------------------------------
# Helper: create judge input for one row
# ---------------------------------------------------------

def prepare_judge_input(row):
    
    retrieved_cases = format_retrieved_cases(
        row["customer_message"],
        top_k=5
    )
    
    return {
        "customer_message": row["customer_message"],
        "previous_message": row["previous_message"],
        "true_intent": row["true_intent"],
        "predicted_intent": row["predicted_intent"],
        "retrieved_cases": retrieved_cases,
        "draft_reply": row["draft_reply"]
    }


# ---------------------------------------------------------
# Main judge runner
# ---------------------------------------------------------

def run_llm_judge(
    max_examples=1,
    delay_seconds=2
):
    """
    Run the Gemini judge on unfinished examples.

    Checkpoints after every successful example.

    max_examples controls how many API calls are made.
    """

    global judge_data

    pending = judge_data[
        judge_data["draft_reply"].notna() &
        (~judge_data["judge_completed"].fillna(False))
    ].copy()

    print("Total evaluation examples:", len(judge_data))
    print("Pending judge evaluations:", len(pending))
    print("Maximum calls this run:", max_examples)

    if len(pending) == 0:
        print("\nNo pending judge evaluations.")
        return

    processed = 0

    for _, row in pending.iterrows():

        if processed >= max_examples:
            break

        evaluation_id = row["evaluation_id"]

        print(
            f"\n[{processed + 1}/{max_examples}] "
            f"Evaluation ID: {evaluation_id}"
        )

        try:
            judge_input = prepare_judge_input(row)

            prompt = build_judge_prompt(judge_input)

            response = gemini_client.models.generate_content(
                model="gemini-3.5-flash",
                contents=prompt,
                config={
                    "temperature": 0.0,
                    "response_mime_type": "application/json"
                }
            )

            judge_result = parse_judge_response(
                response.text
            )

            # ---------------------------------------------
            # Save judge result
            # ---------------------------------------------

            for key, value in judge_result.items():
                judge_data.loc[
                    judge_data["evaluation_id"] == evaluation_id,
                    f"judge_{key}"
                ] = value

            judge_data.loc[
                judge_data["evaluation_id"] == evaluation_id,
                "judge_completed"
            ] = True

            # Checkpoint immediately.
            judge_data.to_csv(
                judge_path,
                index=False
            )

            processed += 1

            print("Judge evaluation saved.")
            print(
                "Overall quality:",
                judge_result["overall_quality"]
            )

            if processed < max_examples:
                time.sleep(delay_seconds)

        except Exception as e:

            error_text = str(e)

            print("\nJudge request failed:")
            print(error_text)

            # Important:
            # Stop immediately on quota errors.
            if (
                "429" in error_text
                or "RESOURCE_EXHAUSTED" in error_text
                or "quota" in error_text.lower()
            ):
                print(
                    "\nQuota error detected."
                    "\nStopping immediately."
                    "\nAlready completed results were saved."
                )
                break

            # For other errors, record them and continue.
            judge_data.loc[
                judge_data["evaluation_id"] == evaluation_id,
                "judge_error"
            ] = error_text

            judge_data.to_csv(
                judge_path,
                index=False
            )

    print("\nRun finished.")
    print("Successfully judged:", processed)
    
    print(
        "Remaining pending:",
        (
            judge_data["draft_reply"].notna() &
            (~judge_data["judge_completed"].fillna(False))
        ).sum()
    )

In [117]:
print("Judge runner loaded successfully.")

print("\nTotal examples:", len(judge_data))

print(
    "Generated replies:",
    judge_data["draft_reply"].notna().sum()
)

print(
    "Already judged:",
    judge_data["judge_completed"].sum()
)

print(
    "Pending judge evaluations:",
    (
        judge_data["draft_reply"].notna() &
        (~judge_data["judge_completed"].fillna(False))
    ).sum()
)

Judge runner loaded successfully.

Total examples: 30
Generated replies: 9
Already judged: 0
Pending judge evaluations: 9


In [119]:
# ---------------------------------------------------------
# Check LLM judge evaluation status
# No Gemini API call
# ---------------------------------------------------------

judge_data = pd.read_csv(judge_path)

# Older version of the CSV may not have this column yet.
# Create it if necessary.
if "judge_completed" not in judge_data.columns:
    judge_data["judge_completed"] = False

generated_mask = judge_data["draft_reply"].notna()
judged_mask = judge_data["judge_completed"].fillna(False).astype(bool)

print("Total examples:", len(judge_data))
print("Generated replies:", generated_mask.sum())
print("Already judged:", (generated_mask & judged_mask).sum())
print(
    "Pending judge evaluations:",
    (generated_mask & ~judged_mask).sum()
)
print(
    "Missing generated replies:",
    (~generated_mask).sum()
)

Total examples: 30
Generated replies: 9
Already judged: 0
Pending judge evaluations: 9
Missing generated replies: 21


In [121]:
# ---------------------------------------------------------
# Recreate 5-fold OOF predictions for the 200-example set
# No Gemini API call
# ---------------------------------------------------------

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Use the existing 200-example model data
# ---------------------------------------------------------

X_text = model_data["model_text"]
y = model_data["intent"]

# ---------------------------------------------------------
# Generate out-of-fold decision scores
# ---------------------------------------------------------

oof_scores = cross_val_predict(
    word_char_svm,
    X_text,
    y,
    cv=cv,
    method="decision_function"
)

# ---------------------------------------------------------
# Convert scores to predictions
# ---------------------------------------------------------

classes = np.unique(y)

predicted_indices = np.argmax(
    oof_scores,
    axis=1
)

predicted_intents = classes[predicted_indices]

# ---------------------------------------------------------
# Calculate classifier confidence margin
#
# Margin = highest class score - second highest score
# Larger margin = more confident prediction
# ---------------------------------------------------------

sorted_scores = np.sort(
    oof_scores,
    axis=1
)

top_score = sorted_scores[:, -1]
second_score = sorted_scores[:, -2]

margins = top_score - second_score

# ---------------------------------------------------------
# Build OOF results table
# ---------------------------------------------------------

oof_results = model_data[
    [
        "customer_tweet_id",
        "customer_message",
        "previous_message",
        "has_context",
        "intent"
    ]
].copy()

oof_results = oof_results.rename(
    columns={
        "intent": "true_intent"
    }
)

oof_results["predicted_intent"] = predicted_intents
oof_results["top_score"] = top_score
oof_results["second_score"] = second_score
oof_results["margin"] = margins

oof_results["is_error"] = (
    oof_results["true_intent"] !=
    oof_results["predicted_intent"]
)

# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print("OOF evaluation recreated.")
print()
print("Examples:", len(oof_results))
print("Errors:", oof_results["is_error"].sum())
print(
    "Accuracy:",
    (~oof_results["is_error"]).mean()
)

print("\nClassification report:")
print(
    classification_report(
        oof_results["true_intent"],
        oof_results["predicted_intent"],
        zero_division=0
    )
)

print("\nMargin statistics:")
print(
    oof_results["margin"].describe()
)

OOF evaluation recreated.

Examples: 200
Errors: 100
Accuracy: 0.5

Classification report:
                        precision    recall  f1-score   support

account_authentication       0.65      0.72      0.68        18
        apple_services       0.32      0.28      0.30        25
        apps_app_store       0.58      0.70      0.64        20
                 audio       0.00      0.00      0.00        12
      battery_charging       0.76      0.84      0.80        19
         camera_photos       0.83      0.91      0.87        11
          connectivity       0.41      0.33      0.37        21
       device_hardware       0.42      0.33      0.37        15
         display_input       0.49      0.69      0.57        35
         icloud_backup       1.00      0.33      0.50         6
            ios_update       0.13      0.15      0.14        13
         other_unclear       0.00      0.00      0.00         5

              accuracy                           0.50       200
           

In [122]:
# ---------------------------------------------------------
# Top classifier confusion pairs
# ---------------------------------------------------------

confusions = (
    oof_results[oof_results["is_error"]]
    .groupby(
        ["true_intent", "predicted_intent"]
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Top confusion pairs:\n")

display(
    confusions.head(15)
)

Top confusion pairs:



,true_intent,predicted_intent,count
0,connectivity,display_input,6
1,ios_update,display_input,6
2,display_input,ios_update,5
3,apple_services,connectivity,4
4,apple_services,apps_app_store,4
5,apple_services,ios_update,3
6,apple_services,display_input,3
7,ios_update,connectivity,3
8,apps_app_store,apple_services,3
9,other_unclear,display_input,3


In [123]:
# ---------------------------------------------------------
# Reload Gemini API key
# ---------------------------------------------------------

from dotenv import load_dotenv
from google import genai
import os

load_dotenv(
    env_path,
    override=True
)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY was not found.")

print("Gemini API key loaded.")
print(
    "Key preview:",
    GEMINI_API_KEY[:8] + "..." + GEMINI_API_KEY[-4:]
)

gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("Gemini client recreated.")

Gemini API key loaded.
Key preview: AQ.Ab8RN...DQ4g
Gemini client recreated.


In [124]:
# ---------------------------------------------------------
# Generate missing Gemini responses for the fixed 30-example
# evaluation set.
#
# Checkpoints after EVERY successful API call.
# Stops automatically on 429/quota errors.
# ---------------------------------------------------------

import json
import time
import pandas as pd

judge_path = evaluation_dir / "apple_support_llm_judge_30.csv"

llm_judge_data = pd.read_csv(judge_path)

# ---------------------------------------------------------
# Identify missing responses
# ---------------------------------------------------------

missing_mask = (
    llm_judge_data["draft_reply"].isna()
)

pending_indices = llm_judge_data[
    missing_mask
].index.tolist()

print("Total evaluation examples:", len(llm_judge_data))
print("Existing generated replies:", (~missing_mask).sum())
print("Missing generated replies:", missing_mask.sum())

# ---------------------------------------------------------
# Generate responses
# ---------------------------------------------------------

successful = 0

for position, idx in enumerate(pending_indices, start=1):

    row = llm_judge_data.loc[idx]

    print(
        f"\n[{position}/{len(pending_indices)}] "
        f"Evaluation ID: {row['evaluation_id']}"
    )

    try:

        # Retrieve the same historical evidence used by the agent.
        retrieved_cases = retrieve_historical_cases(
            row["customer_message"],
            top_k=5
        )

        analysis = {
            "customer_message": row["customer_message"],
            "previous_message": row["previous_message"],
            "intent": row["predicted_intent"],
            "retrieved_cases": retrieved_cases,
            "top_similarity": row["top_similarity"]
        }

        # -------------------------------------------------
        # Gemini generation
        # -------------------------------------------------

        result = generate_support_response(
            analysis,
            model="gemini-3.5-flash"
        )

        # -------------------------------------------------
        # Deterministic escalation policy
        # -------------------------------------------------

        escalation = decide_escalation(
            intent=row["predicted_intent"],
            classifier_margin=row["classifier_margin"],
            top_similarity=row["top_similarity"],
            customer_message=row["customer_message"]
        )

        # -------------------------------------------------
        # Save generated response
        # -------------------------------------------------

        llm_judge_data.loc[idx, "draft_reply"] = (
            result["draft_reply"]
        )

        llm_judge_data.loc[idx, "evidence_used"] = json.dumps(
            result.get("evidence_used", [])
        )

        llm_judge_data.loc[idx, "escalate"] = (
            escalation["escalate"]
        )

        llm_judge_data.loc[idx, "escalation_reason"] = (
            escalation["escalation_reason"]
        )

        # Mark generation as completed.
        llm_judge_data.loc[idx, "has_generated_reply"] = True

        # -------------------------------------------------
        # CHECKPOINT IMMEDIATELY
        # -------------------------------------------------

        llm_judge_data.to_csv(
            judge_path,
            index=False
        )

        successful += 1

        print("Saved successfully.")
        print(
            "Intent:",
            row["predicted_intent"]
        )
        print(
            "Reply:",
            result["draft_reply"]
        )

        # Small delay between requests.
        time.sleep(2)

    except Exception as e:

        error_text = str(e)

        print("\nGemini request failed:")
        print(error_text)

        # Stop immediately for quota errors.
        if (
            "429" in error_text
            or "RESOURCE_EXHAUSTED" in error_text
            or "quota" in error_text.lower()
        ):
            print(
                "\nQuota error detected."
                "\nStopping. All successful responses are already saved."
            )
            break

        # Save non-quota error and continue.
        llm_judge_data.loc[
            idx,
            "generation_error"
        ] = error_text

        llm_judge_data.to_csv(
            judge_path,
            index=False
        )

print("\n" + "=" * 60)
print("GENERATION RUN COMPLETE")
print("=" * 60)

print("Successfully generated this run:", successful)

print(
    "Total generated:",
    llm_judge_data["draft_reply"].notna().sum()
)

print(
    "Still missing:",
    llm_judge_data["draft_reply"].isna().sum()
)

print("\nSaved to:")
print(judge_path)

Total evaluation examples: 30
Existing generated replies: 9
Missing generated replies: 21

[1/21] Evaluation ID: 10
Saved successfully.
Intent: apple_services
Reply: We'd like to look into this with you. Please send us a DM with your watchOS version, and we'll get started there.

[2/21] Evaluation ID: 11
Saved successfully.
Intent: display_input
Reply: We want to help get things working smoothly. We recently released iOS 11.1.1, which contains a fix for this autocorrect issue. Please back up your device prior to updating: https://t.co/4f8hwT5to6. If you still need help, DM us.

[3/21] Evaluation ID: 12
Saved successfully.
Intent: ios_update
Reply: We want to help get your iMessage working as soon as possible. Please send us a DM so we can look into this further and help find a solution.

[4/21] Evaluation ID: 13
Saved successfully.
Intent: display_input
Reply: We want to help you get this resolved. We've recently released an update to address these issues. Please back up your device an

In [125]:
# ---------------------------------------------------------
# Continue Gemini generation with automatic rate-limit wait
# ---------------------------------------------------------

import json
import time
import pandas as pd

judge_path = evaluation_dir / "apple_support_llm_judge_30.csv"

llm_judge_data = pd.read_csv(judge_path)

missing_mask = llm_judge_data["draft_reply"].isna()
pending_indices = llm_judge_data[missing_mask].index.tolist()

print("Total examples:", len(llm_judge_data))
print(
    "Already generated:",
    (~missing_mask).sum()
)
print(
    "Still missing:",
    missing_mask.sum()
)

successful = 0

for position, idx in enumerate(pending_indices, start=1):

    row = llm_judge_data.loc[idx]

    print(
        f"\n[{position}/{len(pending_indices)}] "
        f"Evaluation ID: {row['evaluation_id']}"
    )

    # -----------------------------------------------------
    # Retry this example if we hit the per-minute limit
    # -----------------------------------------------------

    while True:

        try:

            retrieved_cases = retrieve_historical_cases(
                row["customer_message"],
                top_k=5
            )

            analysis = {
                "customer_message": row["customer_message"],
                "previous_message": row["previous_message"],
                "intent": row["predicted_intent"],
                "retrieved_cases": retrieved_cases,
                "top_similarity": row["top_similarity"]
            }

            result = generate_support_response(
                analysis,
                model="gemini-3.5-flash"
            )

            # -------------------------------------------------
            # Deterministic escalation
            # -------------------------------------------------

            escalation = decide_escalation(
                intent=row["predicted_intent"],
                classifier_margin=row["classifier_margin"],
                top_similarity=row["top_similarity"],
                customer_message=row["customer_message"]
            )

            # -------------------------------------------------
            # Save response
            # -------------------------------------------------

            llm_judge_data.loc[
                idx,
                "draft_reply"
            ] = result["draft_reply"]

            llm_judge_data.loc[
                idx,
                "evidence_used"
            ] = json.dumps(
                result.get("evidence_used", [])
            )

            llm_judge_data.loc[
                idx,
                "escalate"
            ] = escalation["escalate"]

            llm_judge_data.loc[
                idx,
                "escalation_reason"
            ] = escalation["escalation_reason"]

            llm_judge_data.loc[
                idx,
                "has_generated_reply"
            ] = True

            # -------------------------------------------------
            # CHECKPOINT
            # -------------------------------------------------

            llm_judge_data.to_csv(
                judge_path,
                index=False
            )

            successful += 1

            print("Saved successfully.")
            print(
                "Intent:",
                row["predicted_intent"]
            )
            print(
                "Reply:",
                result["draft_reply"]
            )

            # Wait between requests to stay safely below
            # the 5 requests/minute free-tier limit.
            time.sleep(15)

            break

        except Exception as e:

            error_text = str(e)

            print("\nGemini request failed:")
            print(error_text)

            # -------------------------------------------------
            # Per-minute quota
            # -------------------------------------------------

            if (
                "429" in error_text
                or "RESOURCE_EXHAUSTED" in error_text
            ):

                print(
                    "\nRate limit reached."
                )

                print(
                    "Waiting 35 seconds before retrying..."
                )

                time.sleep(35)

                print(
                    "Retrying same evaluation..."
                )

                continue

            # -------------------------------------------------
            # Other error
            # -------------------------------------------------

            llm_judge_data.loc[
                idx,
                "generation_error"
            ] = error_text

            llm_judge_data.to_csv(
                judge_path,
                index=False
            )

            print(
                "Non-quota error. Skipping this example."
            )

            break


print("\n" + "=" * 60)
print("GENERATION RUN COMPLETE")
print("=" * 60)

print(
    "Generated during this run:",
    successful
)

print(
    "Total generated:",
    llm_judge_data["draft_reply"].notna().sum()
)

print(
    "Still missing:",
    llm_judge_data["draft_reply"].isna().sum()
)

print("\nSaved to:")
print(judge_path)

Total examples: 30
Already generated: 15
Still missing: 15

[1/15] Evaluation ID: 16
Saved successfully.
Intent: display_input
Reply: Here’s what you can do to work around the issue until it’s fixed in a future software update: https://t.co/xXaXeeSRt9

[2/15] Evaluation ID: 17
Saved successfully.
Intent: apple_services
Reply: We'd love to assist you with the issues you're experiencing after the update. Please DM us with more details, including your iOS version, so we can help. https://t.co/GDrqU22YpT

[3/15] Evaluation ID: 18
Saved successfully.
Intent: audio
Reply: We're here to help. Here’s what you can do to work around the issue until it’s fixed in a future software update: https://t.co/xXaXeeSRt9. Please DM us with your iPhone model and iOS version so we can look into this with you.

[4/15] Evaluation ID: 19
Saved successfully.
Intent: apple_services
Reply: We'd like to help with the green line on your iPhone X. Please send us a DM with your location and the iOS version you are cu

In [126]:
# ---------------------------------------------------------
# Final LLM-as-judge rubric
# ---------------------------------------------------------

JUDGE_RUBRIC = """
You are evaluating an AI-generated customer-support reply for AppleSupport.

Your job is to judge the QUALITY OF THE REPLY, not whether the
classification label is correct.

Use only:
- the customer message
- previous message, if available
- predicted intent
- retrieved historical AppleSupport cases
- AI-generated reply

Evaluate the following dimensions.

1. relevance (1-5)
5 = Directly addresses the customer's actual problem.
4 = Mostly addresses the problem with minor omissions.
3 = Partially relevant but misses an important aspect.
2 = Mostly unrelated or poorly targeted.
1 = Does not address the customer's problem.

2. grounding (1-5)
5 = The reply is strongly consistent with the historical AppleSupport
    responses provided as evidence.
4 = Mostly supported by historical evidence.
3 = Some support exists, but important parts are weakly grounded.
2 = Mostly introduces information not supported by the evidence.
1 = Contradicts or ignores the historical evidence.

3. helpfulness (1-5)
5 = Provides a useful and appropriate next step.
4 = Useful but could be more specific.
3 = Somewhat useful but incomplete.
2 = Provides little practical help.
1 = Not useful.

4. overall_quality (1-5)
5 = Strong response suitable for the situation.
4 = Good response with minor issues.
3 = Acceptable but noticeably incomplete.
2 = Poor response requiring substantial revision.
1 = Unacceptable.

5. unsupported_claim (0-1)
1 = The reply contains at least one factual claim, troubleshooting
    instruction, guarantee, policy statement, or URL that is not
    supported by the provided historical evidence or customer context.
0 = No clearly unsupported claim.

Also provide:
- short_reason: concise explanation of the evaluation
- major_issue: the most important problem, or "none"

Return ONLY valid JSON:

{
    "relevance": 1,
    "grounding": 1,
    "helpfulness": 1,
    "overall_quality": 1,
    "unsupported_claim": 0,
    "short_reason": "...",
    "major_issue": "..."
}
"""


def build_judge_prompt(row, retrieved_cases):
    """
    Build the final unbiased LLM-as-judge prompt.
    True intent is deliberately NOT provided to the judge.
    """

    return f"""
{JUDGE_RUBRIC}

CUSTOMER MESSAGE:
{row["customer_message"]}

PREVIOUS MESSAGE:
{row["previous_message"]}

PREDICTED INTENT:
{row["predicted_intent"]}

RETRIEVED HISTORICAL APPLESUPPORT CASES:
{retrieved_cases}

AI-GENERATED REPLY:
{row["draft_reply"]}
"""

In [127]:
# ---------------------------------------------------------
# Run LLM-as-judge on all generated examples
#
# Features:
# - checkpoints after every successful evaluation
# - 15-second spacing for 5 requests/minute limit
# - automatic 35-second wait on 429
# - stops on daily quota
# ---------------------------------------------------------

import json
import time
import pandas as pd

judge_path = evaluation_dir / "apple_support_llm_judge_30.csv"

judge_data = pd.read_csv(judge_path)

# ---------------------------------------------------------
# Ensure tracking columns exist
# ---------------------------------------------------------

if "judge_completed" not in judge_data.columns:
    judge_data["judge_completed"] = False

judge_data["judge_completed"] = (
    judge_data["judge_completed"]
    .fillna(False)
    .astype(bool)
)

# ---------------------------------------------------------
# Find pending evaluations
# ---------------------------------------------------------

pending_mask = (
    judge_data["draft_reply"].notna()
    & ~judge_data["judge_completed"]
)

pending_indices = judge_data[pending_mask].index.tolist()

print("Total examples:", len(judge_data))
print("Generated replies:", judge_data["draft_reply"].notna().sum())
print("Already judged:", judge_data["judge_completed"].sum())
print("Pending judge evaluations:", len(pending_indices))

# ---------------------------------------------------------
# Run judge
# ---------------------------------------------------------

successful = 0

for position, idx in enumerate(pending_indices, start=1):

    row = judge_data.loc[idx]

    print(
        f"\n[{position}/{len(pending_indices)}] "
        f"Evaluation ID: {row['evaluation_id']}"
    )

    while True:

        try:

            # ---------------------------------------------
            # Retrieve the same historical evidence
            # ---------------------------------------------

            retrieved_cases = retrieve_historical_cases(
                row["customer_message"],
                top_k=5
            )

            formatted_cases = format_retrieved_cases(
                row["customer_message"],
                top_k=5
            )

            # ---------------------------------------------
            # Build unbiased judge prompt
            # ---------------------------------------------

            prompt = build_judge_prompt(
                row,
                formatted_cases
            )

            # ---------------------------------------------
            # Gemini judge call
            # ---------------------------------------------

            response = gemini_client.models.generate_content(
                model="gemini-3.5-flash",
                contents=prompt,
                config={
                    "temperature": 0.0,
                    "response_mime_type": "application/json"
                }
            )

            judge_result = parse_judge_response(
                response.text
            )

            # ---------------------------------------------
            # Save result
            # ---------------------------------------------

            for key, value in judge_result.items():

                judge_data.loc[
                    idx,
                    f"judge_{key}"
                ] = value

            judge_data.loc[
                idx,
                "judge_completed"
            ] = True

            # ---------------------------------------------
            # CHECKPOINT
            # ---------------------------------------------

            judge_data.to_csv(
                judge_path,
                index=False
            )

            successful += 1

            print(
                "Overall quality:",
                judge_result["overall_quality"]
            )

            print(
                "Unsupported claim:",
                judge_result["unsupported_claim"]
            )

            # Stay below 5 requests/minute.
            time.sleep(15)

            break

        except Exception as e:

            error_text = str(e)

            print("\nJudge request failed:")
            print(error_text)

            # ---------------------------------------------
            # Rate limit
            # ---------------------------------------------

            if "429" in error_text:

                # Daily quota is different from per-minute quota.
                daily_quota = (
                    "PerDay" in error_text
                    or "per_day" in error_text.lower()
                    or "daily" in error_text.lower()
                )

                if daily_quota:
                    print(
                        "\nDaily Gemini quota reached."
                        "\nStopping safely."
                    )
                    break

                print(
                    "\nPer-minute rate limit reached."
                    "\nWaiting 35 seconds..."
                )

                time.sleep(35)

                print("Retrying...")
                continue

            # ---------------------------------------------
            # Other errors
            # ---------------------------------------------

            judge_data.loc[
                idx,
                "judge_error"
            ] = error_text

            judge_data.to_csv(
                judge_path,
                index=False
            )

            print(
                "Non-quota error. Skipping this example."
            )

            break

    # Stop outer loop if daily quota was hit.
    if (
        "error_text" in locals()
        and "429" in error_text
        and (
            "PerDay" in error_text
            or "per_day" in error_text.lower()
            or "daily" in error_text.lower()
        )
    ):
        break


print("\n" + "=" * 60)
print("LLM JUDGE RUN COMPLETE")
print("=" * 60)

print(
    "Successfully judged this run:",
    successful
)

print(
    "Total judged:",
    judge_data["judge_completed"].sum()
)

print(
    "Remaining:",
    (
        judge_data["draft_reply"].notna()
        & ~judge_data["judge_completed"]
    ).sum()
)

print("\nSaved to:")
print(judge_path)

Total examples: 30
Generated replies: 30
Already judged: 0
Pending judge evaluations: 30

[1/30] Evaluation ID: 1

Judge request failed:
429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 11.981616536s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 

In [128]:
from dotenv import load_dotenv
from google import genai
import os

load_dotenv(env_path, override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("Gemini client updated.")
print(
    "Key:",
    GEMINI_API_KEY[:8] + "..." + GEMINI_API_KEY[-4:]
)

Gemini client updated.
Key: AQ.Ab8RN...kVuw


In [130]:
# ---------------------------------------------------------
# Prepare judge dataset for the new Gemini key
# No Gemini API call yet
# ---------------------------------------------------------

import pandas as pd
import json

judge_path = evaluation_dir / "apple_support_llm_judge_30.csv"

judge_data = pd.read_csv(judge_path)

# Create tracking column if it doesn't exist.
if "judge_completed" not in judge_data.columns:
    judge_data["judge_completed"] = False

judge_data["judge_completed"] = (
    judge_data["judge_completed"]
    .fillna(False)
    .astype(bool)
)

print("Judge dataset prepared.")
print("Total examples:", len(judge_data))
print(
    "Generated replies:",
    judge_data["draft_reply"].notna().sum()
)
print(
    "Already judged:",
    judge_data["judge_completed"].sum()
)

Judge dataset prepared.
Total examples: 30
Generated replies: 30
Already judged: 0


In [131]:
# ---------------------------------------------------------
# Test ONE Gemini judge request
# ---------------------------------------------------------

pending = judge_data[
    ~judge_data["judge_completed"]
]

if len(pending) == 0:
    print("All examples have already been judged.")
else:

    test_row = pending.iloc[0]

    print(
        "Testing evaluation ID:",
        test_row["evaluation_id"]
    )

    formatted_cases = format_retrieved_cases(
        test_row["customer_message"],
        top_k=5
    )

    prompt = build_judge_prompt(
        test_row,
        formatted_cases
    )

    try:

        response = gemini_client.models.generate_content(
            model="gemini-3.5-flash",
            contents=prompt,
            config={
                "temperature": 0.0,
                "response_mime_type": "application/json"
            }
        )

        result = parse_judge_response(
            response.text
        )

        print("\nSUCCESS — Gemini judge responded.")
        print(
            json.dumps(
                result,
                indent=2
            )
        )

    except Exception as e:

        print("\nGemini request failed:")
        print(e)

Testing evaluation ID: 1

SUCCESS — Gemini judge responded.
{
  "relevance": 5,
  "grounding": 5,
  "helpfulness": 4,
  "overall_quality": 4,
  "unsupported_claim": 0,
  "short_reason": "The reply is highly relevant to the customer's query about Siri and third-party apps, and it aligns perfectly with the historical support cases which consistently direct customers to DM for assistance.",
  "major_issue": "none"
}


In [132]:
# ---------------------------------------------------------
# Complete LLM-as-Judge evaluation
# ---------------------------------------------------------

import json
import time
import pandas as pd

judge_path = evaluation_dir / "apple_support_llm_judge_30.csv"

judge_data = pd.read_csv(judge_path)

# ---------------------------------------------------------
# Prepare tracking columns
# ---------------------------------------------------------

if "judge_completed" not in judge_data.columns:
    judge_data["judge_completed"] = False

judge_data["judge_completed"] = (
    judge_data["judge_completed"]
    .fillna(False)
    .astype(bool)
)

# ---------------------------------------------------------
# Save the successful test result for Evaluation ID 1
# ---------------------------------------------------------

test_id = test_row["evaluation_id"]

for key, value in result.items():
    judge_data.loc[
        judge_data["evaluation_id"] == test_id,
        f"judge_{key}"
    ] = value

judge_data.loc[
    judge_data["evaluation_id"] == test_id,
    "judge_completed"
] = True

judge_data.to_csv(
    judge_path,
    index=False
)

print(
    f"Saved successful test judgment for Evaluation ID {test_id}."
)

# ---------------------------------------------------------
# Find remaining examples
# ---------------------------------------------------------

pending_mask = (
    judge_data["draft_reply"].notna()
    & ~judge_data["judge_completed"]
)

pending_indices = judge_data[pending_mask].index.tolist()

print(
    "\nRemaining judge evaluations:",
    len(pending_indices)
)

# ---------------------------------------------------------
# Run remaining evaluations
# ---------------------------------------------------------

successful = 0

for position, idx in enumerate(
    pending_indices,
    start=1
):

    row = judge_data.loc[idx]

    print(
        f"\n[{position}/{len(pending_indices)}] "
        f"Evaluation ID: {row['evaluation_id']}"
    )

    while True:

        try:

            # ---------------------------------------------
            # Retrieve historical evidence
            # ---------------------------------------------

            formatted_cases = format_retrieved_cases(
                row["customer_message"],
                top_k=5
            )

            # ---------------------------------------------
            # Build judge prompt
            # ---------------------------------------------

            prompt = build_judge_prompt(
                row,
                formatted_cases
            )

            # ---------------------------------------------
            # Gemini judge
            # ---------------------------------------------

            response = gemini_client.models.generate_content(
                model="gemini-3.5-flash",
                contents=prompt,
                config={
                    "temperature": 0.0,
                    "response_mime_type": "application/json"
                }
            )

            judge_result = parse_judge_response(
                response.text
            )

            # ---------------------------------------------
            # Store result
            # ---------------------------------------------

            for key, value in judge_result.items():

                judge_data.loc[
                    idx,
                    f"judge_{key}"
                ] = value

            judge_data.loc[
                idx,
                "judge_completed"
            ] = True

            # ---------------------------------------------
            # Checkpoint
            # ---------------------------------------------

            judge_data.to_csv(
                judge_path,
                index=False
            )

            successful += 1

            print(
                "Overall quality:",
                judge_result["overall_quality"]
            )

            print(
                "Unsupported claim:",
                judge_result["unsupported_claim"]
            )

            # Stay safely below 5 requests/minute.
            time.sleep(15)

            break

        except Exception as e:

            error_text = str(e)

            print("\nJudge request failed:")
            print(error_text)

            # ---------------------------------------------
            # 429 rate limit
            # ---------------------------------------------

            if "429" in error_text:

                # Daily quota
                if (
                    "PerDay" in error_text
                    or "per_day" in error_text.lower()
                    or "perday" in error_text.lower()
                    or "GenerateRequestsPerDay" in error_text
                ):

                    print(
                        "\nDaily quota reached."
                        "\nStopping safely."
                    )

                    break

                # Per-minute quota
                print(
                    "\nPer-minute quota reached."
                    "\nWaiting 35 seconds..."
                )

                time.sleep(35)

                print(
                    "Retrying same evaluation..."
                )

                continue

            # ---------------------------------------------
            # Other error
            # ---------------------------------------------

            judge_data.loc[
                idx,
                "judge_error"
            ] = error_text

            judge_data.to_csv(
                judge_path,
                index=False
            )

            print(
                "Non-quota error. Skipping."
            )

            break

    # Stop if daily quota was reached.
    if (
        "error_text" in locals()
        and "429" in error_text
        and (
            "PerDay" in error_text
            or "per_day" in error_text.lower()
            or "perday" in error_text.lower()
            or "GenerateRequestsPerDay" in error_text
        )
    ):
        break


# ---------------------------------------------------------
# Final status
# ---------------------------------------------------------

judge_data = pd.read_csv(judge_path)

judge_completed = (
    judge_data["judge_completed"]
    .fillna(False)
    .astype(bool)
)

print("\n" + "=" * 60)
print("LLM JUDGE COMPLETE")
print("=" * 60)

print(
    "Total examples:",
    len(judge_data)
)

print(
    "Generated replies:",
    judge_data["draft_reply"].notna().sum()
)

print(
    "Judged:",
    judge_completed.sum()
)

print(
    "Remaining:",
    (
        judge_data["draft_reply"].notna()
        & ~judge_completed
    ).sum()
)

print("\nSaved to:")
print(judge_path)

Saved successful test judgment for Evaluation ID 1.

Remaining judge evaluations: 29

[1/29] Evaluation ID: 2
Overall quality: 5
Unsupported claim: 0

LLM JUDGE COMPLETE
Total examples: 30
Generated replies: 30
Judged: 2
Remaining: 28

Saved to:
c:\Projects\hiver-support-agent\data\evaluation\apple_support_llm_judge_30.csv


In [133]:
# ---------------------------------------------------------
# Continue LLM-as-Judge evaluation
# Fixed quota handling
# ---------------------------------------------------------

import json
import time
import pandas as pd

judge_path = evaluation_dir / "apple_support_llm_judge_30.csv"

judge_data = pd.read_csv(judge_path)

# ---------------------------------------------------------
# Prepare tracking column
# ---------------------------------------------------------

if "judge_completed" not in judge_data.columns:
    judge_data["judge_completed"] = False

judge_data["judge_completed"] = (
    judge_data["judge_completed"]
    .fillna(False)
    .astype(bool)
)

# ---------------------------------------------------------
# Find pending examples
# ---------------------------------------------------------

pending_mask = (
    judge_data["draft_reply"].notna()
    & ~judge_data["judge_completed"]
)

pending_indices = judge_data[pending_mask].index.tolist()

print("Total examples:", len(judge_data))
print(
    "Already judged:",
    judge_data["judge_completed"].sum()
)
print(
    "Pending:",
    len(pending_indices)
)

# ---------------------------------------------------------
# We already used 2 requests on this new project today.
# Maximum additional calls before the 20/day limit:
# 20 - 2 = 18
# ---------------------------------------------------------

MAX_NEW_CALLS = 18

print(
    "Maximum calls in this run:",
    min(MAX_NEW_CALLS, len(pending_indices))
)

successful = 0
daily_quota_hit = False

# ---------------------------------------------------------
# Process pending examples
# ---------------------------------------------------------

for position, idx in enumerate(
    pending_indices[:MAX_NEW_CALLS],
    start=1
):

    row = judge_data.loc[idx]

    print(
        f"\n[{position}/{min(MAX_NEW_CALLS, len(pending_indices))}] "
        f"Evaluation ID: {row['evaluation_id']}"
    )

    # ---------------------------------------------
    # Retry only for temporary rate limits
    # ---------------------------------------------

    while True:

        try:

            formatted_cases = format_retrieved_cases(
                row["customer_message"],
                top_k=5
            )

            prompt = build_judge_prompt(
                row,
                formatted_cases
            )

            response = gemini_client.models.generate_content(
                model="gemini-3.5-flash",
                contents=prompt,
                config={
                    "temperature": 0.0,
                    "response_mime_type": "application/json"
                }
            )

            judge_result = parse_judge_response(
                response.text
            )

            # -----------------------------------------
            # Save judge result
            # -----------------------------------------

            for key, value in judge_result.items():

                judge_data.loc[
                    idx,
                    f"judge_{key}"
                ] = value

            judge_data.loc[
                idx,
                "judge_completed"
            ] = True

            # -----------------------------------------
            # Checkpoint immediately
            # -----------------------------------------

            judge_data.to_csv(
                judge_path,
                index=False
            )

            successful += 1

            print(
                "Saved successfully."
            )

            print(
                "Overall quality:",
                judge_result["overall_quality"]
            )

            print(
                "Unsupported claim:",
                judge_result["unsupported_claim"]
            )

            # 4 requests/minute maximum.
            time.sleep(15)

            break

        except Exception as e:

            error_text = str(e)

            print(
                "\nJudge request failed:"
            )
            print(error_text)

            # -----------------------------------------
            # Daily quota
            # -----------------------------------------

            if (
                "GenerateRequestsPerDay" in error_text
                or "PerDayPerProject" in error_text
                or "daily" in error_text.lower()
            ):

                print(
                    "\nDaily quota reached."
                )

                daily_quota_hit = True
                break

            # -----------------------------------------
            # Per-minute quota
            # -----------------------------------------

            if "429" in error_text:

                print(
                    "\nPer-minute rate limit."
                    "\nWaiting 35 seconds..."
                )

                time.sleep(35)

                print(
                    "Retrying same evaluation..."
                )

                continue

            # -----------------------------------------
            # Other error
            # -----------------------------------------

            judge_data.loc[
                idx,
                "judge_error"
            ] = error_text

            judge_data.to_csv(
                judge_path,
                index=False
            )

            print(
                "Non-quota error. Skipping."
            )

            break

    # Stop only if THIS run actually hit daily quota.
    if daily_quota_hit:
        break


# ---------------------------------------------------------
# Final status
# ---------------------------------------------------------

judge_data = pd.read_csv(judge_path)

judge_completed = (
    judge_data["judge_completed"]
    .fillna(False)
    .astype(bool)
)

print("\n" + "=" * 60)
print("LLM JUDGE RUN COMPLETE")
print("=" * 60)

print(
    "New judgments this run:",
    successful
)

print(
    "Total judged:",
    judge_completed.sum()
)

print(
    "Remaining:",
    (
        judge_data["draft_reply"].notna()
        & ~judge_completed
    ).sum()
)

print("\nSaved to:")
print(judge_path)

Total examples: 30
Already judged: 2
Pending: 28
Maximum calls in this run: 18

[1/18] Evaluation ID: 3
Saved successfully.
Overall quality: 5
Unsupported claim: 0

[2/18] Evaluation ID: 4
Saved successfully.
Overall quality: 5
Unsupported claim: 0

[3/18] Evaluation ID: 5
Saved successfully.
Overall quality: 5
Unsupported claim: 0

[4/18] Evaluation ID: 6
Saved successfully.
Overall quality: 5
Unsupported claim: 0

[5/18] Evaluation ID: 7
Saved successfully.
Overall quality: 5
Unsupported claim: 0

[6/18] Evaluation ID: 8
Saved successfully.
Overall quality: 5
Unsupported claim: 0

[7/18] Evaluation ID: 9
Saved successfully.
Overall quality: 4
Unsupported claim: 0

[8/18] Evaluation ID: 10
Saved successfully.
Overall quality: 5
Unsupported claim: 0

[9/18] Evaluation ID: 11
Saved successfully.
Overall quality: 5
Unsupported claim: 0

[10/18] Evaluation ID: 12
Saved successfully.
Overall quality: 4
Unsupported claim: 0

[11/18] Evaluation ID: 13
Saved successfully.
Overall quality: 5
U

In [134]:
from dotenv import load_dotenv
from google import genai
import os

load_dotenv(env_path, override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("Last Gemini API key loaded.")

Last Gemini API key loaded.


In [135]:
# ---------------------------------------------------------
# Test the last API key with ONE judge request
# ---------------------------------------------------------

judge_path = evaluation_dir / "apple_support_llm_judge_30.csv"

judge_data = pd.read_csv(judge_path)

if "judge_completed" not in judge_data.columns:
    judge_data["judge_completed"] = False

judge_data["judge_completed"] = (
    judge_data["judge_completed"]
    .fillna(False)
    .astype(bool)
)

pending = judge_data[
    judge_data["draft_reply"].notna()
    & ~judge_data["judge_completed"]
]

print("Remaining:", len(pending))

test_row = pending.iloc[0]

formatted_cases = format_retrieved_cases(
    test_row["customer_message"],
    top_k=5
)

prompt = build_judge_prompt(
    test_row,
    formatted_cases
)

try:
    response = gemini_client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt,
        config={
            "temperature": 0.0,
            "response_mime_type": "application/json"
        }
    )

    test_judge_result = parse_judge_response(
        response.text
    )

    print("\nSUCCESS")
    print(json.dumps(test_judge_result, indent=2))

except Exception as e:
    print("\nFAILED")
    print(e)

Remaining: 10

SUCCESS
{
  "relevance": 5,
  "grounding": 5,
  "helpfulness": 4,
  "overall_quality": 4,
  "unsupported_claim": 0,
  "short_reason": "The reply is polite, relevant, and strongly grounded in the provided historical cases, offering a standard DM transition to troubleshoot the issues.",
  "major_issue": "none"
}


In [136]:
# ---------------------------------------------------------
# Finish the remaining LLM-as-judge evaluations
# ---------------------------------------------------------

import json
import time
import pandas as pd

judge_path = evaluation_dir / "apple_support_llm_judge_30.csv"

judge_data = pd.read_csv(judge_path)

if "judge_completed" not in judge_data.columns:
    judge_data["judge_completed"] = False

judge_data["judge_completed"] = (
    judge_data["judge_completed"]
    .fillna(False)
    .astype(bool)
)

# ---------------------------------------------------------
# Save the successful test result
# ---------------------------------------------------------

test_id = test_row["evaluation_id"]

if not judge_data.loc[
    judge_data["evaluation_id"] == test_id,
    "judge_completed"
].iloc[0]:

    for key, value in test_judge_result.items():
        judge_data.loc[
            judge_data["evaluation_id"] == test_id,
            f"judge_{key}"
        ] = value

    judge_data.loc[
        judge_data["evaluation_id"] == test_id,
        "judge_completed"
    ] = True

    judge_data.to_csv(
        judge_path,
        index=False
    )

    print(
        f"Saved test judgment: Evaluation ID {test_id}"
    )

# ---------------------------------------------------------
# Find remaining
# ---------------------------------------------------------

pending_mask = (
    judge_data["draft_reply"].notna()
    & ~judge_data["judge_completed"]
)

pending_indices = judge_data[pending_mask].index.tolist()

print(
    "\nRemaining judge evaluations:",
    len(pending_indices)
)

successful = 0
daily_quota_hit = False

# ---------------------------------------------------------
# Process remaining examples
# ---------------------------------------------------------

for position, idx in enumerate(
    pending_indices,
    start=1
):

    row = judge_data.loc[idx]

    print(
        f"\n[{position}/{len(pending_indices)}] "
        f"Evaluation ID: {row['evaluation_id']}"
    )

    while True:

        try:

            formatted_cases = format_retrieved_cases(
                row["customer_message"],
                top_k=5
            )

            prompt = build_judge_prompt(
                row,
                formatted_cases
            )

            response = gemini_client.models.generate_content(
                model="gemini-3.5-flash",
                contents=prompt,
                config={
                    "temperature": 0.0,
                    "response_mime_type": "application/json"
                }
            )

            judge_result = parse_judge_response(
                response.text
            )

            # Save scores
            for key, value in judge_result.items():

                judge_data.loc[
                    idx,
                    f"judge_{key}"
                ] = value

            judge_data.loc[
                idx,
                "judge_completed"
            ] = True

            # Checkpoint
            judge_data.to_csv(
                judge_path,
                index=False
            )

            successful += 1

            print(
                "Overall quality:",
                judge_result["overall_quality"]
            )

            print(
                "Unsupported claim:",
                judge_result["unsupported_claim"]
            )

            # Stay below 5 requests/minute.
            time.sleep(15)

            break

        except Exception as e:

            error_text = str(e)

            print("\nRequest failed:")
            print(error_text)

            # Daily quota
            if (
                "GenerateRequestsPerDay" in error_text
                or "PerDayPerProject" in error_text
            ):

                print(
                    "\nDaily quota reached. "
                    "Stopping safely."
                )

                daily_quota_hit = True
                break

            # Per-minute quota
            if "429" in error_text:

                print(
                    "\nPer-minute limit reached."
                    "\nWaiting 35 seconds..."
                )

                time.sleep(35)
                continue

            # Other error
            judge_data.loc[
                idx,
                "judge_error"
            ] = error_text

            judge_data.to_csv(
                judge_path,
                index=False
            )

            print("Skipping this example.")
            break

    if daily_quota_hit:
        break


# ---------------------------------------------------------
# Final status
# ---------------------------------------------------------

judge_data = pd.read_csv(judge_path)

completed = (
    judge_data["judge_completed"]
    .fillna(False)
    .astype(bool)
)

print("\n" + "=" * 60)
print("FINAL JUDGE STATUS")
print("=" * 60)

print("Total examples:", len(judge_data))
print("Generated replies:", judge_data["draft_reply"].notna().sum())
print("Judged:", completed.sum())
print(
    "Remaining:",
    (
        judge_data["draft_reply"].notna()
        & ~completed
    ).sum()
)

print("\nSaved to:")
print(judge_path)

Saved test judgment: Evaluation ID 21

Remaining judge evaluations: 9

[1/9] Evaluation ID: 22
Overall quality: 5
Unsupported claim: 0

[2/9] Evaluation ID: 23
Overall quality: 5
Unsupported claim: 0

[3/9] Evaluation ID: 24
Overall quality: 5
Unsupported claim: 0

[4/9] Evaluation ID: 25
Overall quality: 5
Unsupported claim: 0

[5/9] Evaluation ID: 26
Overall quality: 5
Unsupported claim: 0

[6/9] Evaluation ID: 27
Overall quality: 5
Unsupported claim: 0

[7/9] Evaluation ID: 28
Overall quality: 5
Unsupported claim: 0

[8/9] Evaluation ID: 29
Overall quality: 5
Unsupported claim: 0

[9/9] Evaluation ID: 30
Overall quality: 4
Unsupported claim: 0

FINAL JUDGE STATUS
Total examples: 30
Generated replies: 30
Judged: 30
Remaining: 0

Saved to:
c:\Projects\hiver-support-agent\data\evaluation\apple_support_llm_judge_30.csv


In [137]:
# ---------------------------------------------------------
# FINAL LLM-AS-JUDGE METRICS
# No API call
# ---------------------------------------------------------

import pandas as pd
import numpy as np

judge_path = evaluation_dir / "apple_support_llm_judge_30.csv"

judge_data = pd.read_csv(judge_path)

judge_data["judge_completed"] = (
    judge_data["judge_completed"]
    .fillna(False)
    .astype(bool)
)

completed = judge_data[
    judge_data["judge_completed"]
].copy()

print("=" * 60)
print("FINAL LLM-AS-JUDGE RESULTS")
print("=" * 60)

print("Total evaluation examples:", len(judge_data))
print("Generated replies:", judge_data["draft_reply"].notna().sum())
print("Completed judgments:", len(completed))

# ---------------------------------------------------------
# Convert scores
# ---------------------------------------------------------

score_cols = [
    "judge_relevance",
    "judge_grounding",
    "judge_helpfulness",
    "judge_overall_quality",
    "judge_unsupported_claim"
]

for col in score_cols:
    completed[col] = pd.to_numeric(
        completed[col],
        errors="coerce"
    )

# ---------------------------------------------------------
# Overall metrics
# ---------------------------------------------------------

metrics = {
    "examples_judged": len(completed),
    "mean_relevance": completed["judge_relevance"].mean(),
    "mean_grounding": completed["judge_grounding"].mean(),
    "mean_helpfulness": completed["judge_helpfulness"].mean(),
    "mean_overall_quality": completed["judge_overall_quality"].mean(),
    "median_overall_quality": completed["judge_overall_quality"].median(),
    "unsupported_claim_rate": completed["judge_unsupported_claim"].mean()
}

print("\nOverall metrics:")

for key, value in metrics.items():

    if "rate" in key:
        print(f"{key}: {value:.1%}")

    elif "examples" in key:
        print(f"{key}: {value}")

    else:
        print(f"{key}: {value:.3f}")

# ---------------------------------------------------------
# Overall quality distribution
# ---------------------------------------------------------

print("\nOverall quality distribution:")

quality_distribution = (
    completed["judge_overall_quality"]
    .value_counts()
    .sort_index()
)

display(
    quality_distribution.rename(
        "count"
    ).to_frame()
)

# ---------------------------------------------------------
# Unsupported claims
# ---------------------------------------------------------

print("\nUnsupported claims:")

unsupported = (
    completed["judge_unsupported_claim"]
    .value_counts()
    .sort_index()
)

display(
    unsupported.rename(
        "count"
    ).to_frame()
)

FINAL LLM-AS-JUDGE RESULTS
Total evaluation examples: 30
Generated replies: 30
Completed judgments: 30

Overall metrics:
examples_judged: 30
mean_relevance: 4.967
mean_grounding: 5.000
mean_helpfulness: 4.767
mean_overall_quality: 4.767
median_overall_quality: 5.000
unsupported_claim_rate: 0.0%

Overall quality distribution:


,count
judge_overall_quality,
4.0,7
5.0,23



Unsupported claims:


,count
judge_unsupported_claim,
0.0,30


In [138]:
# ---------------------------------------------------------
# LLM quality by predicted intent
# ---------------------------------------------------------

intent_quality = (
    completed
    .groupby("predicted_intent")
    .agg(
        examples=("evaluation_id", "count"),
        relevance=("judge_relevance", "mean"),
        grounding=("judge_grounding", "mean"),
        helpfulness=("judge_helpfulness", "mean"),
        overall_quality=("judge_overall_quality", "mean"),
        unsupported_claim_rate=(
            "judge_unsupported_claim",
            "mean"
        )
    )
    .sort_values(
        "overall_quality",
        ascending=False
    )
)

display(
    intent_quality.round(3)
)

,examples,relevance,grounding,helpfulness,overall_quality,unsupported_claim_rate
predicted_intent,,,,,,
audio,1,5.00,5.0,5.00,5.00,0.0
device_hardware,1,5.00,5.0,5.00,5.00,0.0
display_input,6,5.00,5.0,5.00,5.00,0.0
connectivity,1,5.00,5.0,5.00,5.00,0.0
ios_update,5,5.00,5.0,4.80,4.80,0.0
apple_services,8,5.00,5.0,4.75,4.75,0.0
account_authentication,2,5.00,5.0,4.50,4.50,0.0
battery_charging,2,5.00,5.0,4.50,4.50,0.0
apps_app_store,4,4.75,5.0,4.50,4.50,0.0


In [139]:
# ---------------------------------------------------------
# Lowest-quality generated responses
# ---------------------------------------------------------

worst_examples = (
    completed
    .sort_values(
        [
            "judge_overall_quality",
            "judge_grounding",
            "judge_helpfulness"
        ],
        ascending=True
    )
    [
        [
            "evaluation_id",
            "customer_message",
            "predicted_intent",
            "draft_reply",
            "judge_relevance",
            "judge_grounding",
            "judge_helpfulness",
            "judge_overall_quality",
            "judge_unsupported_claim",
            "judge_major_issue",
            "judge_short_reason"
        ]
    ]
)

display(
    worst_examples.head(10)
)

,evaluation_id,customer_message,predicted_intent,draft_reply,judge_relevance,judge_grounding,judge_helpfulness,judge_overall_quality,judge_unsupported_claim,judge_major_issue,judge_short_reason
0,1,Hey @115858 how can Siri really be my personal assistant if it won't integrate into 3rd party apps why can't I have it play Spotify where is the functionality?,apple_services,We understand wanting Siri to work with your favorite apps. Let's look into this together. Please send us a DM so we can help.,5.0,5.0,4.0,4.0,0.0,none,"The reply is highly relevant to the customer's query about Siri and third-party apps, and it aligns perfectly with the historical support cases which consistently direct customers to DM for assist..."
8,9,"@AppleSupport hey why is the remove episode option gone from the podcast app, subscribed podcasts keep popping up after I delete them",apps_app_store,We'd like to help. Does this issue with deleted episodes reappearing occur with all podcasts or a specific one? Please meet us in DM so we can look into this further.,5.0,5.0,4.0,4.0,0.0,none,"The reply is highly relevant, well-grounded in the provided historical cases (specifically Case 3), and offers a helpful next step by asking a clarifying question and inviting the customer to DM."
11,12,@AppleSupport my iMessage has not worked in over a month at this point. I’ve spent considerable amount of time with your customer support to get this solved but there has been no solution yet. Ple...,ios_update,We want to help get your iMessage working as soon as possible. Please send us a DM so we can look into this further and help find a solution.,5.0,5.0,4.0,4.0,0.0,none,"The reply is relevant and grounded in the historical cases, offering to continue the conversation in DM. However, it could have acknowledged the customer's frustration regarding their previous uns..."
14,15,Anyone know how to covert an existing iCloud account to a child account? My kids would hate to lose any game data. cc @AppleSupport,account_authentication,We'd love to help you with your family's iCloud accounts and make sure no game data is lost. Please join us in DM so we can gather more details and guide you through this: https://t.co/GDrqU22YpT,5.0,5.0,4.0,4.0,0.0,none,"The reply politely invites the customer to DM to resolve their specific account migration issue, which is consistent with how Apple Support handles complex account queries in the provided cases."
19,20,"@AppleSupport why offer app ""controls"" in a managed account when apple apps mysteriously enable themselves when login back, such as safari? #unbelievable",apps_app_store,We're here for you. Let's take a look at this further for you. Reach out to us in DM.,4.0,5.0,4.0,4.0,0.0,none,"The reply appropriately invites the customer to DM to troubleshoot the complex managed account issue further, which is strongly supported by the historical cases."
20,21,"@115858 please say your developers are hard at work on iOS 11.0.4 so that my headphones, charger and volume buttons work again, ughhhh 🙄🙄🙄🙄🙄",apple_services,"We want your device to work seamlessly. Join us in DM with more details about what's happening, and we'll take a look into this with you. https://t.co/GDrqU22YpT",5.0,5.0,4.0,4.0,0.0,none,"The reply is polite, relevant, and strongly grounded in the provided historical cases, offering a standard DM transition to troubleshoot the issues."
29,30,@AppleSupport The “done” button will not let me select my picture. It’s covered in the top right corner of the phone with the battery and wifi icons. It gets stuck and I have to close out the app.,battery_charging,We'd like to help look into this with you. Go ahead and reach out to us in DM and we'll continue there.,5.0,5.0,4.0,4.0,0.0,none,The reply is highly relevant and grounded in the historical cases where customers experiencing UI issues with photos are invited to DM. It could be improved by including a direct DM link.
1,2,@AppleSupport Was able to help him via iTunes. Still weird the 

In [140]:
# ---------------------------------------------------------
# Create a fixed 10-example human evaluation subset
# ---------------------------------------------------------

import pandas as pd

judge_path = evaluation_dir / "apple_support_llm_judge_30.csv"

judge_data = pd.read_csv(judge_path)

completed = judge_data[
    judge_data["judge_completed"].fillna(False).astype(bool)
].copy()

# Fixed random sample for reproducibility.
human_subset = (
    completed
    .sample(
        n=10,
        random_state=2026
    )
    .sort_values("evaluation_id")
    .reset_index(drop=True)
)

# Keep only information a human evaluator needs.
human_review = human_subset[
    [
        "evaluation_id",
        "customer_message",
        "previous_message",
        "predicted_intent",
        "draft_reply"
    ]
].copy()

# Human scoring fields.
human_review["human_relevance"] = ""
human_review["human_grounding"] = ""
human_review["human_helpfulness"] = ""
human_review["human_overall_quality"] = ""
human_review["human_unsupported_claim"] = ""
human_review["human_notes"] = ""

human_review_path = (
    evaluation_dir /
    "apple_support_human_review_10.csv"
)

human_review.to_csv(
    human_review_path,
    index=False
)

print("Human review set created.")
print("Examples:", len(human_review))
print("\nEvaluation IDs:")
print(
    human_review["evaluation_id"].tolist()
)

print("\nSaved:")
print(human_review_path)

Human review set created.
Examples: 10

Evaluation IDs:
[4, 9, 10, 11, 12, 15, 19, 22, 23, 24]

Saved:
c:\Projects\hiver-support-agent\data\evaluation\apple_support_human_review_10.csv


In [141]:
# ---------------------------------------------------------
# Create a readable human-review interface
# No Gemini API calls
# ---------------------------------------------------------

import pandas as pd

judge_path = evaluation_dir / "apple_support_llm_judge_30.csv"

judge_data = pd.read_csv(judge_path)

completed = judge_data[
    judge_data["judge_completed"].fillna(False).astype(bool)
].copy()

# Same fixed 10 examples as before
human_subset = (
    completed
    .sample(
        n=10,
        random_state=2026
    )
    .sort_values("evaluation_id")
    .reset_index(drop=True)
)

print("=" * 80)
print("HUMAN REVIEW — 10 EXAMPLES")
print("=" * 80)

for i, row in human_subset.iterrows():

    print("\n" + "=" * 80)
    print(f"EXAMPLE {i + 1} / 10")
    print(f"Evaluation ID: {row['evaluation_id']}")
    print("=" * 80)

    print("\nCUSTOMER MESSAGE:")
    print("-" * 80)
    print(row["customer_message"])

    if (
        pd.notna(row["previous_message"])
        and str(row["previous_message"]).strip()
    ):
        print("\nPREVIOUS MESSAGE:")
        print("-" * 80)
        print(row["previous_message"])

    print("\nPREDICTED INTENT:")
    print("-" * 80)
    print(row["predicted_intent"])

    print("\nAI-GENERATED REPLY:")
    print("-" * 80)
    print(row["draft_reply"])

    print("\nYOUR SCORES:")
    print("-" * 80)
    print("Relevance       : 1-5")
    print("Grounding       : 1-5")
    print("Helpfulness     : 1-5")
    print("Overall quality : 1-5")
    print("Unsupported     : 0 = No, 1 = Yes")

    print("\n")

HUMAN REVIEW — 10 EXAMPLES

EXAMPLE 1 / 10
Evaluation ID: 4

CUSTOMER MESSAGE:
--------------------------------------------------------------------------------
Hey @AppleSupport iOS11 update made my $900 iPad Pro unusable. The battery lasts 1/2 day even if just sitting there in airplane mode. #SUCKS

PREDICTED INTENT:
--------------------------------------------------------------------------------
battery_charging

AI-GENERATED REPLY:
--------------------------------------------------------------------------------
We'd be happy to look into this with you. Please meet us in DM so we can gather some more information and better assist. https://t.co/GDrqU22YpT

YOUR SCORES:
--------------------------------------------------------------------------------
Relevance       : 1-5
Grounding       : 1-5
Helpfulness     : 1-5
Overall quality : 1-5
Unsupported     : 0 = No, 1 = Yes



EXAMPLE 2 / 10
Evaluation ID: 9

CUSTOMER MESSAGE:
----------------------------------------------------------------

In [142]:
import pandas as pd

human_scores = {
    4:  (5, 5, 3, 4, 0),
    9:  (5, 5, 4, 4, 0),
    10: (5, 5, 4, 4, 0),
    11: (5, 5, 5, 5, 0),
    12: (5, 4, 4, 3, 0),
    15: (5, 5, 4, 5, 0),
    19: (5, 4, 4, 4, 0),
    22: (5, 5, 5, 5, 0),
    23: (5, 5, 5, 5, 0),
    24: (5, 5, 5, 5, 0),
}

human_path = evaluation_dir / "apple_support_human_review_10.csv"

human_review = pd.read_csv(human_path)

# Remove accidental ** characters if they exist in column names
human_review.columns = (
    human_review.columns
    .str.replace("*", "", regex=False)
    .str.strip()
)

for eval_id, scores in human_scores.items():
    mask = human_review["evaluation_id"] == eval_id

    human_review.loc[mask, "human_relevance"] = scores[0]
    human_review.loc[mask, "human_grounding"] = scores[1]
    human_review.loc[mask, "human_helpfulness"] = scores[2]
    human_review.loc[mask, "human_overall_quality"] = scores[3]
    human_review.loc[mask, "human_unsupported_claim"] = scores[4]

human_review.to_csv(human_path, index=False)

print(human_review.to_string(index=False))

 evaluation_id                                                                                                                                                                                                                                                               customer_message                                                                                                           previous_message       predicted_intent                                                                                                                                                                                                                                                               draft_reply  human_relevance  human_grounding  human_helpfulness  human_overall_quality  human_unsupported_claim  human_notes
             4                                                                                                                                   Hey @AppleSupport iOS11 update made my $900

In [145]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score

# ---------------------------------------------------------
# Load files
# ---------------------------------------------------------

human_path = evaluation_dir / "apple_support_human_review_10.csv"
judge_path = evaluation_dir / "apple_support_llm_judge_30.csv"

human = pd.read_csv(human_path)
judge = pd.read_csv(judge_path)

# Clean column names
human.columns = (
    human.columns
    .str.replace("*", "", regex=False)
    .str.strip()
)

judge.columns = (
    judge.columns
    .str.replace("*", "", regex=False)
    .str.strip()
)

# ---------------------------------------------------------
# Merge human and LLM scores
# ---------------------------------------------------------

human_columns = [
    "evaluation_id",
    "human_relevance",
    "human_grounding",
    "human_helpfulness",
    "human_overall_quality",
    "human_unsupported_claim"
]

judge_columns = [
    "evaluation_id",
    "judge_relevance",
    "judge_grounding",
    "judge_helpfulness",
    "judge_overall_quality",
    "judge_unsupported_claim"
]

comparison = human[human_columns].merge(
    judge[judge_columns],
    on="evaluation_id",
    how="inner"
)

print("Matched examples:", len(comparison))

# ---------------------------------------------------------
# Calculate agreement
# ---------------------------------------------------------

dimensions = [
    ("Relevance", "human_relevance", "judge_relevance"),
    ("Grounding", "human_grounding", "judge_grounding"),
    ("Helpfulness", "human_helpfulness", "judge_helpfulness"),
    ("Overall Quality", "human_overall_quality", "judge_overall_quality"),
]

results = []

for name, human_col, judge_col in dimensions:

    human_scores = comparison[human_col].astype(int)
    judge_scores = comparison[judge_col].astype(int)

    exact_agreement = (
        human_scores == judge_scores
    ).mean()

    mean_absolute_difference = np.abs(
        human_scores - judge_scores
    ).mean()

    weighted_kappa = cohen_kappa_score(
        human_scores,
        judge_scores,
        weights="quadratic"
    )

    results.append({
        "dimension": name,
        "exact_agreement": exact_agreement,
        "mean_absolute_difference": mean_absolute_difference,
        "quadratic_weighted_kappa": weighted_kappa
    })

# ---------------------------------------------------------
# Unsupported claims
# ---------------------------------------------------------

human_scores = comparison["human_unsupported_claim"].astype(int)
judge_scores = comparison["judge_unsupported_claim"].astype(int)

results.append({
    "dimension": "Unsupported Claim",
    "exact_agreement": (
        human_scores == judge_scores
    ).mean(),
    "mean_absolute_difference": np.abs(
        human_scores - judge_scores
    ).mean(),
    "quadratic_weighted_kappa": cohen_kappa_score(
        human_scores,
        judge_scores
    )
})

# ---------------------------------------------------------
# Display results
# ---------------------------------------------------------

agreement_results = pd.DataFrame(results)

print("\n" + "=" * 80)
print("HUMAN vs LLM JUDGE AGREEMENT")
print("=" * 80)

print(
    agreement_results.to_string(
        index=False,
        formatters={
            "exact_agreement": "{:.1%}".format,
            "mean_absolute_difference": "{:.2f}".format,
            "quadratic_weighted_kappa": "{:.3f}".format,
        }
    )
)

Matched examples: 10

HUMAN vs LLM JUDGE AGREEMENT
        dimension exact_agreement mean_absolute_difference quadratic_weighted_kappa
        Relevance          100.0%                     0.00                      NaN
        Grounding           80.0%                     0.20                    0.000
      Helpfulness           70.0%                     0.40                    0.231
  Overall Quality           50.0%                     0.50                    0.324
Unsupported Claim          100.0%                     0.00                      NaN


c:\Projects\hiver-support-agent\.venv\Lib\site-packages\sklearn\metrics\_classification.py:614: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Projects\hiver-support-agent\.venv\Lib\site-packages\sklearn\utils\_param_validation.py:218: UndefinedMetricWarning: `y1`, `y2` and `labels` have only one label in common. `cohen_kappa_score` is undefined and set to the value defined by the the `replace_undefined_by` param, which is set to nan.
  return func(*args, **kwargs)
c:\Projects\hiver-support-agent\.venv\Lib\site-packages\sklearn\metrics\_classification.py:614: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Projects\hiver-support-agent\.venv\Lib\site-packages\sklearn\utils\_param_validation.py:218: UndefinedM

In [146]:
# ---------------------------------------------------------
# Save human-vs-LLM agreement results
# ---------------------------------------------------------

agreement_path = evaluation_dir / "human_llm_judge_agreement.csv"

agreement_results.to_csv(
    agreement_path,
    index=False
)

print(f"Saved: {agreement_path}")
print("\nFinal agreement results:")
print(agreement_results)

Saved: c:\Projects\hiver-support-agent\data\evaluation\human_llm_judge_agreement.csv

Final agreement results:
           dimension  exact_agreement  mean_absolute_difference  \
0          Relevance              1.0                       0.0   
1          Grounding              0.8                       0.2   
2        Helpfulness              0.7                       0.4   
3    Overall Quality              0.5                       0.5   
4  Unsupported Claim              1.0                       0.0   

   quadratic_weighted_kappa  
0                       NaN  
1                  0.000000  
2                  0.230769  
3                  0.324324  
4                       NaN  


In [147]:
print(pd.read_csv(agreement_path))

           dimension  exact_agreement  mean_absolute_difference  \
0          Relevance              1.0                       0.0   
1          Grounding              0.8                       0.2   
2        Helpfulness              0.7                       0.4   
3    Overall Quality              0.5                       0.5   
4  Unsupported Claim              1.0                       0.0   

   quadratic_weighted_kappa  
0                       NaN  
1                  0.000000  
2                  0.230769  
3                  0.324324  
4                       NaN  


In [148]:
# ---------------------------------------------------------
# Failure analysis dataset
# ---------------------------------------------------------

oof_errors = oof_results[
    oof_results["true_intent"] != oof_results["predicted_intent"]
].copy()

print("Total classifier errors:", len(oof_errors))

print("\nTop confusion pairs:")
print(
    oof_errors
    .groupby(["true_intent", "predicted_intent"])
    .size()
    .sort_values(ascending=False)
    .head(15)
)


Total classifier errors: 100

Top confusion pairs:
true_intent      predicted_intent
connectivity     display_input       6
ios_update       display_input       6
display_input    ios_update          5
apple_services   connectivity        4
                 apps_app_store      4
                 ios_update          3
                 display_input       3
ios_update       connectivity        3
apps_app_store   apple_services      3
other_unclear    display_input       3
display_input    apple_services      3
audio            display_input       3
                 ios_update          2
                 apple_services      2
device_hardware  display_input       2
dtype: int64


In [154]:
# ---------------------------------------------------------
# Extract real examples for the top 5 failure modes
# ---------------------------------------------------------

failure_examples = oof_errors[
    [
        "customer_tweet_id",
        "customer_message",
        "previous_message",
        "true_intent",
        "predicted_intent",
        "margin"
    ]
].copy()

pairs_to_inspect = [
    ("connectivity", "display_input"),
    ("ios_update", "display_input"),
    ("display_input", "ios_update"),
    ("apple_services", "connectivity"),
    ("apple_services", "apps_app_store"),
]

for true_intent, predicted_intent in pairs_to_inspect:

    pair_errors = failure_examples[
        (failure_examples["true_intent"] == true_intent) &
        (failure_examples["predicted_intent"] == predicted_intent)
    ]

    examples = pair_errors.head(3)

    print("\n" + "=" * 80)
    print(f"TRUE: {true_intent}  →  PREDICTED: {predicted_intent}")
    print(f"Number of errors: {len(pair_errors)}")
    print("=" * 80)

    for _, row in examples.iterrows():

        print(f"\nTweet ID: {row['customer_tweet_id']}")
        print(f"Margin: {row['margin']:.4f}")

        print("\nCUSTOMER:")
        print(row["customer_message"])

        previous = str(row["previous_message"]).strip()

        if previous and previous.lower() != "nan":
            print("\nPREVIOUS MESSAGE:")
            print(previous)


TRUE: connectivity  →  PREDICTED: display_input
Number of errors: 6

Tweet ID: 146553.0
Margin: 0.5366

CUSTOMER:
@115858 @4792 CarPlay and iPhoneX and the 2017 M240i don‘t work together. When is a fix for that coming?

Tweet ID: 1437557.0
Margin: 0.0937

CUSTOMER:
Absolutely hate it when you airdrop a bunch of photos &amp; they go all jumbled up 😖 @115858 #whyyoutestingme

Tweet ID: 1484796.0
Margin: 0.0423

CUSTOMER:
@115858 you really need to fix this update or something because every time me and my friend try to FaceTime or call, the other person doesn’t get any notification for it at all and I’m gonna scream

TRUE: ios_update  →  PREDICTED: display_input
Number of errors: 6

Tweet ID: 400853.0
Margin: 0.0295

CUSTOMER:
@AppleSupport Was able to help him via iTunes. Still weird the OTA didn’t work

PREVIOUS MESSAGE:
@210813 Let us know if the update has issues through iTunes.

Tweet ID: 1687712.0
Margin: 0.0154

CUSTOMER:
Good afternoon, @AppleSupport! It’s always some shit with y

In [152]:
print(oof_errors.columns.tolist())

['customer_tweet_id', 'customer_message', 'previous_message', 'has_context', 'true_intent', 'predicted_intent', 'top_score', 'second_score', 'margin', 'is_error']


In [155]:
# ---------------------------------------------------------
# One representative real example for each major failure
# ---------------------------------------------------------

pairs_to_inspect = [
    ("connectivity", "display_input"),
    ("ios_update", "display_input"),
    ("display_input", "ios_update"),
    ("apple_services", "connectivity"),
    ("apple_services", "apps_app_store"),
]

for true_intent, predicted_intent in pairs_to_inspect:

    pair_errors = oof_errors[
        (oof_errors["true_intent"] == true_intent) &
        (oof_errors["predicted_intent"] == predicted_intent)
    ].sort_values("margin")

    row = pair_errors.iloc[0]

    print("\n" + "=" * 80)
    print(f"{true_intent} → {predicted_intent}")
    print("=" * 80)
    print(f"Tweet ID: {row['customer_tweet_id']}")
    print(f"Margin: {row['margin']:.4f}")
    print(f"Customer: {row['customer_message']}")

    previous = str(row["previous_message"]).strip()

    if previous and previous.lower() != "nan":
        print(f"Previous: {previous}")


connectivity → display_input
Tweet ID: 308550.0
Margin: 0.0208
Customer: Completely fed up with bluetooth issues in iOS 11 - @115858, sort them out!

ios_update → display_input
Tweet ID: 1687712.0
Margin: 0.0154
Customer: Good afternoon, @AppleSupport! It’s always some shit with y’all and these iPhone updates, huh? https://t.co/cGR2qXnGNA

display_input → ios_update
Tweet ID: 34645.0
Margin: 0.0128
Customer: @AppleSupport It's 10.3.3. Although I'm convinced it's a screen issue rather than a software one.
Previous: @123511 We're happy to help. What version of iOS are you currently running? Find that in Settings &gt; General &gt; About.

apple_services → connectivity
Tweet ID: 2799435.0
Margin: 0.0095
Customer: @115858 @6990 why isn’t FaceTime working right now? I need to see @107462 face 😭😭😭

apple_services → apps_app_store
Tweet ID: 2503732.0
Margin: 0.0359
Customer: @AppleSupport I went on iTunes and re downloaded it and it still isn’t in my library. I turned my phone off for 1 hour 

In [156]:
# ---------------------------------------------------------
# Headline Results
# ---------------------------------------------------------

headline_results = {
    "Intent classifier accuracy": "50.0%",
    "Intent classifier macro F1": "43.7%",
    "Intent classifier weighted F1": "46.6%",
    "Golden evaluation set": "200 examples / 12 intents",
    "Historical retrieval top-1 similarity": "0.809 mean",
    "Historical retrieval top-5 intent alignment": "95.5%*",
    "Auto-handle rate": "25.5%",
    "Auto-handled accuracy": "84.3%",
    "Classifier errors captured by escalation": "92.0%",
    "LLM judge overall quality": "4.77 / 5",
    "LLM judge unsupported-claim rate": "0.0%",
    "Human-vs-LLM relevance agreement": "100%",
    "Human-vs-LLM grounding agreement": "80%",
    "Human-vs-LLM helpfulness agreement": "70%",
    "Human-vs-LLM overall-quality agreement": "50%",
}

for metric, value in headline_results.items():
    print(f"{metric}: {value}")

print("\n* Retrieval intent alignment is a diagnostic using the classifier's")
print("  predicted intents, not independent ground-truth retrieval labels.")

Intent classifier accuracy: 50.0%
Intent classifier macro F1: 43.7%
Intent classifier weighted F1: 46.6%
Golden evaluation set: 200 examples / 12 intents
Historical retrieval top-1 similarity: 0.809 mean
Historical retrieval top-5 intent alignment: 95.5%*
Auto-handle rate: 25.5%
Auto-handled accuracy: 84.3%
Classifier errors captured by escalation: 92.0%
LLM judge overall quality: 4.77 / 5
LLM judge unsupported-claim rate: 0.0%
Human-vs-LLM relevance agreement: 100%
Human-vs-LLM grounding agreement: 80%
Human-vs-LLM helpfulness agreement: 70%
Human-vs-LLM overall-quality agreement: 50%

* Retrieval intent alignment is a diagnostic using the classifier's
  predicted intents, not independent ground-truth retrieval labels.


## What is misleading about my headline number?

The 50.0% intent-classification accuracy should not be interpreted as the overall
quality of the support agent.

First, the classifier was evaluated on only 200 manually reviewed examples
covering 12 intents. Several intents have very small support, so the estimate is
high variance and should not be treated as production-level accuracy.

Second, the headline accuracy hides substantial class imbalance and uneven
performance across intents. For example, battery_charging and camera_photos
performed substantially better than ios_update, audio, and other_unclear.

Third, the agent is not based on classification alone. It combines intent
classification, historical-case retrieval, an LLM response generator, and a
deterministic escalation policy. Therefore, classification accuracy is only one
component of the complete system.

The 92.0% error-capture figure is also easy to overinterpret. It means that,
on this 200-example development evaluation, the conservative escalation policy
routed 92 of the 100 classifier errors to a human. It does not mean that 92%
of all real-world support failures will be detected.

The escalation threshold was selected using the same 200-example development
set, so the reported coverage and error-capture figures are not an unbiased
estimate of production performance.

Similarly, the 4.77/5 LLM-judge score comes from only 30 generated responses.
The judge-human agreement audit contains only 10 examples, so it provides a
useful sanity check but is too small to establish strong judge reliability.

Finally, the historical retrieval "intent alignment" metric is a diagnostic
rather than an independent retrieval accuracy measure because the retrieved
cases were assigned intents using the same final classifier rather than
independent human labels.

## Next Week Plan

### 1. Expand the golden evaluation set
Increase the labeled evaluation set beyond 200 examples, with deliberate
coverage of minority intents and ambiguous boundary cases.

### 2. Improve intent boundaries
Focus additional annotation on the major confusion pairs:
- connectivity vs display_input
- ios_update vs display_input
- apple_services vs connectivity
- apple_services vs apps_app_store

### 3. Add context-aware classification
Use previous customer/brand context more systematically, while ensuring that
future customer messages are not contaminated by response information.

### 4. Improve escalation calibration
Tune the classifier-margin and retrieval-similarity thresholds on a separate
validation set rather than the same set used for final evaluation.

### 5. Improve retrieval evaluation
Create independently labeled retrieval relevance judgments instead of using
classifier predictions as a proxy for retrieved-case intent.

### 6. Improve response safety
Prevent raw historical social-media URLs from being copied into generated
responses unless they have been explicitly validated.

### 7. Strengthen LLM-judge validation
Increase the human audit set and measure agreement on a larger, independently
reviewed sample before relying on the judge for regression testing.

In [158]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: c:\Projects\hiver-support-agent


In [159]:
from src.intent_classifier import train_final_model

intent_model = train_final_model()

print("Model trained successfully.")

Model trained successfully.


In [160]:
from src.retrieval import build_retriever

retriever = build_retriever()

print("Retriever ready.")

Retrieval corpus size: 106,323


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/3323 [00:00<?, ?it/s]

Retriever ready.


In [161]:
results = retriever.retrieve(
    "My iPhone battery is draining really fast after the latest update.",
    top_k=5
)

for result in results:
    print("\nRank:", result["rank"])
    print("Similarity:", round(result["similarity"], 4))
    print("Customer:", result["customer_message"])
    print("Response:", result["brand_response"])


Rank: 1
Similarity: 0.8858
Customer: @115858 Why is my iPhone battery draining so rapidly after the update?
Response: @327697 We'd like to see what we can do to help. Are you running iOS 11.0.3? We released it just the other day.

Rank: 2
Similarity: 0.8644
Customer: @AppleSupport: after installing several updates on my iPhone, the battery is draining very quickly with minimal usage. Can you help, pls?
Response: @556995 We know how vital the battery is. Please check out this article: https://t.co/TpjqFp3jxD  

DM us the result. https://t.co/GDrqU22YpT

Rank: 3
Similarity: 0.8567
Customer: My iPhone battery drains so quickly with this new update ! @AppleSupport @115858 FIX IT !!!!
Response: @664427 We want you to be able to rely on your battery. Have you had a chance to take a look at the tips here for maximizing your battery life: https://t.co/bivpdfBNJ6 Is the iOS version you're using iOS 11.1.1?

Rank: 4
Similarity: 0.847
Customer: @AppleSupport why is my battery draining sooo very 

In [162]:
from src.escalation import decide_escalation

result = decide_escalation(
    intent="battery_charging",
    classifier_margin=0.89,
    top_similarity=0.89,
    customer_message=(
        "My iPhone battery is draining really fast "
        "after the latest update."
    ),
)

print(result)

{'escalate': False, 'escalation_reason': 'The issue has sufficient intent confidence and relevant historical support evidence for automated handling.'}


In [163]:
result = decide_escalation(
    intent="apple_services",
    classifier_margin=0.08,
    top_similarity=0.60,
    customer_message="Having the same issue",
)

print(result)

{'escalate': True, 'escalation_reason': 'The customer message does not provide enough specific information. The intent classifier is not sufficiently confident. No sufficiently similar historical support case was retrieved.'}


In [164]:
from src.agent import support_agent

result = support_agent(
    "My iPhone battery is draining really fast "
    "after the latest update."
)

print("Intent:", result["intent"])
print("Classifier margin:", result["classifier_margin"])
print("Retrieval similarity:", result["top_similarity"])
print("Escalate:", result["escalate"])
print("Reason:", result["escalation_reason"])
print("\nDraft reply:")
print(result["draft_reply"])

Training intent classifier...
Building historical retrieval index...
Retrieval corpus size: 106,323


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/3323 [00:00<?, ?it/s]

Agent initialized.
Intent: battery_charging
Classifier margin: 0.9161
Retrieval similarity: 0.8858
Escalate: False
Reason: The issue has sufficient intent confidence and relevant historical support evidence for automated handling.

Draft reply:
We want to make sure you can rely on your battery. We're glad to look at this with you. Could you send us a DM with your iPhone model and the iOS version you installed?


In [165]:
from src.retrieval import build_retriever

retriever = build_retriever()

Retrieval corpus size: 106,323


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/3323 [00:00<?, ?it/s]

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

c:\Projects\hiver-support-agent


In [2]:
from src.agent import create_agent

agent = create_agent()

print("Agent object created successfully.")

Agent object created successfully.


In [4]:
from pathlib import Path

cache_dir = Path.cwd().parent / "data" / "processed" / "retrieval_index"

print("Cache directory:", cache_dir)
print("Directory exists:", cache_dir.exists())

if cache_dir.exists():
    print("Files:")
    for file in cache_dir.iterdir():
        print(" -", file.name, file.stat().st_size / (1024 * 1024), "MB")

Cache directory: c:\Projects\hiver-support-agent\data\processed\retrieval_index
Directory exists: True
Files:


In [6]:
import src.retrieval
import inspect

print(src.retrieval.__file__)
print("Has caching:", "INDEX_PATH" in dir(src.retrieval))

c:\Projects\hiver-support-agent\src\retrieval.py
Has caching: True


In [7]:
print(inspect.getsource(src.retrieval.build_retriever))

def build_retriever(
    clean_path: Path = CLEAN_APPLE_PATH,
) -> HistoricalRetriever:
    """
    Load or build the historical retrieval index.

    First run:
        Build embeddings → save FAISS index.

    Later runs:
        Load the saved FAISS index directly.
    """

    RETRIEVAL_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    # ---------------------------------------------
    # Load cached index if available
    # ---------------------------------------------

    if INDEX_PATH.exists() and CORPUS_PATH.exists():

        print("Loading cached retrieval index...")

        corpus = pd.read_csv(
            CORPUS_PATH
        )

        index = faiss.read_index(
            str(INDEX_PATH)
        )

        print(
            f"Loaded retrieval index with "
            f"{index.ntotal:,} cases."
        )

        return HistoricalRetriever(
            corpus=corpus,
            index=index,
        )

    # ---------------------------------------------

In [8]:
from src.retrieval import build_retriever

retriever = build_retriever()

Retrieval corpus size: 106,323


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding 106,323 historical messages...


Batches:   0%|          | 0/3323 [00:00<?, ?it/s]

Saving retrieval index...
Saved index to: C:\Projects\hiver-support-agent\data\processed\retrieval_index\apple_support.faiss


In [9]:
from pathlib import Path

cache_dir = Path.cwd().parent / "data" / "processed" / "retrieval_index"

for file in cache_dir.iterdir():
    print(
        file.name,
        round(file.stat().st_size / (1024 * 1024), 2),
        "MB"
    )

apple_support.faiss 155.75 MB
retrieval_corpus.csv 31.67 MB


In [10]:
from src.retrieval import build_retriever

retriever = build_retriever()

Loading cached retrieval index...
Loaded retrieval index with 106,323 cases.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [12]:
import pandas as pd
from src.config import GOLDEN_SET_PATH

golden_set = pd.read_csv(GOLDEN_SET_PATH)

print("Golden set shape:", golden_set.shape)
print("Intents:", golden_set["intent"].nunique())
print("Missing labels:", golden_set["intent"].isna().sum())

Golden set shape: (200, 6)
Intents: 12
Missing labels: 0


In [13]:
from src.evaluation import (
    evaluate_per_intent,
    build_word_char_svm,
)

per_intent, predictions = evaluate_per_intent(
    build_word_char_svm(),
    golden_set,
)

per_intent

,intent,precision,recall,f1,support
0,account_authentication,0.619048,0.722222,0.666667,18.0
1,apple_services,0.333333,0.280000,0.304348,25.0
2,apps_app_store,0.583333,0.700000,0.636364,20.0
3,audio,0.000000,0.000000,0.000000,12.0
4,battery_charging,0.761905,0.842105,0.800000,19.0
5,camera_photos,0.818182,0.818182,0.818182,11.0
6,connectivity,0.411765,0.333333,0.368421,21.0
7,device_hardware,0.363636,0.266667,0.307692,15.0
8,display_input,0.462963,0.714286,0.561798,35.0
9,icloud_backup,1.000000,0.333333,0.500000,6.0


In [14]:
from src.evaluation import get_confusion_pairs

confusion_pairs = get_confusion_pairs(
    golden_set["intent"].values,
    predictions,
)

confusion_pairs

,true_intent,predicted_intent,count
0,ios_update,display_input,6
1,connectivity,display_input,5
2,apple_services,display_input,4
3,display_input,ios_update,4
4,apple_services,connectivity,4
5,apple_services,apps_app_store,4
6,connectivity,apple_services,3
7,audio,display_input,3
8,ios_update,connectivity,3
9,display_input,apple_services,3
